# Spain EV Charging Infrastructure Analysis
### IE Datathon 2026 — Iberdrola

---

| Section | Content |
|---|---|
| **0 — Setup** | Imports, constants, shared paths |
| **1 — Demand** | 1.1 EV Fleet Forecast · 1.2 Demand per Region |
| **2 — Supply** | 2.1 Roads & Traffic · 2.2 Existing Chargers · 2.3 Power Grid · 2.4 Points of Interest |
| **3 — Combined Analysis** | 3.1 Corridor Sizing · 3.2 Candidate Scoring · 3.3 Grid Status · 3.4–3.6 Output Files · 3.7 BI Map |

**Datathon outputs:** `File 1.csv` · `File 2.csv` · `File 3.csv` · `BI_map.html`


---
## 0. Setup & Imports

All imports, constants, and shared paths. Run this cell before any other section.

In [ ]:
# ── Standard library ─────────────────────────────────────────────────────────
import io, os, json, math, warnings, glob
import numpy as np
import pandas as pd
import topojson as tp
import re, requests, time
from pathlib import Path
from shapely.geometry import LineString, MultiLineString, Point
import unicodedata

# ── Plotting ──────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# ── Time-series models ────────────────────────────────────────────────────────
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_absolute_error, mean_squared_error
from prophet import Prophet

# ── Geospatial ────────────────────────────────────────────────────────────────
import geopandas as gpd
import pyproj

# ── Visualisation ─────────────────────────────────────────────────────────────
import folium
from folium.plugins import HeatMap, MarkerCluster
from IPython.display import IFrame, display

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_palette('colorblind')

# ── Output directory ──────────────────────────────────────────────────────────
OUT = Path('outputs')
OUT.mkdir(exist_ok=True)

# ── Shared path aliases (used by demand and supply sections) ──────────────────
REPO_ROOT  = os.path.abspath(os.path.join(os.getcwd(), '..'))
PARQUET_DIR = os.path.join(REPO_ROOT, 'mobility_electric_routes', 'Codigo', 'Data', 'parquet')
DATA_DIR   = os.path.join(REPO_ROOT, 'mobility_electric_routes', 'Codigo', 'Data', 'csv')
OUT_DIR    = str(OUT)       # string alias — used by demand section
OUTPUT_DIR = str(OUT)       # string alias — used by fleet section
BASE_DIR   = os.path.abspath(os.path.join(os.getcwd(), '..'))
FLEET_CSV  = str(OUT / 'total_ev_projected_2027.csv')
INPUT_CSV  = str(OUT / 'province_demand_2027.csv')

os.makedirs(OUT_DIR, exist_ok=True)

# ── CRS ───────────────────────────────────────────────────────────────────────
CRS_GEO    = 'EPSG:4326'
CRS_METRIC = 'EPSG:25830'

# ── Map defaults ──────────────────────────────────────────────────────────────
SPAIN_CENTER = [40.2, -3.7]
DEFAULT_ZOOM = 6
MAP_TILES    = 'CartoDB positron'

# ── Charger constants (fixed by datathon rules) ───────────────────────────────
CHARGER_KW        = 150
MIN_CHARGERS      = 2
MAX_CHARGERS      = 20
MAX_GAP_KM        = 150
COVERAGE_BUFFER_M = 50_000

# ── Grid status thresholds ────────────────────────────────────────────────────
GRID_SUFFICIENT_MW  = 5.0
GRID_MODERATE_MW    = 1.0
GRID_SNAP_RADIUS_M  = 10_000
CHARGER_GAP_MIN_M   = 40_000
TRAFFIC_BUFFER_M    = 8_000
CHARGER_KW_STANDARD = 150   # alias used in scoring section

# ── Road priority weights ─────────────────────────────────────────────────────
ROAD_PRIORITY = {'AP': 1.0, 'A': 0.95, 'N': 0.75, 'R': 0.60, 'M': 0.50}

# ── Distributor name normalisation ────────────────────────────────────────────
DISTRIBUTOR_MAP = {
    'i-DE (Iberdrola)': 'i-DE',
    'UFD (Naturgy)':    'i-DE',
    'e-dist (Endesa)':  'Endesa',
    'Viesgo':           'Viesgo'
}

# ── AFIR thresholds ───────────────────────────────────────────────────────────
AFIR_CORE_GAP_KM          = 60
AFIR_COMPREHENSIVE_GAP_KM = 100

# ── EV filter columns (shared across demand sections) ────────────────────────
EV_CATEGORIES = ['BEV', 'PHEV', 'REEV', 'FCEV']
EV_COL        = 'CATEGORÍA_VEHÍCULO_ELÉCTRICO'
PROV_COL      = 'COD_PROVINCIA_VEH'
YEARS         = [2021, 2022, 2023]

# ── Demand model parameters (Section 1.3) ────────────────────────────────────
INTERURBAN_TRIP_RATE = 0.03
CHARGING_NEED_RATE   = 0.45
PEAK_HOUR_SHARE      = 0.15
SESSION_DURATION_H   = 20 / 60
UTILIZATION_TARGET   = 0.75

# ── Global output variable — set by Section 1.1 ───────────────────────────────
TOTAL_EV_PROJECTED_2027 = 0

print('✓ Setup complete')
print(f'  Parquet dir : {PARQUET_DIR}')
print(f'Parquet files found: {len(glob.glob(os.path.join(PARQUET_DIR, "*.parquet")))}')
print(f'  CSV dir     : {DATA_DIR}')
print(f'  Outputs     : {OUT.resolve()}')


---
## 1. Demand

Pure demand-side analysis. No infrastructure proposals in this section.

| Sub-section | Key output |
|---|---|
| **1.1 EV Fleet Forecast** | `total_ev_projected_2027` · `outputs/total_ev_projected_2027.csv` |
| **1.2 Demand per Region** | `demand` DataFrame · `outputs/province_demand_2027.csv` |


---
### 1.1 EV Fleet Forecast

Fits ARIMA / SARIMA / Holt-Winters / Prophet against DGT monthly registrations.
Selects the best model by RMSE and projects the cumulative EV fleet to December 2027.

**Key output:** `total_ev_projected_2027` (integer) · saved to `outputs/total_ev_projected_2027.csv`


In [ ]:
EV_CATEGORIES = ['BEV', 'PHEV', 'REEV', 'FCEV']
CATEGORIA_COL = 'CATEGORÍA_VEHÍCULO_ELÉCTRICO'

records = []
parquet_files = sorted(glob.glob(os.path.join(PARQUET_DIR, '*.parquet')))

for path in parquet_files:
    fname = os.path.basename(path)          # e.g. '2023_01.parquet'
    year, month = fname.replace('.parquet', '').split('_')
    year, month = int(year), int(month)

    df = pd.read_parquet(path)

    # Normalise column name (encoding may vary across environments)
    cat_col = next((c for c in df.columns if 'CTRICO' in c or 'CTRICA' in c or 'ECTRICO' in c), None)
    if cat_col is None:
        continue

    total        = len(df)
    ev_mask      = df[cat_col].isin(EV_CATEGORIES)
    ev_count     = ev_mask.sum()
    bev_count    = (df[cat_col] == 'BEV').sum()
    phev_count   = (df[cat_col] == 'PHEV').sum()

    records.append({
        'year'        : year,
        'month'       : month,
        'date'        : pd.Timestamp(year=year, month=month, day=1),
        'total_reg'   : total,
        'ev_reg'      : ev_count,
        'bev_reg'     : bev_count,
        'phev_reg'    : phev_count,
    })

monthly = pd.DataFrame(records).sort_values('date').reset_index(drop=True)
monthly.set_index('date', inplace=True)

print(f'Period covered  : {monthly.index.min().strftime("%b %Y")} → {monthly.index.max().strftime("%b %Y")}')
print(f'Total months    : {len(monthly)}')
print(f'Total EV reg.   : {monthly["ev_reg"].sum():,}')
print()
print(monthly[['total_reg', 'ev_reg', 'bev_reg', 'phev_reg']].tail(12))

#### Exploratory Analysis

##### Monthly EV Registrations Over Time

Before fitting any model, we need to **understand the shape and behaviour of the data** — how many plug-in vehicles are being registered each month in Spain, how that has evolved since 2015, and whether there are clear seasonal or growth patterns a model needs to capture.

The chart below plots two lines:
- **Total plug-in** — every vehicle that requires an external charging point (BEV + PHEV + REEV + FCEV)
- **BEV only** — pure battery-electric vehicles with no combustion engine at all

And a second panel showing the **EV market share** — what percentage of all new car registrations that month were plug-in EVs.

---

#### What do the EV type acronyms mean?

| Code | Full name | How it works | Needs public charging? |
|------|-----------|-------------|----------------------|
| **BEV** | Battery Electric Vehicle | Runs 100% on a battery. No combustion engine. Must be plugged in to recharge. | Yes — fully dependent on it |
| **PHEV** | Plug-in Hybrid Electric Vehicle | Has both a battery and a petrol/diesel engine. Can be charged externally and can also run on fuel. Typically 40–80 km of electric range. | Yes — to use electric mode |
| **REEV** | Range-Extended Electric Vehicle | Runs primarily on battery but carries a small combustion engine that acts only as a generator to extend range — it never drives the wheels directly. | Yes |
| **FCEV** | Fuel Cell Electric Vehicle | Uses hydrogen fuel cells to generate electricity on board. No plug-in charging — refuels with hydrogen. Very rare in Spain. | No (hydrogen stations) |
| *HEV* | *(Hybrid Electric Vehicle)* | *Battery charged only by regenerative braking and the engine — cannot be plugged in. Excluded from this analysis.* | *No* |

> **Why exclude HEV?** A hybrid like the Toyota Yaris HEV recharges itself while driving — it never needs a public charger. Including it would inflate our demand figures without adding any real need for charging infrastructure.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# — Top: absolute monthly registrations
ax = axes[0]
ax.plot(monthly.index, monthly['ev_reg'],  label='Total plug-in (BEV+PHEV+REEV+FCEV)', lw=2)
ax.plot(monthly.index, monthly['bev_reg'], label='BEV only', lw=1.5, linestyle='--')
ax.set_title('Monthly EV Registrations — Spain (DGT Microdatos)', fontsize=13)
ax.set_ylabel('Vehicles registered')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# — Bottom: EV share of total registrations
ax2 = axes[1]
monthly['ev_share'] = monthly['ev_reg'] / monthly['total_reg'] * 100
ax2.fill_between(monthly.index, monthly['ev_share'], alpha=0.4)
ax2.plot(monthly.index, monthly['ev_share'], lw=1.5)
ax2.set_title('EV Market Share (% of monthly registrations)', fontsize=13)
ax2.set_ylabel('EV share (%)')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'ev_registrations_eda.png'), dpi=150, bbox_inches='tight')
plt.show()

##### Annual EV Registrations by Type

Zooming out from monthly noise to an annual view lets us see three things clearly:
1. **How fast the overall EV market is growing year on year**
2. **How the mix between BEV and PHEV is shifting** — BEV steadily taking share from PHEV
3. **Whether growth is accelerating or plateauing** — relevant for choosing a trend model

The chart uses **stacked bars** so the height of each bar = total plug-in registrations that year, while the colour split shows the BEV vs PHEV composition. YoY growth refers to the total plug-in count.

In [ ]:
annual = monthly.groupby('year')[['ev_reg', 'bev_reg', 'phev_reg', 'total_reg']].sum()
annual['other_reg']  = annual['ev_reg'] - annual['bev_reg'] - annual['phev_reg']  # REEV + FCEV
annual['ev_share']   = annual['ev_reg'] / annual['total_reg'] * 100
annual['yoy_growth'] = annual['ev_reg'].pct_change() * 100

years = annual.index.astype(str)
x     = np.arange(len(annual))
BAR_W = 0.55

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 9), gridspec_kw={'height_ratios': [3, 1]})
fig.subplots_adjust(hspace=0.08)

# ── Top panel: stacked bars ──────────────────────────────────────────────────
b_bev   = ax1.bar(x, annual['bev_reg'],   BAR_W, label='BEV',          color='#2E86AB')
b_phev  = ax1.bar(x, annual['phev_reg'],  BAR_W, label='PHEV',         color='#F6AE2D', bottom=annual['bev_reg'])
b_other = ax1.bar(x, annual['other_reg'], BAR_W, label='REEV / FCEV',  color='#A23B72',
                  bottom=annual['bev_reg'] + annual['phev_reg'])

# Total value label on top of each stack
for i, (yr, row) in enumerate(annual.iterrows()):
    ax1.text(i, row['ev_reg'] + 800, f"{int(row['ev_reg']):,}",
             ha='center', va='bottom', fontsize=8.5, fontweight='bold', color='#222')

# BEV share label inside each BEV bar (only if bar is tall enough)
for i, (yr, row) in enumerate(annual.iterrows()):
    bev_share = row['bev_reg'] / row['ev_reg'] * 100 if row['ev_reg'] > 0 else 0
    if row['bev_reg'] > 3000:
        ax1.text(i, row['bev_reg'] / 2, f"BEV\n{bev_share:.0f}%",
                 ha='center', va='center', fontsize=7.5, color='white', fontweight='bold')

ax1.set_title('Annual Plug-in EV Registrations by Type — Spain (DGT Microdatos)', fontsize=13, pad=10)
ax1.set_ylabel('Vehicles registered', fontsize=10)
ax1.set_xticks(x)
ax1.set_xticklabels([])          # hidden — shared with bottom panel
ax1.set_xlim(-0.5, len(x) - 0.5)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{int(v):,}'))
ax1.legend(loc='upper left', fontsize=9)
ax1.spines[['top', 'right']].set_visible(False)

# ── Bottom panel: YoY growth bars ───────────────────────────────────────────
colors_yoy = ['#2E86AB' if v >= 0 else '#E84855' for v in annual['yoy_growth'].fillna(0)]
ax2.bar(x[1:], annual['yoy_growth'].iloc[1:], BAR_W, color=colors_yoy[1:])
ax2.axhline(0, color='#555', lw=0.8)

for i, val in enumerate(annual['yoy_growth'].iloc[1:]):
    ax2.text(i + 1, val + (2 if val >= 0 else -5),
             f'{val:+.0f}%', ha='center', va='bottom', fontsize=8.5, fontweight='bold',
             color='#2E86AB' if val >= 0 else '#E84855')

ax2.set_ylabel('YoY growth', fontsize=10)
ax2.set_xticks(x)
ax2.set_xticklabels(years, fontsize=10)
ax2.set_xlim(-0.5, len(x) - 0.5)
ax2.set_ylim(annual['yoy_growth'].min() - 15, annual['yoy_growth'].max() + 20)
ax2.spines[['top', 'right']].set_visible(False)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:+.0f}%'))

plt.savefig(os.path.join(OUTPUT_DIR, 'ev_annual_registrations.png'), dpi=150, bbox_inches='tight')
plt.show()

print(annual[['ev_reg', 'bev_reg', 'phev_reg', 'ev_share', 'yoy_growth']].to_string())

##### Stationarity Test (Augmented Dickey-Fuller)

Before fitting ARIMA or SARIMA, we need to check whether the series is **stationary** — meaning its statistical properties (mean, variance) don't change over time.

**Why does this matter?** ARIMA and SARIMA are built on the assumption that the series has been made stationary. If you feed a non-stationary series directly into ARIMA, the model will produce unreliable estimates because it can't distinguish a real signal from a drifting trend.

**What is the ADF test?** The Augmented Dickey-Fuller test checks for the presence of a **unit root** — the statistical signature of a non-stationary series. The null hypothesis is *"the series has a unit root (non-stationary)"*, so:
- **p-value < 0.05** → reject the null → series **is stationary** ✓
- **p-value ≥ 0.05** → fail to reject → series **is non-stationary** ✗

We test three versions of the series to understand how many differencing steps are needed:
1. **Raw** — the original monthly EV registration counts
2. **1st-order differenced** — the month-on-month *change* in registrations (removes trend)
3. **Seasonal differenced (lag-12)** — the year-on-year *change* for the same month (removes annual seasonality)

In [ ]:
series = monthly['ev_reg'].asfreq('MS')  # monthly start frequency

def adf_report(s, label):
    result = adfuller(s.dropna())
    print(f'ADF Test — {label}')
    print(f'  Statistic : {result[0]:.4f}')
    print(f'  p-value   : {result[1]:.4f}  {"→ STATIONARY" if result[1] < 0.05 else "→ non-stationary"}')
    print()

adf_report(series, 'Raw EV registrations')
adf_report(series.diff(1), '1st-order differenced')
adf_report(series.diff(12), 'Seasonal differenced (lag-12)')

In [ ]:
s_raw       = series                      # raw
s_d1        = series.diff(1)              # 1st-order diff   (removes trend)
s_D1        = series.diff(12)             # seasonal diff    (removes seasonality)
s_d1_D1     = series.diff(1).diff(12)    # both             (what SARIMA works on internally)

fig, axes = plt.subplots(4, 2, figsize=(14, 14))
fig.suptitle('ACF & PACF — Four transformations of the EV registration series', fontsize=12, y=1.01)

labels = [
    ('Raw series',                        s_raw),
    ('1st-order differenced  (d=1)',       s_d1),
    ('Seasonal differenced  (D=1, s=12)', s_D1),
    ('d=1  +  D=1  (SARIMA working form)',s_d1_D1),
]

for row, (label, s) in enumerate(labels):
    plot_acf( s.dropna(), lags=36, ax=axes[row, 0], title=f'ACF  — {label}',  alpha=0.05)
    plot_pacf(s.dropna(), lags=36, ax=axes[row, 1], title=f'PACF — {label}', alpha=0.05)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'acf_pacf_all.png'), dpi=150, bbox_inches='tight')
plt.show()

#### What are ACF and PACF, and how do we read them?

**ACF (Autocorrelation Function)** — measures how correlated the series is with its own past values at each lag. A spike at lag *k* means "the value today is correlated with the value *k* months ago."

**PACF (Partial Autocorrelation Function)** — same idea, but *removes* the effect of all intermediate lags. So a spike at lag 2 in the PACF means "there is a direct relationship between today and 2 months ago, even after accounting for lag 1."

The **shaded blue band** is the 95% confidence interval — spikes outside it are statistically significant. Spikes inside are noise.

We plot **four transformations** to fully understand the structure before choosing model orders — raw, trend-differenced, seasonally-differenced, and the combined form SARIMA works on internally.

---

> **On the p-value of 0.0000 for the 1st-order differenced series:** not suspicious at all. The ADF statistic of -8.87 is so far into the rejection region (1% critical value is -3.5) that the true p-value is around 10^-14 — Python just rounds it to 0.0000. Think of it as a z-score of nearly 9. This is a strong, unambiguous result.

---

#### Reading the four rows

**Row 1 — Raw series:**
- **ACF:** Starts at ~1.0 and decays extremely slowly — still positive and significant past lag 25. Textbook signature of a trending, non-stationary series. The series has a very long memory because it never stops drifting upward.
- **PACF:** One massive spike at lag 1 (~0.9), then essentially zero after that. All the trend information is concentrated in a single one-month lag.

**Row 2 — 1st-order differenced (d=1):**
- **ACF:** Collapses immediately — a significant **negative spike at lag 1** (~-0.35), then most lags fall inside the confidence band. A small positive spike around lag 12 just outside the band confirms annual seasonality is still present. Negative lag-1 spike signals **MA(q=1)**.
- **PACF:** Large **negative spike at lag 1** (~-0.45) and a smaller one at lag 2 — consistent with **AR(p=1 or 2)**. Seasonal spikes at lags 12/24 confirm D=1 is also needed.

**Row 3 — Seasonal differenced only (D=1, s=12):**
- **ACF:** Positive autocorrelations at lags 1-4 are still clearly significant — the upward trend has *not* been removed. Still showing the slow-decline shape of a non-stationary series, just less extreme than raw. Seasonal differencing alone is not enough.
- **PACF:** Dominant spike at lag 1 and a notable **negative spike around lag 11-12** — seasonal structure partially addressed but trend stationarity still violated. Cross-validates the ADF p-value of 0.2629.

**Row 4 — d=1 + D=1 combined (SARIMA working form):**
- **ACF:** The cleanest picture — **significant negative spike at lag 1** (~-0.5), then virtually everything inside the confidence band. A faint signal around lag 12-13 suggests residual seasonal MA structure (Q=1).
- **PACF:** A **large positive spike at lag 1** (~0.75), a smaller negative one at lag 2, and a notable **negative spike around lag 11-12** — the seasonal AR fingerprint (P=1). This is the key diagnostic row.

---

#### Model order summary from the plots

| Component | Diagnostic signal | Chosen order |
|-----------|------------------|-------------|
| Regular differencing | Raw ACF: very slow decay | **d = 1** |
| Seasonal differencing | d=1 ACF: spikes at lag 12/24 remain | **D = 1, s = 12** |
| AR order (p) | PACF d=1: spikes at lags 1-2, cuts off | **p = 1-2** |
| MA order (q) | ACF d=1 and d=1+D=1: negative spike at lag 1 | **q = 1** |
| Seasonal AR (P) | PACF d=1+D=1: significant spike at lag 11-12 | **P = 1** |
| Seasonal MA (Q) | ACF d=1+D=1: faint spike at lag 12-13 | **Q = 1** |

These plots justify the orders **ARIMA(2,1,2)** and **SARIMA(1,1,1)(1,1,1,12)** used in the models below.

#### Train / Test Split

To evaluate our models honestly, we need to **withhold some data the model has never seen** and measure how well it predicts those known values. This is the test set.

##### Why 12 months?

Choosing 12 months (one full calendar year) as the test set is the standard choice for monthly time series with annual seasonality, and here is why:

- **Covers one complete seasonal cycle.** Vehicle registrations have a strong 12-month pattern (August dip, December peak, etc.). Holding out exactly 12 months means the model must correctly forecast *every seasonal phase* — not just the easy months. A shorter test set (e.g., 3 months) could accidentally pick an easy period.
- **Meaningful absolute size.** 12 data points is enough for RMSE, MAPE and MAE to be stable estimates of forecast error. Fewer than 6 months gives noisy metrics.
- **Preserves enough training data.** We have 108 months total (2015-2023). Holding out 12 leaves 96 months for training — still ~8 full seasonal cycles for SARIMA to learn from. Going larger (e.g., 24 months) would leave only 84 training points, which starts to hurt seasonal models.

##### Could we use a different number?

Yes — and it is a valid modelling choice with real trade-offs:

| Test size | Training months | Pros | Cons |
|-----------|----------------|------|------|
| **6 months** | 102 | More data to train on | Only half a seasonal cycle tested — biased metric |
| **12 months** <- *our choice* | 96 | Full seasonal cycle, stable metrics, ample training | — |
| **24 months** | 84 | Tests two full cycles, more robust metrics | Fewer training points, seasonal models may underfit |
| **Rolling / walk-forward** | Varies | Most rigorous — tests the model repeatedly | Much more compute, harder to visualise |

> For this datathon, **12 months is the right balance**. We are forecasting 3 years ahead (2025-2027), so the test horizon should be at least as long as one cycle to confirm the model can handle seasonality reliably.

In [ ]:
TEST_MONTHS = 12

train = series.iloc[:-TEST_MONTHS]
test  = series.iloc[-TEST_MONTHS:]

print(f'Train: {train.index[0].strftime("%b %Y")} → {train.index[-1].strftime("%b %Y")}  ({len(train)} months)')
print(f'Test : {test.index[0].strftime("%b %Y")}  → {test.index[-1].strftime("%b %Y")}  ({len(test)} months)')

#### Evaluation Helper Functions

We define two utility functions used consistently across all models:

- **`mape(y_true, y_pred)`** — Mean Absolute Percentage Error, masked to exclude zero values (avoids division by zero during months with no registrations).
- **`evaluate(y_true, y_pred, model_name)`** — returns a standardised metrics dict with RMSE, MAE, MAPE, and optionally AIC/BIC (available for ARIMA-family models only).

All results are collected into `results` (list of metric dicts) and `forecasts` (dict of predicted series), which feed the comparison table in Section 10.

In [ ]:
def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def evaluate(y_true, y_pred, model_name, aic=None, bic=None):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    mape_val = mape(y_true, y_pred)
    return {
        'Model'  : model_name,
        'RMSE'   : round(rmse, 1),
        'MAE'    : round(mae, 1),
        'MAPE %' : round(mape_val, 2),
        'AIC'    : round(aic, 1) if aic is not None else '—',
        'BIC'    : round(bic, 1) if bic is not None else '—',
    }

results = []   # collect metrics from all models
forecasts = {} # store test-period & future forecasts

#### Model 1 — ARIMA

**ARIMA(p, d, q)** models the series as a linear function of its own past values (AR), past forecast errors (MA), and differencing to achieve stationarity (I).

**Order selection rationale:**
- `d=1` — one round of differencing removes the upward trend (confirmed by ADF test in Section 3)
- `p=2, q=2` — selected via ACF/PACF inspection of the differenced series
- A **log1p transform** is applied before fitting to stabilise variance (EV registrations grow exponentially); predictions are back-transformed with `expm1`

**Reading the output:**
- `ar.L1` is the only statistically significant coefficient (p < 0.001), suggesting most predictive power comes from the immediately prior month
- **Ljung-Box p = 0.46** → residuals are uncorrelated (white noise) ✓
- **Jarque-Bera p = 0.00, kurtosis = 9.47** → residuals are heavy-tailed, not normally distributed — 95% confidence intervals should be treated as approximate
- **AIC: 103.5 | BIC: 116.3** — reference values for cross-model comparison in Section 10


In [ ]:
log_train = np.log1p(train)

# Order selected via ACF/PACF inspection: d=1 (non-stationary), p=2, q=2
ARIMA_ORDER = (2, 1, 2)

arima_model  = ARIMA(log_train, order=ARIMA_ORDER).fit()
arima_pred   = np.expm1(arima_model.forecast(steps=TEST_MONTHS))
arima_pred.index = test.index

print(arima_model.summary())
print(f'\nAIC: {arima_model.aic:.1f}  |  BIC: {arima_model.bic:.1f}')

results.append(evaluate(test, arima_pred, f'ARIMA{ARIMA_ORDER}', arima_model.aic, arima_model.bic))
forecasts['ARIMA'] = arima_pred
print('\nTest-period metrics:', results[-1])

#### Model 2 — SARIMA (1,1,1)

**SARIMA(p,d,q)(P,D,Q,s)** extends ARIMA with seasonal AR and MA components.  
`s=12` captures annual seasonality in vehicle registrations (August dip, December peak).

**Order selection:**
- Non-seasonal: same as ARIMA — `(1,1,1)` 
- Seasonal: `(1,1,1,12)` — one seasonal AR, one seasonal difference, one seasonal MA

**Reading the output:**
- `ma.L1` is the only clearly significant coefficient (p < 0.001)
- `ma.S.L12` coefficient ≈ −1.000 with std err of 617 — a near-unit-root warning, suggesting the seasonal MA term is poorly identified; the model may be over-differenced seasonally
- **Ljung-Box p = 0.93** → residuals are white noise ✓
- **Jarque-Bera p = 0.00, kurtosis = 11.55** → heavy-tailed residuals, same caveat as ARIMA on confidence intervals
- **AIC: 73.0 | BIC: 84.1** — better than ARIMA on information criteria, but **RMSE: 3604 > ARIMA's 3345** — the seasonal fit improves in-sample but not out-of-sample


In [ ]:
SARIMA_ORDER         = (1, 1, 1)
SARIMA_SEASONAL_ORDER = (1, 1, 1, 12) 
sarima_model = SARIMAX(
    log_train,
    order=SARIMA_ORDER,
    seasonal_order=SARIMA_SEASONAL_ORDER,
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

sarima_pred = np.expm1(sarima_model.forecast(steps=TEST_MONTHS))
sarima_pred.index = test.index

print(sarima_model.summary())
print(f'\nAIC: {sarima_model.aic:.1f}  |  BIC: {sarima_model.bic:.1f}')

results.append(evaluate(test, sarima_pred, f'SARIMA{SARIMA_ORDER}x{SARIMA_SEASONAL_ORDER}', sarima_model.aic, sarima_model.bic))
forecasts['SARIMA'] = sarima_pred
print('\nTest-period metrics:', results[-1])

##### SARIMA Variant — Dampened Seasonal (D=0)

The standard SARIMA(1,1,1)(1,1,1,12) above applies **double differencing** (d=1 regular + D=1 seasonal). While this produces excellent AIC/BIC on the training window, it tends to **amplify the trend exponentially** over long forecast horizons (48 months to Dec 2027), leading to unrealistic registration counts in 2026–2027.

We test an alternative: **SARIMA(1,1,1)(1,0,1,12)** — same regular differencing (d=1) and same seasonal AR/MA terms, but **D=0** (no seasonal differencing). This:

- Still captures the 12-month seasonal cycle via the SAR(1) and SMA(1) terms
- Avoids the compounding trend explosion caused by double differencing
- Produces more conservative, planning-appropriate long-range forecasts

We include both variants in the model comparison to show the trade-off between in-sample fit (AIC) and long-range forecast stability.

In [ ]:
SARIMA_D0_ORDER         = (1, 1, 1)
SARIMA_D0_SEASONAL_ORDER = (1, 0, 1, 12)   # D=0: no seasonal differencing

sarima_d0_model = SARIMAX(
    log_train,
    order=SARIMA_D0_ORDER,
    seasonal_order=SARIMA_D0_SEASONAL_ORDER,
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

sarima_d0_pred = np.expm1(sarima_d0_model.forecast(steps=TEST_MONTHS))
sarima_d0_pred.index = test.index

print(sarima_d0_model.summary())
print(f'\nAIC: {sarima_d0_model.aic:.1f}  |  BIC: {sarima_d0_model.bic:.1f}')

results.append(evaluate(test, sarima_d0_pred, f'SARIMA{SARIMA_D0_ORDER}x{SARIMA_D0_SEASONAL_ORDER}', sarima_d0_model.aic, sarima_d0_model.bic))
forecasts['SARIMA_D0'] = sarima_d0_pred
print('\nTest-period metrics:', results[-1])

#### Model 3 — Holt-Winters ETS

**Triple Exponential Smoothing** decomposes the series into level, trend, and seasonal components,  
each updated with separate smoothing parameters (α, β, γ).  
We use additive trend + multiplicative seasonality, which suits series that grow with a proportional seasonal pattern.

**Reading the output:**
- **Alpha (level) = 0.259** — moderate responsiveness to recent level changes
- **Beta (trend) = 0.259** — moderate trend tracking
- **Gamma (seasonality) = 0.000** — the optimizer zeroed out the seasonal component entirely; despite specifying multiplicative seasonality, the model found no seasonal signal worth capturing in the training window, effectively reducing to Holt's double smoothing
- This likely explains the worst test performance of all models: **RMSE: 4373, MAPE: 34.6%**
- **AIC/BIC (1333/1374) are not comparable** to ARIMA/SARIMA — those were fitted on log-transformed data; likelihood scales differ

> Gamma = 0 is a warning: EV adoption in Spain may not yet exhibit a stable enough seasonal cycle for ETS to exploit.

In [ ]:
hw_model = ExponentialSmoothing(
    train,
    trend='add',
    seasonal='mul',
    seasonal_periods=12,
    initialization_method='estimated'
).fit(optimized=True)

hw_pred = hw_model.forecast(steps=TEST_MONTHS)
hw_pred.index = test.index

print(f'AIC: {hw_model.aic:.1f}  |  BIC: {hw_model.bic:.1f}')
print(f'Alpha (level): {hw_model.params["smoothing_level"]:.4f}')
print(f'Beta  (trend): {hw_model.params["smoothing_trend"]:.4f}')
print(f'Gamma (seas.) : {hw_model.params["smoothing_seasonal"]:.4f}')

results.append(evaluate(test, hw_pred, 'Holt-Winters (add+mul)', hw_model.aic, hw_model.bic))
forecasts['Holt-Winters'] = hw_pred
print('\nTest-period metrics:', results[-1])

#### Model 4 — Prophet

Facebook's **Prophet** fits a piecewise-linear trend with automatic changepoint detection,  
plus Fourier-series seasonality. It requires no stationarity transformation.

**Configuration choices:**
- `yearly_seasonality=True` — monthly data, annual cycle is the relevant frequency
- `weekly_seasonality=False`, `daily_seasonality=False` — irrelevant at monthly resolution
- `seasonality_mode='multiplicative'` — seasonality scales with the level of the series
- `changepoint_prior_scale=0.3` — moderate flexibility; higher values risk overfitting the trend
- `clip(lower=0)` — prevents negative registration forecasts

**Reading the component plots:**
- **Trend**: piecewise-linear with detected changepoints around 2020–2021 (COVID dip + recovery surge), then continued acceleration — correctly captures the structural break
- **Yearly seasonality**: the ±1000% swings are an **artifact of multiplicative mode on a near-zero series** (2015–2016); dividing by tiny early values inflates percentages. The *shape* is interpretable (August dip, late-December peak) but the magnitude is not meaningful
- **RMSE: 3357** — second-best overall, only marginally behind ARIMA (3345), despite no log transformation

> Prophet does not output AIC/BIC — information-criteria comparison with ARIMA/SARIMA is not applicable.

In [ ]:
# Prophet requires a DataFrame with columns 'ds' (date) and 'y' (value)
prophet_train = pd.DataFrame({
    'ds': train.index,
    'y' : train.values
})

prophet_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    seasonality_mode='multiplicative',
    changepoint_prior_scale=0.3   # allow moderate flexibility in trend changes
)
prophet_model.fit(prophet_train)

future = prophet_model.make_future_dataframe(periods=TEST_MONTHS, freq='MS')
prophet_forecast = prophet_model.predict(future)

# Extract test-period predictions
prophet_pred = prophet_forecast.set_index('ds')['yhat'].loc[test.index]
prophet_pred = prophet_pred.clip(lower=0)  # no negative registrations

results.append(evaluate(test, prophet_pred, 'Prophet (multiplicative)', aic=None, bic=None))
forecasts['Prophet'] = prophet_pred
print('Test-period metrics:', results[-1])

# Prophet component plots
fig = prophet_model.plot_components(prophet_forecast)
plt.suptitle('Prophet — Trend & Seasonality Components', y=1.02)
plt.tight_layout()
plt.show()

##### Prophet Component Interpretation

**Trend (upper panel):**  
The trend is piecewise-linear with two distinct phases:
- **2015–2020**: slow, nearly flat growth — EV adoption in Spain was nascent
- **2020–2021**: sharp acceleration — post-COVID recovery coincided with Spain's MOVES II/III 
  subsidy programmes driving a structural break in adoption rate
- **2022–2024**: continued steep growth, with a slightly narrowing confidence band as 
  the model anchors on the most recent changepoint

**Yearly seasonality (lower panel):**  
The shape is interpretable despite the distorted scale:
- **August trough** — summer holiday period, dealerships and buyers inactive
- **November dip** — pre-December pause
- **Late December spike** — year-end registration surge (tax incentives, fleet renewals, 
  dealer targets)

> ⚠️ The ±1000% amplitude is an artefact: multiplicative seasonality divides by near-zero  
> values from 2015–2016, inflating percentages to meaningless magnitudes.  
> The *pattern* is real; the *scale* is not.


#### Model Comparison

All **five models** are evaluated on the same held-out test set (last 12 months of historical data, Jan–Dec 2023). This includes:

| Model | Type | Key characteristic |
|-------|------|--------------------|
| ARIMA(2,1,2) | Univariate | No seasonality; mean-reverting |
| SARIMA(1,1,1)(1,1,1,12) | Seasonal | Double differencing (d=1, D=1) |
| SARIMA(1,1,1)(1,0,1,12) | Seasonal | Single differencing (d=1, D=0) — dampened seasonal |
| Holt-Winters (add+mul) | ETS | Triple exponential smoothing |
| Prophet (multiplicative) | Bayesian | Piecewise trend + Fourier seasonality |

Metrics reported: **RMSE** (absolute accuracy), **MAE** (robust to outliers), **MAPE %** (scale-free accuracy), **AIC** and **BIC** (information criteria — penalise complexity, available only for ARIMA-family models).

The best model is selected using a combination of AIC (model quality) and RMSE/MAPE (out-of-sample accuracy), prioritising models that perform well on both dimensions.

In [ ]:
metrics_df = (
    pd.DataFrame(results)
    .drop_duplicates(subset='Model', keep='last')
    .set_index('Model')
    .replace('—', np.nan)
)

metrics_df[['AIC', 'BIC']] = metrics_df[['AIC', 'BIC']].apply(pd.to_numeric, errors='coerce')

# Identify ARIMA-family model names dynamically
arima_models = [m for m in metrics_df.index if 'ARIMA' in m]

styled = (
    metrics_df.style
    .highlight_min(subset=['RMSE', 'MAE', 'MAPE %'], color="#15d041")
    .highlight_max(subset=['RMSE', 'MAE', 'MAPE %'], color="#da0719")
    .highlight_min(subset=pd.IndexSlice[arima_models, ['AIC', 'BIC']], color="#13cd3e")
    .highlight_max(subset=pd.IndexSlice[arima_models, ['AIC', 'BIC']], color="#cf1726")
    .format({'RMSE': '{:.1f}', 'MAE': '{:.1f}', 'MAPE %': '{:.2f}',
             'AIC': '{:.1f}', 'BIC': '{:.1f}'}, na_rep='—')
    .set_caption('Model Comparison — Test Set (last 12 months)')
)

# Select best model by RMSE — SARIMA(1,0,1,12) wins on both RMSE and AIC
best_model_name = metrics_df['RMSE'].idxmin()

display(styled)
print(f'\nBest model: {best_model_name}')

##### Results Interpretation

**Winner: SARIMA(1,1,1)(1,0,1,12)** — best RMSE (1,393), best MAPE (9.27%), and joint-best AIC (73). It is the only model that combines strong information-theoretic fit with excellent out-of-sample accuracy.

| Model | RMSE | MAPE % | AIC | Strengths | Weaknesses |
|-------|------|--------|-----|-----------|------------|
| **SARIMA(1,0,1,12)** ✅ | **1,393** | **9.27** | **73** | Best on all accuracy metrics; captures seasonality; stable long-range trend | Slightly less aggressive trend — may underestimate if EV adoption accelerates sharply |
| SARIMA(1,1,1,12) | 3,605 | 20.73 | 73 | Joint-best AIC; captures seasonal shape | Double differencing causes explosive forecasts beyond 18 months |
| ARIMA(2,1,2) | 3,346 | 20.17 | 104 | Simple; reasonable short-run accuracy | No seasonal component; flat forecast |
| Prophet | 3,357 | 22.12 | — | Robust to changepoints; no transformation | Underestimates 2023 surge; no AIC/BIC |
| Holt-Winters | 4,373 | 34.55 | 1,333 | Interpretable parameters | γ ≈ 0 — seasonal component discarded; worst on all metrics |

**Why SARIMA(1,0,1,12) over SARIMA(1,1,1,12)?**
Both models share the same AIC (73), meaning they are equally parsimonious in-sample. However, removing the seasonal differencing (D=0) prevents the double-differencing instability that causes SARIMA(D=1) to project implausibly high registration counts (>100k/month) over a 48-month horizon. With D=0, the seasonal pattern is captured entirely through the SAR(1) and SMA(1) terms, which generalise more reliably to unseen data — as confirmed by the MAPE of 9.27% vs 20.73%.

**Why not ARIMA or Prophet for the final forecast?**
ARIMA produces a flat mean-reversion forecast that ignores Spain's clear seasonal EV registration cycle. Prophet underestimates recent growth momentum. Neither is appropriate for a 48-month projection where seasonal structure and trend continuation are critical.

#### Visual Comparison

In [ ]:
# Visual comparison: actual vs each model on test period
fig, axes = plt.subplots(5, 1, figsize=(15, 25))

colors = {
    'ARIMA'      : 'tab:blue',
    'SARIMA'     : 'tab:orange',
    'SARIMA_D0'  : 'tab:purple',
    'Holt-Winters': 'tab:green',
    'Prophet'    : 'tab:red',
}

# Map forecasts short key -> full metrics_df index label
# SARIMA_D0 needs special handling since it won't prefix-match
def resolve_name(short, index):
    if short == 'SARIMA_D0':
        # match the D=0 variant specifically
        matches = [m for m in index if 'SARIMA' in m and '(1, 0, 1, 12)' in m]
    else:
        matches = [m for m in index if m.startswith(short)]
    return matches[0] if matches else short

for i, (name, pred) in enumerate(forecasts.items()):
    ax = axes[i]
    ax.plot(train.index[-24:], train.values[-24:], label='Train (last 24m)', color='gray', lw=1.5)
    ax.plot(test.index, test.values, label='Actual (test)', color='black', lw=2, marker='o', ms=4)
    ax.plot(pred.index, pred.values, label=f'{name} forecast', color=colors[name], lw=2, linestyle='--', marker='x', ms=4)
    ax.fill_between(test.index, test.values * 0.85, test.values * 1.15, alpha=0.08, color='black', label='±15% band')

    full_name = resolve_name(name, metrics_df.index)
    row = metrics_df.loc[full_name]
    aic_str = f'  AIC={row["AIC"]:.0f}' if pd.notna(row['AIC']) else ''
    title = f'{full_name}\nRMSE={row["RMSE"]:,.0f}  MAPE={row["MAPE %"]:.2f}%{aic_str}'
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Model Comparison — Test Period (2024)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

##### Model Comparison — Visual Interpretation

Each panel shows the **last 24 months of training data** (gray), the **12-month held-out test set** (black, Jan–Dec 2023), and each model's forecast (coloured dashed line) with a ±15% tolerance band around the actuals.

---

**ARIMA(2,1,2) — RMSE 3,346 | MAPE 20.2% | AIC 104**

ARIMA produces a near-flat forecast around 8,500–9,000 registrations per month. It captures the general level of the series but is completely blind to seasonality — it misses the sharp May 2023 peak (~14,500) and the September dip entirely. The model predicts *steady-state mean reversion* with no cyclical structure. Decent short-run accuracy but structurally weak.

---

**SARIMA(1,1,1)(1,1,1,12) — RMSE 3,605 | MAPE 20.7% | AIC 73**

Reproduces the seasonal *shape* of EV registrations (spring rise, autumn dip) but **overestimates** the final months, projecting up to ~22,000 by January 2024 when the actual was ~15,000. The double differencing (d=1 + D=1) amplifies the upward trend over time, making this variant unsuitable for a 48-month forecast horizon despite its strong AIC.

---

**Holt-Winters (additive trend + multiplicative seasonal) — RMSE 4,373 | MAPE 34.6% | AIC 1,333**

Clearly the weakest performer. The forecast stays in the 6,000–10,000 range while actuals reach 14,500+. The fitted seasonal smoothing parameter γ ≈ 0, meaning the model *discarded* its seasonal component during optimisation and ran as a non-seasonal smoother. The extremely high AIC (1,333) confirms a poor fit.

---

**Prophet (multiplicative seasonality) — RMSE 3,357 | MAPE 22.1%**

Competitive raw accuracy (RMSE close to ARIMA). Captures some seasonal structure but systematically *underestimates* the 2023 surge — the changepoint prior dampened the recent upward shift in trend. No AIC/BIC available (Prophet uses MAP estimation, not MLE).

---

**SARIMA(1,1,1)(1,0,1,12) — RMSE 1,393 | MAPE 9.27% | AIC 73** ✅ **Winner**

This is the standout model. The purple forecast line tracks the actual series closely throughout the entire test period, staying well within the ±15% band for almost every month. By removing the seasonal differencing (D=0 instead of D=1), the model:
- Still captures the 12-month seasonal cycle via the SAR(1) and SMA(1) terms
- Avoids the trend amplification that destabilises the D=1 variant over long horizons
- Achieves an RMSE of just **1,393** — less than half of any other model — and a MAPE of **9.27%**, well within the 10% threshold for a reliable forecast

It ties SARIMA(D=1) on AIC (73) while being dramatically more accurate on the test set.

---

**Bottom line:** SARIMA(1,1,1)(1,0,1,12) wins on every metric that matters — **best RMSE, best MAPE, and joint-best AIC** — while also being the most stable model for long-range projection. It is selected as the final model for the 2027 EV fleet forecast.

#### Best Model — Forecast to December 2027

Having evaluated all five models, **SARIMA(1,1,1)(1,0,1,12)** is selected as the final model. It achieves the best performance across every key metric:

- **RMSE: 1,393** — less than half of the next best model (ARIMA: 3,346)
- **MAPE: 9.27%** — well within the 10% reliability threshold
- **AIC: 73** — joint-best among all ARIMA-family models

Crucially, by using D=0 (no seasonal differencing), it avoids the explosive trend extrapolation that makes SARIMA(D=1) unreliable over a 48-month horizon, while still capturing Spain's 12-month EV registration cycle through its SAR(1) and SMA(1) seasonal terms.

##### Why re-fit on the full series?

During evaluation (Section 10), each model was trained on ~96% of the data and tested on the last 12 months. Now that we have confirmed which model to trust, we **re-fit on 100% of the available data** (Jan 2015 → Dec 2023) before forecasting forward. This gives the model:

- **More signal** — the full 108-month series, including the 2023 acceleration in EV registrations
- **Better trend and seasonal estimates** — especially for the SAR(1) and SMA(1) seasonal terms
- **No wasted data** — the test set is no longer held back; every observation informs the final forecast

##### Forecast horizon

We project **48 months ahead**: January 2024 → December 2027. This covers the full period needed to estimate Spain's EV fleet size at end-2027, which feeds directly into the charging infrastructure demand model.

In [ ]:
FORECAST_HORIZON = 48   # months: Jan 2024 → Dec 2027
log_full = np.log1p(series)

print('Selected model: SARIMA(1,1,1)(1,0,1,12) — best RMSE (1,393), MAPE (9.27%) and joint-best AIC (73)')
print(f'Forecasting {FORECAST_HORIZON} months beyond {series.index[-1].strftime("%b %Y")} → Dec 2027\n')

# Re-fit SARIMA(1,1,1)(1,0,1,12) on full series (Jan 2015 → Dec 2023)
final_model = SARIMAX(
    log_full,
    order=SARIMA_D0_ORDER,
    seasonal_order=SARIMA_D0_SEASONAL_ORDER,
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

fc_res    = final_model.get_forecast(steps=FORECAST_HORIZON)
fc_values = np.expm1(fc_res.predicted_mean)
fc_conf   = np.expm1(fc_res.conf_int())

fc_index  = pd.date_range(
    start=series.index[-1] + pd.DateOffset(months=1),
    periods=FORECAST_HORIZON,
    freq='MS'
)
fc_values = pd.Series(np.array(fc_values).flatten(), index=fc_index)
print(fc_values.tail(12).to_string())

##### Forecast Output — Interpretation

The **SARIMA(1,1,1)(1,0,1,12)** model re-fitted on the full series (Jan 2015 → Dec 2023) produces the following forward projections for Jan 2024 → Dec 2027:

**What the numbers mean:**
Each value is the model's estimate of **monthly new EV registrations in Spain** for that month. These are *new vehicles added to the road*, not the cumulative fleet — that calculation comes in Section 12.

**Seasonal pattern preserved:**
With D=0, the 12-month cycle is captured entirely through the SAR(1) and SMA(1) terms, which proved more stable over the 48-month horizon than seasonal differencing. The forecast correctly carries forward the seasonal rhythm observed in historical data:
- **Spring peaks** (Mar–Jun): registration surges driven by fleet renewals and dealer incentive periods
- **Summer/autumn dip** (Aug–Sep): holiday slowdown in vehicle purchases
- **Year-end acceleration** (Nov–Dec): dealers push to hit annual targets; buyers rush before fiscal year-end

**Trend direction:**
The forecast projects a moderate, realistic upward trend from ~15,000/month (Dec 2023) to ~30,000–32,000/month (Dec 2027) — roughly a 2× increase over 4 years. This is consistent with Spain's MOVES III subsidy programme (active through 2025–2026), EU fleet emission mandates, and growing model availability from manufacturers.

**Why these values are more trustworthy than SARIMA(D=1):**
The previous SARIMA(1,1,1,12) variant projected >100,000 registrations/month by Dec 2027 due to compounding trend amplification from double differencing. The D=0 variant avoids this by anchoring its seasonal structure to the level of the series rather than its differences — producing forecasts that align with realistic EV market growth scenarios.

**Uncertainty note:**
Confidence intervals widen beyond 12–18 months, as is standard for SARIMA projections. The point estimates for 2026–2027 should be treated as indicative — the value of this forecast is in the *order of magnitude* and *seasonal shape*, not exact monthly figures. The cumulative fleet total (Section 12) smooths out month-to-month noise and is the more reliable output for infrastructure planning.

##### Forecast Chart

In [ ]:
FINAL_MODEL_LABEL = 'SARIMA(1,1,1)(1,0,1,12)'

fig, ax = plt.subplots(figsize=(16, 6))

forecast_start = fc_values.index[0]

# --- Historical observed ---
ax.plot(series.index, series.values,
        label='Observed (DGT)', color='steelblue', lw=2)

# --- Forecast point estimate ---
ax.plot(fc_values.index, fc_values.values,
        label=f'{FINAL_MODEL_LABEL} forecast',
        color='darkorange', lw=2.5, linestyle='--')

# --- Confidence interval clipped to point-forecast scale ---
if fc_conf is not None:
    ci_arr   = np.array(fc_conf)
    y_cap    = fc_values.values * 2
    ci_upper = np.minimum(ci_arr[:, 1], y_cap)
    ci_lower = np.maximum(ci_arr[:, 0], 0)
    ax.fill_between(fc_values.index, ci_lower, ci_upper,
                    alpha=0.20, color='darkorange', label='95% CI (clipped)')

# --- Shade forecast region ---
ax.axvspan(forecast_start, fc_values.index[-1],
           alpha=0.04, color='darkorange', zorder=0)

# --- Vertical separator at forecast start ---
ax.axvline(forecast_start, color='dimgray', lw=1.2, linestyle='--', alpha=0.6)

# --- 2027 boundary ---
ax.axvline(pd.Timestamp('2027-01-01'), color='red', lw=1, linestyle=':', alpha=0.5)

# --- Annotate last observed & final forecast ---
last_obs  = series.iloc[-1]
last_fc   = fc_values.iloc[-1]

obs_label = f'{last_obs:,.0f}' + '\nDec 2023'
fc_label  = f'{last_fc:,.0f}' + '\nDec 2027'

ax.annotate(obs_label,
            xy=(series.index[-1], last_obs),
            xytext=(-55, 25), textcoords='offset points',
            fontsize=8.5, color='steelblue',
            arrowprops=dict(arrowstyle='->', color='steelblue', lw=1))
ax.annotate(fc_label,
            xy=(fc_values.index[-1], last_fc),
            xytext=(-70, 25), textcoords='offset points',
            fontsize=8.5, color='darkorange',
            arrowprops=dict(arrowstyle='->', color='darkorange', lw=1))

# --- Axes ---
y_max = fc_values.max() * 1.6
ax.set_ylim(0, y_max)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.set_ylabel('Monthly EV registrations')
ax.set_title(f'Monthly EV Registrations — Observed + {FINAL_MODEL_LABEL} Forecast to Dec 2027',
             fontsize=13, pad=12)
ax.legend(loc='upper left', fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'ev_forecast_2027.png'), dpi=150, bbox_inches='tight')
plt.show()

##### Forecast Chart — Interpretation

The chart above shows Spain's observed monthly EV registrations (Jan 2015 → Dec 2023) extended with the SARIMA(1,1,1)(1,0,1,12) forecast through December 2027. The dashed vertical line marks the Jan 2024 forecast start; the dotted red line marks Jan 2027.

**Key takeaways:**

**Realistic growth trajectory:**
The model projects monthly registrations growing from **14,775 (Dec 2023)** to **31,718 (Dec 2027)** — a 2.1× increase over 4 years (~20% YoY). This is consistent with Spain's MOVES III subsidy targets and the EU's 2035 zero-emission mandate pushing OEMs to accelerate EV supply.

**Seasonal cycle preserved:**
The forecast carries forward the 12-month rhythm visible in the historical data — spring registration peaks (Mar–Jun driven by fleet renewals and dealer incentive periods), a summer/early-autumn dip (Aug–Sep), and a year-end acceleration (Nov–Dec). This seasonal structure is critical for infrastructure planning, as charging demand follows the same cycle.

**Smooth handoff at Jan 2024:**
The forecast connects seamlessly to the last observed value — there is no discontinuity or level jump, confirming the model has correctly learned the trend and level of the series.

**Confidence interval:**
The 95% CI (shaded orange) widens progressively as the horizon extends, as expected for any time series model. By 2027 the uncertainty band spans roughly 15k–47k registrations/month. This is why the **cumulative fleet total (Section 12)** — which sums across all forecast months — is a more reliable single number for infrastructure planning than any individual monthly point estimate.


#### Cumulative Fleet — `total_ev_projected_2027`

The datathon requires the **total EV fleet in Spain at end-2027**, not just new registrations.  
We estimate this by starting from the historical EV stock and then updating it month by month using forecasted registrations and a small attrition factor.

##### Source for the attrition assumption

We use the **ANFAC Informe Anual 2024** as the source for Spain's aggregate fleet turnover. On page 9, ANFAC reports:

- **Total vehicle fleet in 2023:** 30,722,465  
- **Total vehicle fleet in 2024:** 31,301,881  
- **Total vehicle registrations in 2024:** 1,219,267  

Source: ANFAC, *Informe Anual 2024*, section **Datos básicos del sector**, p. 9.

---

##### How the annual attrition rate is calculated

We derive annual withdrawals from the standard stock identity:

$$
\text{Ending fleet} = \text{Starting fleet} + \text{new registrations} - \text{withdrawals}
$$

Rearranging:

$$
\text{withdrawals} = \text{Starting fleet} + \text{new registrations} - \text{Ending fleet}
$$

Substituting ANFAC's 2024 values:

$$
\text{withdrawals} = 30{,}722{,}465 + 1{,}219{,}267 - 31{,}301{,}881 = 639{,}851
$$

Then the implied annual attrition rate is:

$$
\text{attrition rate} = \frac{639{,}851}{30{,}722{,}465} \approx 0.0208 = 2.08\%
$$

This value is used directly in the model (no rounding).

---

##### How it is used in the EV fleet forecast

The fleet stock is updated monthly using:

$$
\text{stock}_{t} = \text{stock}_{t-1} \times (1 - \text{monthly attrition}) + \widehat{y}_{t}
$$

where:

$$
\text{monthly attrition} = \frac{0.0208}{12} \approx 0.001736
$$

and $\widehat{y}_{t}$ is the forecasted number of new EV registrations in month $t$.

This means:
- the **existing fleet decays slightly each month** due to deregistrations, scrappage, or exports;
- **new EV registrations are added in full** each month.

---

##### Key assumption

We apply the **same 2.08% annual attrition rate** to the EV fleet as a proxy for Spain’s overall vehicle turnover.

---

##### Caveat

This is a conservative assumption because the **2.08% rate reflects the overall Spanish vehicle fleet**, which includes older internal-combustion vehicles with higher deregistration rates. The EV fleet is significantly newer on average, so real EV-specific attrition is likely lower.

As a result, this approach likely produces a **conservative lower-bound estimate** of the EV fleet in 2027, which is appropriate for infrastructure planning.

In [ ]:
# ANFAC Informe Anual 2024, p. 9
# Starting fleet (2023): 30,722,465
# Ending fleet (2024):   31,301,881
# New registrations:     1,219,267
#
# Implied withdrawals = starting_fleet + new_registrations - ending_fleet
#                      = 30,722,465 + 1,219,267 - 31,301,881
#                      = 639,851
#
# Implied annual attrition rate = withdrawals / starting_fleet
#                               = 639,851 / 30,722,465
#                               ≈ 0.0208 = 2.08%

ANFAC_FLEET_2023 = 30_722_465
ANFAC_FLEET_2024 = 31_301_881
ANFAC_NEW_REG_2024 = 1_219_267

ANFAC_WITHDRAWALS_2024 = ANFAC_FLEET_2023 + ANFAC_NEW_REG_2024 - ANFAC_FLEET_2024
ANNUAL_ATTRITION_RATE = ANFAC_WITHDRAWALS_2024 / ANFAC_FLEET_2023   # ≈ 2.08%
MONTHLY_ATTRITION = ANNUAL_ATTRITION_RATE / 12

# Cumulative stock from all historical EV registrations (baseline stock at forecast start)
historic_stock = monthly["ev_reg"].sum()

# Project forward month by month:
# existing stock decays slightly due to attrition,
# then forecasted new EV registrations are added
stock = historic_stock
monthly_stock = []

for date, new_reg in zip(fc_values.index, fc_values.values):
    stock = stock * (1 - MONTHLY_ATTRITION) + max(new_reg, 0)
    monthly_stock.append({
        "date": date,
        "cumulative_fleet": round(stock)
    })

stock_df = pd.DataFrame(monthly_stock).set_index("date")

# Main output: projected EV fleet at December 2027
total_ev_projected_2027 = int(stock_df.loc["2027-12-01", "cumulative_fleet"])

print("=" * 65)
print(f"total_ev_projected_2027 = {total_ev_projected_2027:,}")
print("=" * 65)
print(f"Historic EV stock at forecast start: {historic_stock:,}")
print(f"Forecasted new registrations (2025-2027): {int(fc_values.sum()):,}")
print(f"ANFAC implied withdrawals in 2024: {ANFAC_WITHDRAWALS_2024:,}")
print(f"Annual attrition rate used: {ANNUAL_ATTRITION_RATE:.4%}")
print(f"Monthly attrition rate used: {MONTHLY_ATTRITION:.4%}")
print()
print(stock_df.tail(12).to_string())

##### Interpretation

Using the ANFAC-derived annual attrition rate of **2.08%**, the projected cumulative EV fleet in Spain reaches **1,412,640 vehicles by December 2027**.

This result suggests that Spain's EV fleet is expected to continue growing very strongly over the forecast horizon, even after accounting for vehicle withdrawals. Starting from a historical EV stock of **460,199 vehicles** at the forecast cutoff, the model adds forecasted monthly registrations while applying a small monthly decay factor to the existing fleet. Over 2025 to 2027, forecasted new registrations total **1,024,432 vehicles**, which largely outweighs the effect of attrition.

The projected growth is therefore driven mainly by **new EV adoption**, not by replacement of older EVs. Attrition plays only a modest role in reducing the stock because the monthly attrition rate is very small (**about 0.1736% per month**) and the EV fleet is still relatively young.

A key implication is that Spain's plug-in EV fleet is projected to grow from about **460 thousand** vehicles to more than **1.4 million** vehicles in just three years. This represents roughly a **tripling of the fleet size**, pointing to a rapid acceleration in electrification.

This estimate should also be interpreted as **conservative**. The attrition rate is derived from the overall Spanish vehicle fleet, which includes many older internal-combustion vehicles with higher deregistration rates than EVs. Since the EV fleet is newer on average, the true EV-specific attrition rate is likely lower. As a result, the true EV fleet in 2027 may be slightly higher than this projection.

Overall, the result supports the conclusion that Spain will likely experience a substantial increase in EV stock by 2027, with important implications for **charging infrastructure, grid readiness, and long-term mobility planning**.

In [ ]:
# Cumulative fleet chart
fig, ax = plt.subplots(figsize=(15, 6))

# Historical cumulative fleet
hist_cumulative = monthly["ev_reg"].cumsum()
ax.plot(
    hist_cumulative.index,
    hist_cumulative.values,
    label="Historical cumulative fleet",
    color="steelblue",
    lw=2.5
)

# Projected cumulative fleet
ax.plot(
    stock_df.index,
    stock_df["cumulative_fleet"],
    label=f"Projected fleet ({best_model_name})",
    color="darkorange",
    lw=2.5,
    linestyle="--"
)

# Forecast start marker
forecast_start = stock_df.index[0]
ax.axvline(forecast_start, color="gray", linestyle=":", lw=1.5, alpha=0.8)
ax.text(
    forecast_start,
    ax.get_ylim()[1] * 0.95,
    " Forecast start",
    color="gray",
    fontsize=10,
    va="top",
    ha="left"
)

# Shade forecast region
ax.axvspan(stock_df.index[0], stock_df.index[-1], alpha=0.08, color="orange")

# Final projected value line
ax.axhline(total_ev_projected_2027, color="red", lw=1.2, linestyle=":", alpha=0.7)

# Highlight final point
ax.scatter(
    stock_df.index[-1],
    total_ev_projected_2027,
    color="red",
    s=60,
    zorder=5
)

# Annotation for Dec 2027
ax.annotate(
    f"Dec 2027: {total_ev_projected_2027:,}",
    xy=(stock_df.index[-1], total_ev_projected_2027),
    xytext=(-90, 20),
    textcoords="offset points",
    fontsize=10,
    color="red",
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="red", alpha=0.9),
    arrowprops=dict(arrowstyle="->", color="red", lw=1.2)
)

# Titles and labels
ax.set_title("Cumulative EV Fleet in Spain: Historical and Projected to 2027", fontsize=14, pad=14)
ax.set_ylabel("Total plug-in EVs on road", fontsize=11)
ax.set_xlabel("Year", fontsize=11)

# X-axis formatting
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

# Y-axis formatting
ax.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M" if x >= 1e6 else f"{x/1e3:.0f}K")
)

# Grid styling
ax.grid(True, which="major", axis="both", linestyle="--", alpha=0.3)

# Better y-limits with padding
ymax = max(hist_cumulative.max(), stock_df["cumulative_fleet"].max())
ax.set_ylim(0, ymax * 1.12)

# Legend
ax.legend(frameon=False, loc="upper left")

# Clean up borders
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "ev_fleet_2027.png"), dpi=150, bbox_inches="tight")
plt.show()

##### Interpretation — Cumulative EV Fleet Projection

The historical data shows a **strong and accelerating adoption of plug-in electric vehicles (EVs)** in Spain over the past decade. The cumulative fleet grows slowly until around 2019, followed by a clear **inflection point after 2020**, where adoption accelerates significantly. This reflects policy support, increased model availability, and rising consumer demand.

##### Key insights from the projection

- The EV fleet is projected to reach approximately **1.41 million vehicles by December 2027**.
- This represents a **~3x increase** compared to the current fleet size (~450k–500k in 2024).
- Growth is **non-linear (convex)**, indicating continued acceleration in adoption rather than saturation within the forecast horizon.

##### Impact of attrition

- The model incorporates an **annual attrition rate of ~2.08%**, derived from ANFAC 2024 fleet statistics.
- Attrition has a **limited impact on total EV stock** because:
  - The EV fleet is relatively young
  - New registrations significantly outweigh vehicle withdrawals
- As a result, fleet growth remains **strongly driven by new registrations**, not constrained by turnover.

##### Structural interpretation

- The transition from historical to forecasted values (2024 onward) shows **continuity with past trends**, suggesting the SARIMA model captures the underlying dynamics well.
- The widening gap between years highlights a **compounding effect**, where each year's additions build on an already growing base.

##### Implications for infrastructure and planning

- The projected fleet size implies a **substantial increase in demand for charging infrastructure**, grid capacity, and energy management systems.
- Since this estimate is based on a **conservative attrition assumption**, the true EV fleet may be even larger, reinforcing the need for proactive infrastructure planning.

##### Conclusion

Spain’s EV adoption is entering a **high-growth phase**, with the total fleet expected to exceed **1.4 million vehicles by 2027**. Even under conservative assumptions, the trajectory indicates a **rapid electrification of the vehicle parc**, with significant implications for energy, urban planning, and mobility ecosystems.

---

#### EV Fleet 2027 by Category (BEV / PHEV / REEV+FCEV)

The total fleet projection (1,412,640) is split by vehicle category using **SARIMA(1,1,1)(1,0,1,12)** — the same winning model used for the national total — fitted independently on each category's monthly registration series (2015–2024).

**Why HEV is excluded:**  
HEV (standard hybrid) charges exclusively via regenerative braking — it never connects to a public charger. Including HEV would inflate our demand estimate with vehicles that generate **zero** charging infrastructure need. Only BEV, PHEV, REEV and FCEV are counted.

**Methodology:**
1. Fit `SARIMA(1,1,1)(1,0,1,12)` independently on the BEV, PHEV, and REEV+FCEV monthly registration series
2. If SARIMA fails on REEV+FCEV (sparse data, convergence issues), fall back to **Holt-Winters with additive trend** — never a flat mean
3. Apply the same attrition accumulation logic as Section 12 to each category to get the end-2027 fleet per category
4. Scale the three category totals so they sum exactly to `total_ev_projected_2027` (1,412,640) — fully consistent with the main forecast

In [ ]:
# ── SARIMA per category — same spec as the winning total model ───────────
# All three categories use SARIMA(1,1,1)(1,0,1,12).
# If REEV+FCEV SARIMA produces an invalid fit (rare, sparse data),
# Holt-Winters is used as fallback — never a flat mean.

from statsmodels.tsa.holtwinters import ExponentialSmoothing

SARIMA_ORDER          = (1, 1, 1)
SARIMA_SEASONAL_ORDER = (1, 0, 1, 12)   # D=0: same as the winning model
HORIZON               = 48               # Jan 2024 → Dec 2027
ATTRITION_RATE        = 0.0208           # ANFAC 2024, same as Section 12
TOTAL_FLEET_2027      = 1_412_640        # from Section 12

# Build monthly series per category
reev_monthly = (monthly['ev_reg'] - monthly['bev_reg'] - monthly['phev_reg']).asfreq('MS')
cat_hist = {
    'BEV':       monthly['bev_reg'].asfreq('MS'),
    'PHEV':      monthly['phev_reg'].asfreq('MS'),
    'REEV+FCEV': reev_monthly,
}

cat_forecasts = {}   # category → pd.Series of monthly forecast values
cat_methods   = {}   # category → model name used

for cat_label, series in cat_hist.items():
    log_s = np.log1p(series)
    try:
        fit = SARIMAX(
            log_s,
            order=SARIMA_ORDER,
            seasonal_order=SARIMA_SEASONAL_ORDER,
            enforce_stationarity=False,
            enforce_invertibility=False,
        ).fit(disp=False)
        fc_log = fit.get_forecast(steps=HORIZON).predicted_mean
        fc_vals = np.expm1(fc_log).clip(lower=0)
        cat_methods[cat_label] = 'SARIMA(1,1,1)(1,0,1,12)'
    except Exception as e:
        # Fallback: Holt-Winters (additive trend, no seasonal on sparse data)
        hw = ExponentialSmoothing(
            series.clip(lower=0),
            trend='add',
            seasonal=None,
            initialization_method='estimated',
        ).fit(optimized=True)
        fc_idx  = pd.date_range(start=series.index[-1] + pd.offsets.MonthBegin(),
                                periods=HORIZON, freq='MS')
        fc_vals = pd.Series(hw.forecast(HORIZON).clip(lower=0).values, index=fc_idx)
        cat_methods[cat_label] = f'Holt-Winters (SARIMA failed: {e})'

    cat_forecasts[cat_label] = fc_vals
    print(f'{cat_label:12s} | {cat_methods[cat_label]:35s} | '
          f'2027 annual: {fc_vals[-12:].sum():>8,.0f}')

# ── Cumulative fleet per category at end-2027 (same attrition logic as Sec 12) ──
def cumulative_fleet(hist_series, forecast_series, attrition):
    """Roll annual registrations forward with attrition → end-of-year fleet."""
    fleet = 0.0
    for regs in hist_series.resample('YE').sum():
        fleet = fleet * (1 - attrition) + regs
    for regs in forecast_series.resample('YE').sum():
        fleet = fleet * (1 - attrition) + regs
    return fleet

raw_fleets = {
    cat: cumulative_fleet(cat_hist[cat], cat_forecasts[cat], ATTRITION_RATE)
    for cat in cat_hist
}

# ── Scale so categories sum exactly to TOTAL_FLEET_2027 ──────────────────
raw_total = sum(raw_fleets.values())
scaled    = {k: v / raw_total * TOTAL_FLEET_2027 for k, v in raw_fleets.items()}

bev_fleet   = round(scaled['BEV'])
phev_fleet  = round(scaled['PHEV'])
other_fleet = TOTAL_FLEET_2027 - bev_fleet - phev_fleet

bev_share_2027   = bev_fleet   / TOTAL_FLEET_2027 * 100
phev_share_2027  = phev_fleet  / TOTAL_FLEET_2027 * 100
other_share_2027 = other_fleet / TOTAL_FLEET_2027 * 100

# ── Figure: category forecasts + 2027 fleet breakdown ────────────────────
COLORS = {'BEV': '#2E86AB', 'PHEV': '#F6AE2D', 'REEV+FCEV': '#A23B72'}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6),
                                gridspec_kw={'width_ratios': [3, 2]})
fig.suptitle('EV Registrations by Category — SARIMA Forecast to 2027',
             fontsize=13, fontweight='bold')

# Left: historical (solid) + forecast (dashed) per category
for cat_label in cat_hist:
    hist  = cat_hist[cat_label].resample('YE').sum()
    fc    = cat_forecasts[cat_label].resample('YE').sum()
    color = COLORS[cat_label]
    ax1.plot(hist.index.year, hist.values, marker='o', ms=5,
             color=color, lw=2, label=f'{cat_label} (actual)')
    ax1.plot(fc.index.year, fc.values, '--', marker='s', ms=4,
             color=color, lw=1.8,
             label=f'{cat_label} ({cat_methods[cat_label].split("(")[0].strip()})')

ax1.axvline(2023.5, color='grey', lw=0.8, ls=':', alpha=0.7)
ax1.text(2023.7, ax1.get_ylim()[1] * 0.97, 'forecast →',
         fontsize=8, color='grey', va='top')
ax1.set_xlabel('Year')
ax1.set_ylabel('Annual new registrations')
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{int(v):,}'))
ax1.legend(fontsize=8, ncol=2)
ax1.spines[['top', 'right']].set_visible(False)

# Right: 2027 fleet breakdown — horizontal bars
cats       = ['BEV', 'PHEV', 'REEV+FCEV']
fleet_vals = [bev_fleet, phev_fleet, other_fleet]
bars = ax2.barh(cats, fleet_vals,
                color=[COLORS[c] for c in cats], height=0.5)
for bar, val in zip(bars, fleet_vals):
    ax2.text(bar.get_width() + 5000,
             bar.get_y() + bar.get_height() / 2,
             f'{val:,.0f}  ({val / TOTAL_FLEET_2027 * 100:.1f}%)',
             va='center', fontsize=9, fontweight='bold')

ax2.set_xlabel('Projected EV fleet 2027')
ax2.set_title(f'Total fleet: {TOTAL_FLEET_2027:,.0f}', fontsize=10)
ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{int(v):,}'))
ax2.set_xlim(0, max(fleet_vals) * 1.4)
ax2.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'ev_category_breakdown_2027.png'),
            dpi=150, bbox_inches='tight')
plt.show()

# ── Summary ──────────────────────────────────────────────────────────────
print('EV Fleet 2027 — Category Breakdown')
print('=' * 50)
print(f'  BEV       : {bev_fleet:>9,.0f}   ({bev_share_2027:.1f}%)')
print(f'  PHEV      : {phev_fleet:>9,.0f}   ({phev_share_2027:.1f}%)')
print(f'  REEV+FCEV : {other_fleet:>9,.0f}   ({other_share_2027:.1f}%)')
print(f'  {"-"*40}')
print(f'  TOTAL     : {TOTAL_FLEET_2027:>9,.0f}   (100.0%)')
print()
print('Models used:')
for cat, method in cat_methods.items():
    print(f'  {cat:12s}: {method}')
print()
print('HEV excluded: charges only via regenerative braking — zero public charging demand.')

#### Summary Output

This section creates the final summary table used for **`File_1.csv`**.

The exported output contains the main forecasting result, model metadata, and the **category breakdown** derived in Section 12.5:

| Column | Description |
|---|---|
| `total_ev_projected_2027` | Projected cumulative EV fleet in Spain by December 2027 |
| `bev_fleet_2027` | BEV share of total fleet (SARIMA-derived) |
| `phev_fleet_2027` | PHEV share of total fleet (SARIMA-derived) |
| `reev_fcev_fleet_2027` | REEV + FCEV share of total fleet |
| `best_model` | Forecasting model selected based on evaluation performance |
| `model_rmse_test` | Out-of-sample RMSE of the selected model |
| `model_mape_test_pct` | Out-of-sample MAPE (%) of the selected model |
| `data_source` | Source used for historical EV registration data |
| `ev_scope` | EV categories included in the forecast |
| `attrition_rate_pct_yr` | Annual attrition rate applied to convert registrations to fleet stock |

In addition to the summary file, the full monthly forecast path (2025–2027) is exported for downstream use in visualisations and other notebooks.

In [ ]:
# ==============================
# 13. Summary Output for File 1
# ==============================

# Category breakdown from Section 12.5 (SARIMA-derived)
# These variables are guaranteed to exist if cells are run top-to-bottom.
# If Section 12.5 was skipped, they default to None so the CSV still saves.
try:
    _bev   = int(bev_fleet)
    _phev  = int(phev_fleet)
    _other = int(other_fleet)
except NameError:
    _bev = _phev = _other = None
    print('Warning: Section 12.5 not run — category columns will be None.')

summary = pd.DataFrame([{
    "total_ev_projected_2027": int(total_ev_projected_2027),
    "bev_fleet_2027":          _bev,
    "phev_fleet_2027":         _phev,
    "reev_fcev_fleet_2027":    _other,
    "best_model":              best_model_name,
    "model_rmse_test":         float(metrics_df.loc[best_model_name, "RMSE"]),
    "model_mape_test_pct":     float(metrics_df.loc[best_model_name, "MAPE %"]),
    "data_source":             "DGT Microdatos Matriculaciones 2015-2024",
    "ev_scope":                "BEV + PHEV + REEV + FCEV  (HEV excluded)",
    "attrition_rate_pct_yr":   round(ANNUAL_ATTRITION_RATE * 100, 4),
}])

print("=" * 65)
print("File 1 — Summary Scorecard Output")
print("=" * 65)
print(summary.T.to_string(header=False))
print("=" * 65)

# Monthly forecast detail for charts / downstream notebooks
fc_export = fc_values.reset_index().copy()
fc_export.columns = ["date", "monthly_ev_registrations_forecast"]
fc_export["monthly_ev_registrations_forecast"] = (
    fc_export["monthly_ev_registrations_forecast"].round().astype(int)
)

summary_path  = str(OUT / 'total_ev_projected_2027.csv')
forecast_path = str(OUT / 'ev_monthly_forecast_2025_2027.csv')

summary.to_csv(summary_path, index=False)
fc_export.to_csv(forecast_path, index=False)

print('\nSaved:')
print(f'  {summary_path}')
print(f'  {forecast_path}')

#### All Models Metrics Summary

This section compares the performance of all candidate forecasting models using multiple evaluation metrics.

##### Evaluation metrics

The models are assessed using:

- **RMSE (Root Mean Squared Error)** → primary selection metric (penalizes large errors)
- **MAE (Mean Absolute Error)** → average absolute deviation
- **MAPE (%) (Mean Absolute Percentage Error)** → relative error in percentage terms  
- **AIC / BIC** → model fit and complexity trade-off (used for reference)

---

##### Model selection

The **best model is selected based on RMSE**, as it captures overall prediction accuracy while penalizing large deviations more strongly.

A secondary ranking is also computed using the **average rank across RMSE, MAE, and MAPE**, to ensure robustness and avoid relying on a single metric.

---

##### Key result

- **Selected model:** `{{best_model_name}}`
- **Projected EV fleet (Dec 2027):** `{{total_ev_projected_2027}}`

---

In [ ]:
# ==============================
# 14. All Models Metrics Summary
# ==============================

print('┌' + '─' * 70 + '┐')
print(f'|{"FULL MODEL COMPARISON RESULTS":^70}|')
print('├' + '─' * 70 + '┤')

print(metrics_df.to_string())

print('├' + '─' * 70 + '┤')
print(f'| ✓ Best model (RMSE): {best_model_name:<42}|')

try:
    print(f'| ✓ total_ev_projected_2027 = {total_ev_projected_2027:<24,}|')
except NameError:
    print(f'| ✓ total_ev_projected_2027 = {"Not computed (run Section 12)":<24}|')

print('└' + '─' * 70 + '┘')

# Rank table
ranked = metrics_df.copy()

ranked['RMSE_rank'] = ranked['RMSE'].rank(method='min')
ranked['MAPE_rank'] = ranked['MAPE %'].rank(method='min')
ranked['MAE_rank']  = ranked['MAE'].rank(method='min')

ranked['avg_rank'] = ranked[['RMSE_rank', 'MAPE_rank', 'MAE_rank']].mean(axis=1)

print("\nRanking by average metric rank (lower = better):")
print(
    ranked[['RMSE', 'MAPE %', 'MAE', 'avg_rank']]
    .sort_values('avg_rank')
    .to_string()
)

##### Interpretation

- The selected model provides the **lowest RMSE**, indicating the best overall fit to the data.
- Other models may perform well on individual metrics (e.g., MAE or MAPE), but the chosen model achieves the **best balance across all metrics**.
- The ranking table confirms that the selected model is **consistently among the top performers**, not just an outlier on one metric.

---

In [ ]:
# ── Capture forecast output into shared constant ──────────────────────────────
# total_ev_projected_2027 is produced by the SARIMA fit above.
# TOTAL_EV_PROJECTED_2027 is the global constant used downstream in:
#   - Section 1.2 (province distribution)
#   - Section 3   (File 1 KPI scorecard)
try:
    TOTAL_EV_PROJECTED_2027 = int(total_ev_projected_2027)
    print(f'✓ TOTAL_EV_PROJECTED_2027 = {TOTAL_EV_PROJECTED_2027:,}')
except NameError:
    # Fallback: load from saved CSV if forecast cells were skipped
    try:
        _kpis = pd.read_csv(FLEET_CSV)
        TOTAL_EV_PROJECTED_2027 = int(_kpis['total_ev_projected_2027'].iloc[0])
        print(f'✓ Loaded from CSV: TOTAL_EV_PROJECTED_2027 = {TOTAL_EV_PROJECTED_2027:,}')
    except Exception:
        TOTAL_EV_PROJECTED_2027 = 0
        print('⚠  TOTAL_EV_PROJECTED_2027 = 0 (placeholder — run Section 1.1 to compute)')


---
### 1.2 Demand per Region

Distributes `TOTAL_EV_PROJECTED_2027` across Spain's 50 provinces using
DGT registration shares (3-year average 2021–2023).

**Key outputs:** `demand` DataFrame (indexed by province code) · `outputs/province_demand_2027.csv`


#### Province & ISO Code Mapping

The DGT uses 2-letter province codes (`M`, `B`, `V`, …).  
We map these to official province names and INE numeric codes for later use in BI tools.

In [ ]:
# DGT code → (Province name, INE numeric code, Autonomous Community)
#
# ALIAS NOTES — DGT dataset mixes two coding conventions:
#   • Car plate codes  (traditional): GE = Girona, OR = Ourense
#   • ISO/administrative codes (modern): GI = Girona, OU = Ourense
#   Both appear in the raw CSV files → mapped to same province (no records lost).
#
# INE CODE RECOVERY — some records (notably Navarra) leave COD_PROVINCIA_VEH
# blank but populate COD_MUNICIPIO_INE_VEH (5-digit, first 2 = INE province code).
# We build INE_TO_DGT below for that fallback.

PROVINCE_MAP = {
    'A':  ('Alicante/Alacant',         '03', 'Comunitat Valenciana'),
    'AB': ('Albacete',                  '02', 'Castilla-La Mancha'),
    'AL': ('Almería',                   '04', 'Andalucía'),
    'AV': ('Ávila',                     '05', 'Castilla y León'),
    'B':  ('Barcelona',                 '08', 'Cataluña'),
    'BA': ('Badajoz',                   '06', 'Extremadura'),
    'BI': ('Bizkaia',                   '48', 'País Vasco'),
    'BU': ('Burgos',                    '09', 'Castilla y León'),
    'C':  ('A Coruña',                  '15', 'Galicia'),
    'CA': ('Cádiz',                     '11', 'Andalucía'),
    'CC': ('Cáceres',                   '10', 'Extremadura'),
    'CE': ('Ceuta',                     '51', 'Ceuta'),
    'CO': ('Córdoba',                   '14', 'Andalucía'),
    'CR': ('Ciudad Real',               '13', 'Castilla-La Mancha'),
    'CS': ('Castellón/Castelló',        '12', 'Comunitat Valenciana'),
    'CU': ('Cuenca',                    '16', 'Castilla-La Mancha'),
    'GC': ('Las Palmas',                '35', 'Canarias'),
    'GE': ('Girona',                    '17', 'Cataluña'),   # plate code (canonical)
    'GI': ('Girona',                    '17', 'Cataluña'),   # ISO alias
    'GR': ('Granada',                   '18', 'Andalucía'),
    'GU': ('Guadalajara',               '19', 'Castilla-La Mancha'),
    'H':  ('Huelva',                    '21', 'Andalucía'),
    'HU': ('Huesca',                    '22', 'Aragón'),
    'IB': ('Illes Balears',             '07', 'Illes Balears'),
    'J':  ('Jaén',                      '23', 'Andalucía'),
    'L':  ('Lleida',                    '25', 'Cataluña'),
    'LE': ('León',                      '24', 'Castilla y León'),
    'LO': ('La Rioja',                  '26', 'La Rioja'),
    'LU': ('Lugo',                      '27', 'Galicia'),
    'M':  ('Madrid',                    '28', 'Comunidad de Madrid'),
    'MA': ('Málaga',                    '29', 'Andalucía'),
    'ML': ('Melilla',                   '52', 'Melilla'),
    'MU': ('Murcia',                    '30', 'Región de Murcia'),
    'NA': ('Navarra',                   '31', 'Navarra'),
    'O':  ('Asturias',                  '33', 'Asturias'),
    'OR': ('Ourense',                   '32', 'Galicia'),    # plate code (canonical)
    'OU': ('Ourense',                   '32', 'Galicia'),    # ISO alias
    'P':  ('Palencia',                  '34', 'Castilla y León'),
    'PM': ('Illes Balears (old code)',  '07', 'Illes Balears'),
    'PO': ('Pontevedra',                '36', 'Galicia'),
    'S':  ('Cantabria',                 '39', 'Cantabria'),
    'SA': ('Salamanca',                 '37', 'Castilla y León'),
    'SE': ('Sevilla',                   '41', 'Andalucía'),
    'SG': ('Segovia',                   '40', 'Castilla y León'),
    'SO': ('Soria',                     '42', 'Castilla y León'),
    'SS': ('Gipuzkoa',                  '20', 'País Vasco'),
    'T':  ('Tarragona',                 '43', 'Cataluña'),
    'TE': ('Teruel',                    '44', 'Aragón'),
    'TF': ('Santa Cruz de Tenerife',   '38', 'Canarias'),
    'TO': ('Toledo',                    '45', 'Castilla-La Mancha'),
    'V':  ('Valencia/València',         '46', 'Comunitat Valenciana'),
    'VA': ('Valladolid',                '47', 'Castilla y León'),
    'VI': ('Álava/Araba',              '01', 'País Vasco'),
    'Z':  ('Zaragoza',                  '50', 'Aragón'),
    'ZA': ('Zamora',                    '49', 'Castilla y León'),
}

# Aliases to normalise after filtering (so GI→GE and OU→OR merge into one row)
ALIAS_MAP = {'GI': 'GE', 'OU': 'OR', 'PM': 'IB'}

# Reverse mapping: INE 2-digit numeric code → canonical DGT plate code
# Used to recover province from COD_MUNICIPIO_INE_VEH (first 2 chars = INE code)
# Skip alias entries so each INE code maps to exactly one canonical plate code.
_ALIASES = set(ALIAS_MAP.keys())
INE_TO_DGT = {
    ine: code
    for code, (_, ine, _) in PROVINCE_MAP.items()
    if code not in _ALIASES
}

print(f'Province mappings : {len(PROVINCE_MAP)} entries ({len(_ALIASES)} aliases)')
print(f'INE→DGT lookup    : {len(INE_TO_DGT)} entries (used for municipality-code fallback)')
print(f'  e.g. INE 31 → {INE_TO_DGT.get("31")} (Navarra), INE 28 → {INE_TO_DGT.get("28")} (Madrid)')

#### Data Loading — EV Registrations by Province (2021–2023)

We load four columns to maximise province recovery, applying them in priority order:

| Priority | Column | Meaning |
|---|---|---|
| 1 | `COD_PROVINCIA_VEH` | Province of the vehicle owner (best — where the EV will be used) |
| 2 | `COD_PROVINCIA_MAT` | Province of the registration office (fallback 1) |
| 3 | `COD_MUNICIPIO_INE_VEH` first 2 digits | INE municipality code → province (fallback 2) |
| 4 | `CODIGO_POSTAL` first 2 digits | Postal code → province (last resort) |

Investigation found that all remaining null-province records (after fallbacks 1–2) have municipality code `31xxx` → **Navarra** — their foral registration system does not consistently populate the province column. Fallback 3 recovers all of them, leaving **0% unassigned** (excluding records with every geographic field blank).

##### Year Selection — Why 2021–2023?

Before fixing our analysis window, we do a full EDA across **all available years (2015–2023)**  
to understand how the EV market evolved and identify any years that should be excluded.

We look at two things:
1. **Volume trend** — is the market large enough to be representative?
2. **Province share stability** — do any years have anomalous geographic distributions?

In [ ]:
import matplotlib.gridspec as gridspec

ALL_YEARS = list(range(2015, 2024))

# ── Load minimal columns for all years ───────────────────────────────────────
_records = []
for fpath in sorted(glob.glob(os.path.join(DATA_DIR, '*.csv'))):
    yr = int(os.path.basename(fpath)[:4])
    if yr not in ALL_YEARS:
        continue
    df = pd.read_csv(fpath, sep=',', encoding='latin1',
                     usecols=['COD_PROVINCIA_VEH', 'COD_PROVINCIA_MAT',
                              'CATEGORÍA_VEHÍCULO_ELÉCTRICO', 'CLAVE_TRAMITE'],
                     dtype=str)
    df = df[df['CLAVE_TRAMITE'] == '1']
    df = df[df['CATEGORÍA_VEHÍCULO_ELÉCTRICO'].isin(EV_CATEGORIES)]
    df['year'] = yr
    _records.append(df)

_raw_all = pd.concat(_records, ignore_index=True)
_raw_all['COD_PROVINCIA_VEH'] = _raw_all['COD_PROVINCIA_VEH'].str.strip().str.upper()
_raw_all['COD_PROVINCIA_MAT'] = _raw_all['COD_PROVINCIA_MAT'].str.strip().str.upper()
_raw_all['COD_PROVINCIA_VEH'] = _raw_all['COD_PROVINCIA_VEH'].fillna(_raw_all['COD_PROVINCIA_MAT'])

# ── Annual totals ─────────────────────────────────────────────────────────────
yr_totals = _raw_all.groupby('year').size().reset_index(name='ev_regs')
yr_totals['yoy_pct'] = yr_totals['ev_regs'].pct_change() * 100

# ── Top-10 province shares per year ──────────────────────────────────────────
TOP_N = 10
share_rows = []
for yr, grp in _raw_all.dropna(subset=['COD_PROVINCIA_VEH']).groupby('year'):
    total = len(grp)
    vc = grp['COD_PROVINCIA_VEH'].value_counts().head(TOP_N)
    for prov, cnt in vc.items():
        share_rows.append({'year': yr, 'province': prov, 'share_pct': cnt / total * 100})

share_df   = pd.DataFrame(share_rows)
pivot_heat = share_df.pivot(index='province', columns='year', values='share_pct').fillna(0)
pivot_heat = pivot_heat.loc[pivot_heat.mean(axis=1).sort_values(ascending=False).index]

# ════════════════════════════════════════════════════════════════════════════════
# FIGURE 1 — Volume & Growth (2 rows × 1 column)
# ════════════════════════════════════════════════════════════════════════════════
fig1, (ax_vol, ax_growth) = plt.subplots(2, 1, figsize=(13, 10), sharex=True)
fig1.subplots_adjust(hspace=0.12)

bar_colors = ['#cccccc'] * 6 + ['#e63946', '#e63946', '#e63946']

# ── Panel 1: Annual EV registrations ─────────────────────────────────────────
bars = ax_vol.bar(yr_totals['year'], yr_totals['ev_regs'],
                  color=bar_colors, edgecolor='white', width=0.7)
ax_vol.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
ax_vol.set_title('Annual EV New Registrations — Spain 2015–2023', fontsize=12, fontweight='bold', pad=10)
ax_vol.set_ylabel('New registrations', fontsize=10)
ax_vol.grid(axis='y', alpha=0.3, linestyle='--')
ax_vol.spines[['top', 'right']].set_visible(False)

for bar, val in zip(bars, yr_totals['ev_regs']):
    ax_vol.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1200,
                f'{val/1000:.0f}k', ha='center', va='bottom', fontsize=9, fontweight='bold')

from matplotlib.patches import Patch
ax_vol.legend(handles=[
    Patch(facecolor='#cccccc', label='Excluded — anomalies or niche market'),
    Patch(facecolor='#e63946', label='Selected — 2021–2023')
], fontsize=9, loc='upper left', framealpha=0.9)

# ── Panel 2: YoY growth ───────────────────────────────────────────────────────
growth_colors = ['#e63946' if yr in YEARS else '#457b9d' for yr in yr_totals['year']]
ax_growth.bar(yr_totals['year'].iloc[1:], yr_totals['yoy_pct'].iloc[1:],
              color=growth_colors[1:], edgecolor='white', width=0.7)
ax_growth.axhline(0, color='black', linewidth=0.8)
ax_growth.set_ylabel('YoY growth (%)', fontsize=10)
ax_growth.set_title('Year-on-Year EV Growth Rate (%)', fontsize=12, fontweight='bold', pad=10)
ax_growth.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax_growth.grid(axis='y', alpha=0.3, linestyle='--')
ax_growth.spines[['top', 'right']].set_visible(False)
ax_growth.set_xticks(ALL_YEARS)
ax_growth.set_xticklabels(ALL_YEARS, fontsize=10)

for x, val in zip(yr_totals['year'].iloc[1:], yr_totals['yoy_pct'].iloc[1:]):
    ax_growth.text(x, val + 1.5, f'{val:.0f}%', ha='center', va='bottom',
                   fontsize=9, fontweight='bold')

fig1.suptitle('EDA — Spain EV Market Overview (2015–2023)',
              fontsize=14, fontweight='bold', y=1.01)
plt.show()

# ════════════════════════════════════════════════════════════════════════════════
# FIGURE 2 — Province share heatmap
# ════════════════════════════════════════════════════════════════════════════════
fig2, ax_heat = plt.subplots(figsize=(16, 9))

cmap_heat = plt.cm.YlOrRd
im = ax_heat.imshow(pivot_heat.values, aspect='auto', cmap=cmap_heat,
                    vmin=0, vmax=pivot_heat.values.max())

ax_heat.set_xticks(range(len(ALL_YEARS)))
ax_heat.set_xticklabels(ALL_YEARS, fontsize=11)
ax_heat.set_yticks(range(len(pivot_heat.index)))
ax_heat.set_yticklabels(
    [f"{code}  —  {(PROVINCE_MAP if 'PROVINCE_MAP' in vars() else {}).get(code, (code,))[0][:22]}" for code in pivot_heat.index],
    fontsize=10
)
ax_heat.set_title(
    'Top-10 Province Share of National EV Registrations by Year (%)\n'
    'Dark red = high share  |  Green outline = selected years (2021–2023)  |  Anomalies annotated',
    fontsize=12, fontweight='bold', pad=14
)

# Cell value labels
for i, prov in enumerate(pivot_heat.index):
    for j, yr in enumerate(ALL_YEARS):
        val = pivot_heat.loc[prov, yr] if yr in pivot_heat.columns else 0
        txt_color = 'white' if val > pivot_heat.values.max() * 0.6 else '#333'
        if val > 0:
            ax_heat.text(j, i, f'{val:.1f}%', ha='center', va='center',
                         fontsize=9, color=txt_color)

# Green outline on selected years
for j, yr in enumerate(ALL_YEARS):
    if yr in YEARS:
        rect = plt.Rectangle((j - 0.5, -0.5), 1, len(pivot_heat),
                              fill=False, edgecolor='#2a9d8f', linewidth=3.0, zorder=3)
        ax_heat.add_patch(rect)

fig2.colorbar(im, ax=ax_heat, fraction=0.015, pad=0.02, label='Share (%)')

# Annotate anomalies
prov_list = list(pivot_heat.index)
anomaly_annotations = []
if 'CA' in prov_list:
    anomaly_annotations.append(
        (prov_list.index('CA'), ALL_YEARS.index(2017), '← Cádiz fleet\nbatch (port)', '#c1121f')
    )
if 'M' in prov_list:
    anomaly_annotations.append(
        (prov_list.index('M'), ALL_YEARS.index(2018), '← Madrid\ncorporate orders', '#c1121f')
    )
if 'TO' in prov_list and 2020 in ALL_YEARS:
    anomaly_annotations.append(
        (prov_list.index('TO'), ALL_YEARS.index(2020), '← Toledo\nlogistics fleet', '#c1121f')
    )

for row_i, col_j, label, color in anomaly_annotations:
    ax_heat.annotate(label,
        xy=(col_j + 0.45, row_i), xytext=(col_j + 1.6, row_i),
        fontsize=8.5, color=color, fontweight='bold',
        arrowprops=dict(arrowstyle='->', color=color, lw=1.3))

fig2.suptitle('EDA — Province Share Heatmap (2015–2023)',
              fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('Annual registration volumes:')
print(yr_totals[['year', 'ev_regs', 'yoy_pct']].to_string(index=False))

##### Year Selection — Conclusion

The EDA above reveals three clear reasons to restrict the analysis window to **2021–2023**:

| Year(s) | Issue | Impact if included |
|---|---|---|
| 2015–2016 | EV market was tiny (2.9k–5.7k regs/yr) — niche early-adopter geography | Barcelona had 20–28% share; would overestimate BCN 2027 demand |
| 2017 | **Cádiz (CA) anomaly: 13.9% share** — a single fleet batch registered at the port | CA would receive 3–4× inflated 2027 demand allocation |
| 2018–2019 | **Madrid (M) spikes to 50–54%** — corporate/public bulk procurement orders | M would be systematically overestimated |
| 2020 | **Toledo (TO) spike: 2.2%** — Amazon logistics fleet at their hub | TO would receive ~3× its actual consumer demand |

**2021–2023** is the correct window because:
- ✅ Largest sample (327k records) — statistically robust
- ✅ No single-entity fleet anomalies distorting provincial geography
- ✅ Represents the **mainstream consumer EV market** that will characterise 2027 demand
- ✅ Section 4.2 confirms shares are stable across all three years (low std dev)

`YEARS = [2021, 2022, 2023]` is kept as defined in Section 1.

In [ ]:
records = []

csv_files    = sorted(glob.glob(os.path.join(DATA_DIR, '*.csv')))
target_files = [f for f in csv_files if any(str(yr) in os.path.basename(f) for yr in YEARS)]

print(f'Loading {len(target_files)} CSV files ({YEARS[0]}–{YEARS[-1]})...')

USECOLS = [PROV_COL, 'COD_PROVINCIA_MAT', 'COD_MUNICIPIO_INE_VEH', 'CODIGO_POSTAL', EV_COL, 'CLAVE_TRAMITE']

for fpath in target_files:
    fname = os.path.basename(fpath)
    yr, mo = int(fname[:4]), int(fname[5:7])
    try:
        df = pd.read_csv(fpath, sep=',', encoding='latin1', usecols=USECOLS, dtype=str)
        df = df[df['CLAVE_TRAMITE'] == '1'].copy()
        df = df[df[EV_COL].isin(EV_CATEGORIES)].copy()
        df['year']  = yr
        df['month'] = mo
        records.append(df)
    except Exception as e:
        print(f'  Warning: {fname} — {e}')

raw = pd.concat(records, ignore_index=True)

# Normalise all geographic columns
for col in [PROV_COL, 'COD_PROVINCIA_MAT']:
    raw[col] = raw[col].str.strip().str.upper()

# ── 4-tier province recovery ──────────────────────────────────────────────────
n_start = raw[PROV_COL].isna().sum()

# Tier 2: fallback to registration-office province
raw[PROV_COL] = raw[PROV_COL].fillna(raw['COD_PROVINCIA_MAT'].str.strip().str.upper())
n_after_mat = raw[PROV_COL].isna().sum()

# Tier 3: extract province from INE municipality code (first 2 digits = INE province)
def ine_mun_to_prov(mun_code):
    if pd.isna(mun_code):
        return None
    ine2 = str(mun_code).strip().zfill(5)[:2]
    return INE_TO_DGT.get(ine2)

still_null = raw[PROV_COL].isna()
raw.loc[still_null, PROV_COL] = raw.loc[still_null, 'COD_MUNICIPIO_INE_VEH'].map(ine_mun_to_prov)
n_after_mun = raw[PROV_COL].isna().sum()

# Tier 4: extract province from postal code (first 2 digits = INE province)
still_null = raw[PROV_COL].isna()
raw.loc[still_null, PROV_COL] = (
    raw.loc[still_null, 'CODIGO_POSTAL']
    .str.strip().str[:2]
    .map(INE_TO_DGT)
)
n_after_cp = raw[PROV_COL].isna().sum()

print(f'\nTotal EV new registrations  : {len(raw):,}')
print(f'Province recovery summary:')
print(f'  PROVINCIA_VEH null at start : {n_start:,}')
print(f'  Recovered via PROVINCIA_MAT : {n_start - n_after_mat:,}')
print(f'  Recovered via MUNICIPIO_INE : {n_after_mat - n_after_mun:,}  (all Navarra foral system)')
print(f'  Recovered via POSTAL code   : {n_after_mun - n_after_cp:,}')
print(f'  Still null (no geo data)    : {n_after_cp:,} ({n_after_cp/len(raw)*100:.2f}%)')
print(f'\nEV type breakdown:\n{raw[EV_COL].value_counts()}')

#### Province-Level EV Registrations

We aggregate EV registrations by province for each year, then compute a 3-year total and the percentage share of each province in national EV registrations.  
Provinces with unknown or null codes are grouped under `'XX'` (unassigned) and excluded from the final demand calculation.

In [ ]:
# Drop only records that are still null after the VEH→MAT fallback,
# OR that have an unrecognised province code after alias normalisation
valid = raw.dropna(subset=[PROV_COL]).copy()
valid = valid[valid[PROV_COL].isin(PROVINCE_MAP.keys())].copy()

dropped = len(raw) - len(valid)
print(f'Records dropped (both province columns null): {dropped:,} ({dropped/len(raw)*100:.2f}%)')

# Normalise aliases → canonical plate codes so GI→GE, OU→OR, PM→IB merge correctly
ALIAS_MAP = {'GI': 'GE', 'OU': 'OR', 'PM': 'IB'}
valid[PROV_COL] = valid[PROV_COL].replace(ALIAS_MAP)

# Aggregate: registrations per province per year
prov_year = (
    valid.groupby([PROV_COL, 'year'])
    .size()
    .reset_index(name='ev_registrations')
)

# Pivot: province × year
prov_pivot = prov_year.pivot(index=PROV_COL, columns='year', values='ev_registrations').fillna(0)
prov_pivot.columns = [str(c) for c in prov_pivot.columns]
prov_pivot['total_2021_2023'] = prov_pivot.sum(axis=1)
prov_pivot = prov_pivot.sort_values('total_2021_2023', ascending=False)

# Add metadata
prov_pivot['province_name']  = prov_pivot.index.map(lambda c: PROVINCE_MAP[c][0])
prov_pivot['ine_code']        = prov_pivot.index.map(lambda c: PROVINCE_MAP[c][1])
prov_pivot['auto_community'] = prov_pivot.index.map(lambda c: PROVINCE_MAP[c][2])
prov_pivot['share_pct']      = (prov_pivot['total_2021_2023'] / prov_pivot['total_2021_2023'].sum() * 100).round(4)

print(f'\nTotal EV registrations 2021-2023 (province assigned): {prov_pivot["total_2021_2023"].sum():,.0f}')
print(f'Share accounts for: {prov_pivot["share_pct"].sum():.2f}% of loaded registrations')
print(f'\nTop 15 provinces by EV registrations:')
display_cols = ['province_name', 'auto_community', '2021', '2022', '2023', 'total_2021_2023', 'share_pct']
print(prov_pivot[display_cols].head(15).to_string())

##### Top 20 Provinces

In [ ]:
top20 = prov_pivot.head(20).copy()
top20['label'] = top20.index + '\n' + top20['province_name'].str[:12]

# Colour by autonomous community (top ones get distinct colours)
community_palette = {
    'Comunidad de Madrid':    '#e63946',
    'Cataluña':               '#457b9d',
    'Comunitat Valenciana':   '#2a9d8f',
    'Andalucía':              '#f4a261',
    'País Vasco':             '#6a4c93',
    'Illes Balears':          '#a8dadc',
    'Canarias':               '#ffb703',
    'Aragón':                 '#8ecae6',
    'Castilla y León':        '#b5838d',
    'Galicia':                '#52b788',
    'Región de Murcia':       '#d62828',
    'Navarra':                '#023e8a',
    'Cantabria':              '#606c38',
    'La Rioja':               '#dda15e',
    'Asturias':               '#bc6c25',
    'Extremadura':            '#adb5bd',
    'Castilla-La Mancha':     '#ced4da',
}
default_color = '#cccccc'
colors = [community_palette.get(c, default_color) for c in top20['auto_community']]

fig, ax = plt.subplots(figsize=(16, 7))
bars = ax.bar(range(len(top20)), top20['total_2021_2023'], color=colors, edgecolor='white', linewidth=0.5)

ax.set_xticks(range(len(top20)))
ax.set_xticklabels(top20['label'], fontsize=8.5)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.set_ylabel('EV Registrations (new vehicles, 2021–2023)')
ax.set_title('Top 20 Provinces by EV Registrations — Spain 2021–2023\n(BEV + PHEV + REEV + FCEV, new registrations only)', fontsize=13)
ax.grid(axis='y', alpha=0.3, linestyle='--')

# Value labels on bars
for bar, val, share in zip(bars, top20['total_2021_2023'], top20['share_pct']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{val:,.0f}\n({share:.1f}%)', ha='center', va='bottom', fontsize=7.5, color='#333')

# Legend patches for communities shown
shown_communities = top20['auto_community'].unique()
legend_patches = [
    mpatches.Patch(color=community_palette.get(c, default_color), label=c)
    for c in shown_communities
]
ax.legend(handles=legend_patches, loc='upper right', fontsize=7.5, title='Autonomous Community',
          title_fontsize=8, framealpha=0.9)

# plt.tight_layout()
plt.show()

##### Year-on-Year Growth by Province

Checking whether the province share is stable over time — if it is, the 3-year average is a reliable proxy for 2027.

In [ ]:
top15 = prov_pivot.head(15).copy()

year_totals = {str(yr): prov_year[prov_year['year'] == yr]['ev_registrations'].sum() for yr in YEARS}

for yr in YEARS:
    top15[f'share_{yr}'] = (top15[str(yr)] / year_totals[str(yr)] * 100).round(2)

share_cols = [f'share_{yr}' for yr in YEARS]
top15['share_std'] = top15[share_cols].std(axis=1).round(2)

fig, ax = plt.subplots(figsize=(16, 7))
x     = np.arange(len(top15))
width = 0.25
year_colors = ['#457b9d', '#2a9d8f', '#e63946']

for i, yr in enumerate(YEARS):
    bars = ax.bar(x + i*width, top15[f'share_{yr}'], width, label=str(yr),
                  color=year_colors[i], alpha=0.85)
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, h + 0.12,
                    f'{h:.1f}%', ha='center', va='bottom', fontsize=6.5, color='#333')

ax.set_xticks(x + width)
ax.set_xticklabels([f"{code}\n{nm[:10]}" for code, nm in
                    zip(top15.index, top15['province_name'])], fontsize=8.5)
ax.set_ylabel('Province share of national EV registrations (%)')
ax.set_title('Province Share of National EV Registrations by Year — Top 15\n'
             '(Stability check: consistent share → valid 3-year proxy for 2027)', fontsize=12)
ax.legend(title='Year')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

print('Share stability (std dev across years — lower = more stable):')
print(top15[['province_name'] + share_cols + ['share_std']].to_string(index=True))

# ── Sensitivity: show what would change if we included 2017 (Cádiz anomaly) ──
print('\n--- Sensitivity: why earlier years distort results ---')
print('Province shares in 2017 vs 2021-2023 average (top anomalies):')
print('  CA (Cádiz) 2017: 13.9%  vs  ~1.5% in 2021-2023  ← fleet batch at port')
print('  M  (Madrid) 2018-2019: 50-54%  vs  ~45% in 2021-2023  ← corporate bulk orders')
print('  TO (Toledo) 2020: 2.2%  vs  ~0.6% in 2021-2023  ← Amazon logistics fleet')
print('\nConclusion: 2021-2023 is the most representative window for mainstream 2027 demand.')

#### Apply 2027 Forecast → Province-Level Fleet

**Method:** We load the `total_ev_projected_2027` value from notebook 2.1's output,  
then allocate it to provinces proportionally to their 3-year registration share.

**Assumption documented:** The province share of new registrations (2021–2023 average) is a stable  
proxy for the cumulative EV fleet distribution in 2027. This holds if:  
1. Inter-province migration of EV owners is negligible  
2. No province experiences a structural shift in EV adoption rate between 2024–2027  

Both are conservative assumptions; urban hotspots (Madrid, Barcelona) may slightly underestimate  
their 2027 share due to faster adoption growth.

In [ ]:
# Load total fleet projection from notebook 2.1
fleet_kpis = pd.read_csv(FLEET_CSV)
TOTAL_EV_2027 = int(fleet_kpis['total_ev_projected_2027'].iloc[0])
SOURCE_MODEL  = fleet_kpis['best_model'].iloc[0]

print(f'Total EV fleet projected for 2027 : {TOTAL_EV_2027:,}')
print(f'Source model (notebook 2.1)        : {SOURCE_MODEL}')

# Apply share to get province-level fleet
demand = prov_pivot[['province_name', 'ine_code', 'auto_community',
                      'total_2021_2023', 'share_pct']].copy()
demand.index.name = 'province_code'

demand['ev_fleet_2027'] = (demand['share_pct'] / 100 * TOTAL_EV_2027).round(0).astype(int)

# Reconcile rounding: add any remainder to the largest province
diff = TOTAL_EV_2027 - demand['ev_fleet_2027'].sum()
demand.iloc[0, demand.columns.get_loc('ev_fleet_2027')] += diff

demand = demand.sort_values('ev_fleet_2027', ascending=False)

print(f'\nSum check — province totals: {demand["ev_fleet_2027"].sum():,} vs national: {TOTAL_EV_2027:,}')
print(f'\nProvince-level EV fleet projection for 2027:')
print(demand[['province_name', 'auto_community', 'share_pct', 'ev_fleet_2027']].head(20).to_string())

##### Projected EV Fleet by Province — 2027

In [ ]:
top_n = demand.head(20).copy()
colors2 = [community_palette.get(c, default_color) for c in top_n['auto_community']]

fig, ax = plt.subplots(figsize=(16, 7))
bars = ax.bar(range(len(top_n)), top_n['ev_fleet_2027'], color=colors2, edgecolor='white', linewidth=0.5)

labels = [
    code + '\n' + nm[:12]
    for code, nm in zip(top_n.index, top_n['province_name'])
]
ax.set_xticks(range(len(top_n)))
ax.set_xticklabels(labels, fontsize=8.5)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.set_ylabel('Projected EV Fleet (vehicles)')
ax.set_title(
    f'Projected EV Fleet by Province — Spain End 2027\n'
    f'National total: {TOTAL_EV_2027:,} EVs | Source: {SOURCE_MODEL} (notebook 2.1)',
    fontsize=12
)
ax.grid(axis='y', alpha=0.3, linestyle='--')

for bar, val, share in zip(bars, top_n['ev_fleet_2027'], top_n['share_pct']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
            f'{val:,}\n({share:.1f}%)', ha='center', va='bottom', fontsize=7.5, color='#333')

shown_communities2 = top_n['auto_community'].unique()
legend_patches2 = [
    mpatches.Patch(color=community_palette.get(c, default_color), label=c)
    for c in shown_communities2
]
ax.legend(handles=legend_patches2, loc='upper right', fontsize=7.5, title='Autonomous Community',
          title_fontsize=8, framealpha=0.9)

plt.tight_layout()
plt.show()

#### Autonomous Community Aggregation

In addition to province-level data, we aggregate to **Autonomous Community** level.  
This gives the regional demand picture that feeds into the strategic network design  
(where to prioritise infrastructure investment at a macro level).

In [ ]:
community_demand = (
    demand
    .groupby('auto_community')
    .agg(
        n_provinces=('ev_fleet_2027', 'count'),
        ev_fleet_2027=('ev_fleet_2027', 'sum'),
        share_pct=('share_pct', 'sum')
    )
    .sort_values('ev_fleet_2027', ascending=False)
    .reset_index()
)

print('Autonomous Community demand (2027):')
print(community_demand.to_string(index=False))

# Horizontal bar chart
fig, ax = plt.subplots(figsize=(12, 8))
comm_colors = [community_palette.get(c, default_color) for c in community_demand['auto_community']]
bars = ax.barh(community_demand['auto_community'], community_demand['ev_fleet_2027'],
               color=comm_colors, edgecolor='white')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.set_xlabel('Projected EV Fleet 2027')
ax.set_title(
    f'Projected EV Fleet by Autonomous Community — Spain 2027\nNational total: {TOTAL_EV_2027:,}',
    fontsize=12
)
ax.invert_yaxis()

for bar, val, share in zip(bars, community_demand['ev_fleet_2027'], community_demand['share_pct']):
    ax.text(bar.get_width() + 1000, bar.get_y() + bar.get_height()/2,
            f'{val:,}  ({share:.1f}%)', va='center', fontsize=9)

ax.grid(axis='x', alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

#### EV Type Mix by Province

Understanding the BEV vs PHEV breakdown per province matters for charger type selection:  
- **BEV** drivers need full charging (AC + DC fast chargers on interurban routes)  
- **PHEV/REEV** drivers can use ICE for long trips, but benefit from AC charging at rest stops  

A province with a high BEV share warrants prioritisation for DC fast chargers on interurban highways.

In [ ]:
# EV type mix by province
type_mix = (
    valid.groupby([PROV_COL, EV_COL])
    .size()
    .reset_index(name='count')
)
type_pivot = type_mix.pivot(index=PROV_COL, columns=EV_COL, values='count').fillna(0)

top10_codes = demand.head(10).index.tolist()
type_top10 = type_pivot.loc[top10_codes].copy()
type_top10['total'] = type_top10.sum(axis=1)

# Convert to percentage
for col in EV_CATEGORIES:
    if col in type_top10.columns:
        type_top10[col + '_pct'] = type_top10[col] / type_top10['total'] * 100

type_colors = {'BEV': '#e63946', 'PHEV': '#457b9d', 'REEV': '#2a9d8f', 'FCEV': '#f4a261'}
pct_cols = [c + '_pct' for c in EV_CATEGORIES if c + '_pct' in type_top10.columns]
plot_cats = [c.replace('_pct', '') for c in pct_cols]

fig, ax = plt.subplots(figsize=(14, 6))
bottom = np.zeros(len(type_top10))
x_labels = [f"{code}\n{PROVINCE_MAP[code][0][:12]}" for code in type_top10.index]

for cat in plot_cats:
    col = cat + '_pct'
    if col in type_top10.columns:
        ax.bar(range(len(type_top10)), type_top10[col], bottom=bottom,
               label=cat, color=type_colors.get(cat, '#ccc'), edgecolor='white')
        bottom += type_top10[col].values

ax.set_xticks(range(len(type_top10)))
ax.set_xticklabels(x_labels, fontsize=9)
ax.set_ylabel('Share of EV registrations (%)')
ax.set_ylim(0, 105)
ax.set_title('EV Type Mix by Province — Top 10 Provinces (2021–2023)', fontsize=12)
ax.legend(loc='upper right', title='EV Type')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(50, color='black', linewidth=0.5, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

print('BEV share by province (top 10):')
if 'BEV_pct' in type_top10.columns:
    for code in type_top10.index:
        print(f'  {code} ({PROVINCE_MAP[code][0][:20]:20s}): BEV = {type_top10.loc[code, "BEV_pct"]:.1f}%')

#### Interactive Province Map — EV Demand Distribution 2027

Each province is coloured by its projected EV fleet in 2027 on a **linear scale**.
Hover over any province to see the province name, EV fleet size, national share, and autonomous community.

**Controls:**
- **Scroll / pinch** — zoom in and out
- **Click + drag** — pan across the map
- **Hover** — see province details in a tooltip

Canary Islands, Ceuta and Melilla appear at their real geographic coordinates.
The GeoJSON boundary data is fetched from a public repository; an internet connection is required to render the map.

In [ ]:

# ── GeoJSON source (public, no API key needed) ───────────────────────────────
GEOJSON_URL = (
    "https://raw.githubusercontent.com/codeforgermany/"
    "click_that_hood/main/public/data/spain-provinces.geojson"
)

# ── Name normalisation: lowercase + strip diacritics ─────────────────────────
def _norm(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', str(s).lower().strip())
        if unicodedata.category(c) != 'Mn'
    )

# Build normalised-name → DGT code lookup from PROVINCE_MAP
name_to_code = {
    _norm(pname): code
    for code, (pname, ine, _) in PROVINCE_MAP.items()
    if code not in _ALIASES
}

# Manual overrides — covers bilingual GeoJSON names (Basque/Valencian + Spanish)
# and other naming variants
name_to_code.update({
    # A Coruña variants
    'a coruna':                  'C',
    'la coruna':                 'C',
    # Basque Country — GeoJSON uses co-official names first
    'araba/alava':               'VI',   # Araba/Álava
    'gipuzkoa/guipuzcoa':        'SS',   # Gipuzkoa/Guipúzcoa
    'bizkaia/vizcaya':           'BI',   # Bizkaia/Vizcaya
    # Valencian Community — GeoJSON uses Valencian name first
    'alacant/alicante':          'A',    # Alacant/Alicante
    'castello/castellon':        'CS',   # Castelló/Castellón
    # Other bilingual / alias variants
    'illes balears':             'IB',
    'islas baleares':            'IB',
    'las palmas':                'GC',
    'santa cruz de tenerife':    'TF',
    'girona':                    'GE',
    'ourense':                   'OR',
    'alicante':                  'A',
    'alacant':                   'A',
})

# ── Fetch province boundaries ─────────────────────────────────────────────────
try:
    geo = requests.get(GEOJSON_URL, timeout=20).json()
except Exception as exc:
    raise RuntimeError(
        f"Could not download GeoJSON. Check internet connection.\n{exc}"
    ) from exc

unmatched = []
for feat in geo['features']:
    raw_name = feat['properties'].get('name', '')
    code = name_to_code.get(_norm(raw_name))
    feat['id'] = code
    if code is None:
        unmatched.append(raw_name)

if unmatched:
    print(f"Warning: {len(unmatched)} province(s) not matched: {unmatched}")
else:
    print(f"All {len(geo['features'])} GeoJSON provinces matched successfully.")

# ── Prepare map DataFrame ─────────────────────────────────────────────────────
map_df = demand.reset_index()[
    ['province_code', 'province_name', 'auto_community', 'share_pct', 'ev_fleet_2027']
].copy()
map_df['share_pct']       = map_df['share_pct'].round(2)
map_df['rank']            = range(1, len(map_df) + 1)
map_df['top10']           = map_df['rank'] <= 10

# Log-scale colour (raw values shown in hover)
map_df['log_fleet'] = np.log10(map_df['ev_fleet_2027'].clip(lower=1))

log_min = map_df['log_fleet'].min()
log_max = map_df['log_fleet'].max()
tick_vals = [log_min + i * (log_max - log_min) / 5 for i in range(6)]
tick_text = [f"{10**v:,.0f}" for v in tick_vals]

# ── Interactive choropleth ────────────────────────────────────────────────────
fig = px.choropleth_mapbox(
    map_df,
    geojson=geo,
    locations='province_code',
    featureidkey='id',
    color='log_fleet',
    color_continuous_scale='YlOrRd',
    mapbox_style='carto-positron',
    zoom=4.5,
    center={'lat': 40.2, 'lon': -3.5},
    opacity=0.78,
    hover_name='province_name',
    hover_data={
        'province_code':  True,
        'rank':           True,
        'ev_fleet_2027':  ':,.0f',
        'share_pct':      ':.2f',
        'auto_community': True,
        'log_fleet':      False,
        'top10':          False,
    },
    title='Projected EV Fleet by Province — Spain 2027  (scroll to zoom · hover for details)',
    labels={
        'ev_fleet_2027':  'EV Fleet 2027',
        'share_pct':      'National Share (%)',
        'auto_community': 'Autonomous Community',
        'province_code':  'Province Code',
        'rank':           'National Rank',
    },
    width=1050,
    height=660,
)

# ── Outline the top-10 provinces with a bold border ──────────────────────────
top10_codes = set(map_df[map_df['top10']]['province_code'])
top10_geo   = {
    'type': 'FeatureCollection',
    'features': [f for f in geo['features'] if f.get('id') in top10_codes]
}

fig.add_trace(
    go.Choroplethmapbox(
        geojson=top10_geo,
        locations=[f['id'] for f in top10_geo['features']],
        z=[1] * len(top10_geo['features']),
        colorscale=[[0, 'rgba(0,0,0,0)'], [1, 'rgba(0,0,0,0)']],  # transparent fill
        marker=dict(line=dict(color='#1a1a2e', width=2.5)),
        showscale=False,
        hoverinfo='skip',
        name='Top 10',
    )
)

fig.update_layout(
    margin={'r': 0, 't': 50, 'l': 0, 'b': 0},
    coloraxis_colorbar=dict(
        title='EV Fleet 2027<br>(log scale)',
        tickvals=tick_vals,
        ticktext=tick_text,
        len=0.7,
        thickness=14,
    ),
    font=dict(family='Arial', size=12),
    legend_title_text='',
)

fig.show()

# ── Top-10 summary table ──────────────────────────────────────────────────────
print("\n Top 10 Priority Provinces — Projected EV Fleet 2027")
print("=" * 62)
top10_df = map_df[map_df['top10']][
    ['rank', 'province_code', 'province_name', 'auto_community', 'ev_fleet_2027', 'share_pct']
].copy()
top10_df.columns = ['Rank', 'Code', 'Province', 'Community', 'EV Fleet 2027', 'Share (%)']
print(top10_df.to_string(index=False))

#### Output — `province_demand_2027.csv`

This file is the key output of notebook 2.2.  
It will be imported by the charging station placement notebook to weight demand by province.

In [ ]:
# Build clean output dataframe
output = demand.reset_index()[[
    'province_code', 'province_name', 'ine_code', 'auto_community',
    'total_2021_2023', 'share_pct', 'ev_fleet_2027'
]].copy()

output = output.sort_values('ev_fleet_2027', ascending=False).reset_index(drop=True)
output['rank'] = output.index + 1

# Add BEV share (useful for charger type weighting downstream)
if 'BEV' in type_pivot.columns:
    output['bev_share_pct'] = output['province_code'].map(
        lambda c: round(type_pivot.loc[c, 'BEV'] / type_pivot.loc[c].sum() * 100, 2)
        if c in type_pivot.index else None
    )
else:
    output['bev_share_pct'] = None

out_path = os.path.join(OUT_DIR, 'province_demand_2027.csv')
output.to_csv(out_path, index=False, encoding='utf-8')

print(f'Saved: {out_path}')
print(f'Rows: {len(output)} provinces')
print(f'BEV share populated: {output["bev_share_pct"].notna().sum()} / {len(output)} provinces')
print()
print(output.to_string(index=False))

#### Summary

##### Methodology
| Step | Detail |
|---|---|
| Data source | DGT Microdatos Matriculaciones — CSV files 2021–2023 (mandatory datos.gob.es fork) |
| Filter | New registrations only (`CLAVE_TRAMITE = 1`), EV types: BEV, PHEV, REEV, FCEV |
| Province column | `COD_PROVINCIA_VEH` (4-tier recovery: VEH → MAT → INE municipality → postal code) |
| Share method | 3-year total registrations per province ÷ national total (2021–2023) |
| National forecast | `total_ev_projected_2027` from notebook 2.1 (SARIMA(1,1,1)(1,0,1,12)) = **1,412,640 EVs** |
| Province fleet | Province share × national total, rounded to integer |
| Visualisation | Interactive Plotly choropleth on real province boundaries (log-scale YlOrRd) |

##### Top 10 Provinces
| Rank | Code | Province | Community | EV Fleet 2027 | Share (%) | Mainland? |
|---:|:---:|:---|:---|---:|---:|:---:|
| 1 | M | Madrid | Comunidad de Madrid | 635,635 | 45.0% | Yes |
| 2 | B | Barcelona | Cataluña | 184,440 | 13.1% | Yes |
| 3 | V | Valencia | Comunitat Valenciana | 56,301 | 4.0% | Yes |
| 4 | A | Alicante | Comunitat Valenciana | 45,556 | 3.2% | Yes |
| 5 | GC | Las Palmas | Canarias | 35,456 | 2.5% | **Island** |
| 6 | IB | Illes Balears | Illes Balears | 33,221 | 2.4% | **Island** |
| 7 | MA | Málaga | Andalucía | 31,745 | 2.2% | Yes |
| 8 | TF | Santa Cruz de Tenerife | Canarias | 23,926 | 1.7% | **Island** |
| 9 | BI | Bizkaia | País Vasco | 22,951 | 1.6% | Yes |
| 10 | SE | Sevilla | Andalucía | 22,554 | 1.6% | Yes |

> **Strategic note for notebook 2.3:** Provinces GC, IB and TF are islands — they have no interurban road corridors connecting them to the mainland network and must be **excluded from the corridor placement analysis**. The effective mainland top 10 continues with Toledo (TO, rank 11), Murcia (MU, rank 12) and Zaragoza (Z, rank 13).

##### Key Findings
- **Madrid alone accounts for 45% of projected EV demand** — any corridor strategy must connect Madrid to the next-tier cities
- **The Madrid–Barcelona axis** (M → GU → Z → L → B) carries the highest end-to-end demand of any interurban route
- **The Mediterranean corridor** (B → GI → CS → V → A → MU) links 4 mainland top-10 provinces continuously
- **Islands are over-represented** in raw rankings (GC, IB, TF take ranks 5, 6, 8) due to high local EV adoption — they require a separate island-specific strategy
- **Province shares are stable year-on-year** (Section 4.2) — validates the 3-year proxy assumption
- The distribution follows a **power law**: the top 7 mainland provinces account for ~73% of mainland EV demand

##### Limitations & Assumptions
1. Province of registration ≠ province of primary use (e.g. company vehicles registered centrally in Madrid)
2. Growth rates may not be uniform across provinces — urban areas may accelerate faster than the national SARIMA forecast implies
3. For interurban charging, **traffic flow on highways** is the more direct demand signal — province fleet is used as a proxy until road-level data is integrated in notebook 2.3

##### Output
`notebooks/outputs/province_demand_2027.csv` — 52 rows (all provinces), imported directly by the charging station placement analysis (notebook 2.3)

---
## 2. Supply

Infrastructure state only. No proposals in this section.

| Sub-section | Key output |
|---|---|
| **2.1 Roads & Traffic** | `roads`, `traffic_gdf`, `traffic_map` |
| **2.2 Existing Chargers** | `sites_gdf` |
| **2.3 Power Grid** | `gdf_grid` |
| **2.4 Points of Interest** | `sites_candidate` |


---
### 2.1 Roads & Traffic

Interurban road network (OSM) joined with Ministry of Transport daily traffic counts.

**Outputs:** `roads`, `traffic_gdf`, `traffic_map`, `traffic_enriched`

#### Road Network (OpenStreetMap)

Fetches interurban road geometries via the Overpass API. Two mirrors are tried in order.

**Filter:** Road codes matching `AP-`, `A-`, `N-`, `R-`, `M-` prefixes (motorways, national roads, ring roads).

> Source: OpenStreetMap contributors, ODbL licence.


In [ ]:
ROADS_FILE = OUT / "roads.gpkg"

if ROADS_FILE.exists():
    print("Loading roads from local file...")
    roads = gpd.read_file(ROADS_FILE, layer="roads")
    print(roads.head())

else:
    print("Fetching road network from OpenStreetMap...")

    QUERY_ROADS = """
    [out:json][timeout:120][bbox:35.9,-9.5,43.8,4.5];
    (
      way["highway"]["ref"~"^(AP-|A-|N-|R-|M-)"];
    );
    out geom;
    """

    OVERPASS_MIRRORS = [
        "https://overpass-api.de/api/interpreter",
        "https://overpass.kumi.systems/api/interpreter",
    ]

    raw = None
    for url in OVERPASS_MIRRORS:
        try:
            resp = requests.post(url, data={"data": QUERY_ROADS}, timeout=300)
            resp.raise_for_status()
            raw = resp.json()
            print(f'{url.split("/")[2]} — {len(raw["elements"]):,} elements')
            break
        except Exception as e:
            print(f'{url.split("/")[2]} failed: {e}')

    if raw is None:
        raise RuntimeError("Both Overpass mirrors failed.")

    rows = []
    for el in raw["elements"]:
        if "geometry" not in el:
            continue
        coords = [(p["lon"], p["lat"]) for p in el["geometry"]]
        if len(coords) < 2:
            continue
        t = el.get("tags", {})
        ref = (t.get("ref") or "").split(";")[0].strip().upper()

        rows.append({
            "ref": ref,
            "highway": t.get("highway", ""),
            "name": t.get("name", ""),
            "geometry": LineString(coords),
        })

    roads = gpd.GeoDataFrame(rows, geometry="geometry", crs=CRS_GEO)

    print(f"\nRoads loaded: {len(roads):,} segments")
    roads.to_file(ROADS_FILE, layer="roads", driver="GPKG")

#### Traffic Data (Ministry of Transport)

Fetches segment-level daily traveller estimates from the Ministry of Transport's ArcGIS REST API.
Results are paginated in batches of 2,000 and CRS-converted from EPSG:3042 to WGS84.

**Field used:** `Total` — total daily travellers across all trip lengths.

> Source: Ministerio de Transportes, Movilidad y Agenda Urbana — BigData Movilidad 2.


In [ ]:
TRAFFIC_FILE = OUT / "traffic.gpkg"

if TRAFFIC_FILE.exists():
    print("Loading traffic from local file...")
    traffic = gpd.read_file(TRAFFIC_FILE, layer="traffic")
    print(traffic.head())

else:
    URL = "https://mapas.fomento.gob.es/arcgis2/rest/services/BigData/Movilidad_Big_Data_2/MapServer/1282/query"

    BATCH_SIZE = 2000
    offset = 0
    rows = []

    def arcgis_polyline_to_shapely(geom):
        paths = geom.get("paths", [])
        if not paths:
            return None
        line_parts = [LineString(path) for path in paths if len(path) >= 2]
        if not line_parts:
            return None
        return line_parts[0] if len(line_parts) == 1 else MultiLineString(line_parts)

    print("Fetching official traffic layer...")

    while True:
        params = {
            "f": "json",
            "where": "1=1",
            "returnGeometry": "true",
            "outFields": "*",
            "orderByFields": "OBJECTID ASC",
            "resultOffset": offset,
            "resultRecordCount": BATCH_SIZE
        }

        r = requests.get(URL, params=params, timeout=120)
        r.raise_for_status()
        data = r.json()

        features = data.get("features", [])
        if not features:
            break

        kept = 0
        for f in features:
            attrs = f.get("attributes", {}).copy()
            shp = arcgis_polyline_to_shapely(f.get("geometry", {}))
            if shp is None:
                continue
            attrs["geometry"] = shp
            rows.append(attrs)
            kept += 1

        print(f"Offset {offset:,}: kept {kept:,} features; total {len(rows):,}")

        if len(features) < BATCH_SIZE:
            break

        offset += BATCH_SIZE
        time.sleep(0.2)

    traffic = gpd.GeoDataFrame(rows, geometry="geometry", crs="EPSG:3042").to_crs(4326)

    print(traffic.shape)
    print(traffic.head())

    # Save to outputs folder
    traffic.to_file(TRAFFIC_FILE, layer="traffic", driver="GPKG")

In [ ]:
# ── Type coercion: cast numeric columns and parse date fields ─────────────────
numeric_cols = [
    "Total", "Corto", "Medio", "Largo",
    "Intra_GAU", "Inter_GAU", "Intra_prov", "Inter_prov",
    "Intra_ccaa", "Inter_ccaa", "Nacional", "Extranjero", "Shape_Length"
]

for col in numeric_cols:
    if col in traffic.columns:
        traffic[col] = pd.to_numeric(traffic[col], errors="coerce")

# Normalise road reference to match OSM road codes
traffic["ref"] = (
    traffic["nombre"].astype(str).str.strip().str.upper()
)

for col in ["Fecha_dato", "Valido_desde", "Valido_hasta"]:
    if col in traffic.columns:
        traffic[col] = pd.to_datetime(traffic[col], unit="ms", errors="coerce")

# Drop rows without a traffic value or geometry
traffic_gdf = traffic[
    traffic["Total"].notna() & traffic.geometry.notna()
].copy()

print("Traffic features:", len(traffic_gdf))
print("Unique road names:", traffic_gdf["nombre"].nunique())


In [ ]:
# ── Filter to interurban road codes only ──────────────────────────────────────
# Keeps AP- (toll motorways), A- (free motorways), N- (national), R-, M- (Madrid ring)
import re as _re
keep_pattern = r"^(AP-\d+[A-Z]*|A-\d+[A-Z]*|N-\d+[A-Z]*|R-\d+[A-Z]*|M-\d+[A-Z]*)$"

traffic_gdf = traffic_gdf[
    traffic_gdf["ref"].str.match(keep_pattern, na=False)
].copy()

print("After road-code filter:", len(traffic_gdf))


In [ ]:
# ── Prepare lightweight copy for map rendering ────────────────────────────────
# Simplify geometries for performance — does NOT affect analysis columns
traffic_map = traffic_gdf.to_crs(3042).copy()
traffic_map["seg_len_m"] = traffic_map.geometry.length
traffic_map = traffic_map[traffic_map["seg_len_m"] >= 20].copy()  # drop sub-20m artefacts
traffic_map["geometry"] = traffic_map.geometry.simplify(10, preserve_topology=True)
traffic_map = traffic_map.drop(columns=["seg_len_m"]).to_crs(4326)

# ── Background road layer: OSM roads simplified for map rendering ─────────────
roads_bg = roads[["ref", "name", "highway", "geometry"]].copy().to_crs(3857)
roads_bg["seg_len_m"] = roads_bg.geometry.length
roads_bg = roads_bg[roads_bg["seg_len_m"] >= 150].copy()
roads_bg["geometry"] = roads_bg.geometry.simplify(80, preserve_topology=True)
roads_bg = roads_bg.drop(columns="seg_len_m").to_crs(4326)

print("Map-ready traffic features:", len(traffic_map))
print("Background road segments:", len(roads_bg))


In [ ]:
# ── Combine roads + traffic into one enriched GeoDataFrame ───────────────────
# Strategy: two-pass join
#   Pass 1 — direct ref match (fast, covers ~80% of segments)
#   Pass 2 — spatial nearest-neighbour snap for unmatched remainder
#
# Result: traffic_enriched — one row per Ministry of Transport segment,
# geometry from traffic_gdf, road metadata (highway type, OSM name) attached.

CRS_JOIN = "EPSG:3857"

# ── Pass 1: direct ref join ───────────────────────────────────────────────────
# roads may have multiple segments per ref — keep the most common highway type
road_meta = (
    roads[["ref", "highway", "name"]]
    .groupby("ref")
    .agg(
        highway=("highway", lambda x: x.mode().iloc[0] if not x.empty else ""),
        road_name=("name",    lambda x: x.mode().iloc[0] if not x.empty else ""),
    )
    .reset_index()
    .rename(columns={"ref": "road_ref", "highway": "road_highway"})
)

traffic_enriched = traffic_gdf.copy()
traffic_enriched = traffic_enriched.merge(
    road_meta,
    left_on="ref",
    right_on="road_ref",
    how="left",
)

matched_direct   = traffic_enriched["road_ref"].notna().sum()
unmatched_mask   = traffic_enriched["road_ref"].isna()
print(f"Pass 1 (ref match):  {matched_direct:,} matched  |  {unmatched_mask.sum():,} unmatched")

# ── Pass 2: spatial snap for unmatched rows ───────────────────────────────────
if unmatched_mask.sum() > 0:
    # Build centroid GDFs in metric CRS
    unmatched = traffic_enriched[unmatched_mask][["ref", "Total", "geometry"]].copy()
    unmatched_c = unmatched.to_crs(CRS_JOIN).copy()
    unmatched_c["geometry"] = unmatched_c.geometry.centroid

    roads_c = roads[["ref", "highway", "name", "geometry"]].copy().to_crs(CRS_JOIN)
    roads_c["geometry"] = roads_c.geometry.centroid
    roads_c = roads_c.rename(columns={
        "ref":     "road_ref",
        "highway": "road_highway",
        "name":    "road_name",
    })

    snapped = gpd.sjoin_nearest(
        unmatched_c,
        roads_c[["road_ref", "road_highway", "road_name", "geometry"]],
        how="left",
        max_distance=5_000,       # only snap within 5 km
        distance_col="snap_dist_m",
    ).drop(columns=["index_right"], errors="ignore")

    # Write spatial results back into the main frame
    for col in ["road_ref", "road_highway", "road_name", "snap_dist_m"]:
        if col in snapped.columns:
            traffic_enriched.loc[unmatched_mask, col] = snapped[col].values

    matched_spatial = traffic_enriched["road_ref"].notna().sum() - matched_direct
    print(f"Pass 2 (spatial):    {matched_spatial:,} additionally matched")

# ── Summary ───────────────────────────────────────────────────────────────────
total_matched   = traffic_enriched["road_ref"].notna().sum()
total_unmatched = traffic_enriched["road_ref"].isna().sum()

print(f"\nFinal: {len(traffic_enriched):,} segments total")
print(f"  Matched:   {total_matched:,}  ({total_matched / len(traffic_enriched) * 100:.1f}%)")
print(f"  Unmatched: {total_unmatched:,}  (no OSM road within 5 km)")
print(f"\nHighway type breakdown:")
print(traffic_enriched["road_highway"].value_counts().head(8).to_string())
print(f"\nSample:")
print(traffic_enriched[["ref","road_ref","road_highway","Total"]].head(6).to_string(index=False))

#### Map 1 — Road Network & Traffic Intensity

The map below shows the interurban road network coloured by **daily traffic intensity**.

**Colour scale (travellers/day):**
- Blue — < 5,000 (low demand)
- Green — 5,000–20,000
- Orange — 20,000–50,000
- Red — > 50,000 (high demand corridors)
- Grey — structural reference (no matched traffic data)

> Red and orange segments are priority corridors for charging deployment.
> Low-traffic blue segments may still require coverage to satisfy the AFIR 150 km gap rule.


In [ ]:
def traffic_color(total):
    if pd.isna(total):   return "#cccccc"
    elif total < 10_000: return "#4575b4"
    elif total < 20_000: return "#91cf60"
    elif total < 50_000: return "#fc8d59"
    else:                return "#d73027"

# Prepare — stricter threshold + fewer columns
plot_df = traffic_enriched[traffic_enriched["Total"] >= 10_000].copy()
plot_df = plot_df[["ref", "Total", "geometry"]].to_crs(4326)
plot_df["color"] = plot_df["Total"].apply(traffic_color)

# TopoJSON with more aggressive simplification
topo = tp.Topology(plot_df, prequantize=False)
geojson_str = topo.toposimplify(
    epsilon=0.05,
    simplify_with="shapely",
    simplify_algorithm="dp",
    prevent_oversimplify=True,
).to_geojson()

# Round coordinates to 5 decimal places (~1 m precision, invisible at zoom 6)
geojson_dict = json.loads(geojson_str)
for feature in geojson_dict["features"]:
    geom = feature["geometry"]
    if geom["type"] == "LineString":
        geom["coordinates"] = [[round(x, 5), round(y, 5)] for x, y in geom["coordinates"]]
    elif geom["type"] == "MultiLineString":
        geom["coordinates"] = [
            [[round(x, 5), round(y, 5)] for x, y in line]
            for line in geom["coordinates"]
        ]

# Build map
m1 = folium.Map(location=[40.2, -3.7], zoom_start=6, tiles=MAP_TILES)
folium.GeoJson(
    geojson_dict,
    style_function=lambda f: {
        "color":   f["properties"]["color"],
        "weight":  2,
        "opacity": 0.8,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["ref", "Total"],
        aliases=["Road", "Travellers/day"],
    ),
).add_to(m1)

m1.save("map1.html")
print(f"File size: {Path('map1.html').stat().st_size / 1024:.0f} KB")
IFrame("map1.html", width="100%", height=500)

In [ ]:
m1

---
### 2.2 Existing Chargers

Official DGT DATEX II registry. One row per station.

**Outputs:** `sites_df`, `connectors_df`, `sites_gdf`

In [ ]:
import xml.etree.ElementTree as ET

url = "https://infocar.dgt.es/datex2/v3/miterd/EnergyInfrastructureTablePublication/electrolineras.xml"

r = requests.get(url, timeout=60)
r.raise_for_status()

root = ET.fromstring(r.content)
print("Root tag:", root.tag)


In [ ]:
# ── XML helper functions ──────────────────────────────────────────────────────

def strip_ns(tag):
    # Remove XML namespace prefix, e.g. "{http://...}latitude" → "latitude"
    return tag.split("}", 1)[-1]

def find_child_text(elem, path_tags):
    # Walk down a sequence of child tags and return the text of the final node.
    # Namespace-agnostic: works regardless of XML namespace declarations.
    cur = elem
    for tag in path_tags:
        nxt = None
        for child in cur:
            if strip_ns(child.tag) == tag:
                nxt = child
                break
        if nxt is None:
            return None
        cur = nxt
    return (cur.text or "").strip() if cur.text else None

def extract_coordinates(site):
    # Robustly find latitude/longitude anywhere inside a site element
    lat = lon = None
    for el in site.iter():
        tag = strip_ns(el.tag)
        txt = (el.text or "").strip()
        if tag == "latitude" and txt:
            try: lat = float(txt)
            except: pass
        elif tag == "longitude" and txt:
            try: lon = float(txt)
            except: pass
    return lat, lon

def safe_float(x):
    try: return float(x) if x not in [None, ""] else None
    except: return None

def extract_address_parts(site):
    # Extract structured address components from addressLine entries
    result = {"street": None, "municipality": None, "province": None,
              "autonomous_community": None, "postcode": None}

    result["postcode"] = find_child_text(
        site, ["locationReference", "_locationReferenceExtension",
               "facilityLocation", "address", "postcode"])

    for addr in site.iter():
        if strip_ns(addr.tag) != "addressLine":
            continue
        line_value = None
        for child in addr:
            if strip_ns(child.tag) == "text":
                line_value = find_child_text(child, ["values", "value"])
        if not line_value:
            continue
        low = line_value.lower()
        if low.startswith("dirección:") or low.startswith("direccion:"):
            result["street"] = line_value.split(":", 1)[1].strip()
        elif low.startswith("municipio:"):
            result["municipality"] = line_value.split(":", 1)[1].strip()
        elif low.startswith("provincia:"):
            result["province"] = line_value.split(":", 1)[1].strip()
        elif low.startswith("comunidad autónoma:") or low.startswith("comunidad autonoma:"):
            result["autonomous_community"] = line_value.split(":", 1)[1].strip()
    return result


In [ ]:
# ── Parse XML into site and connector tables ──────────────────────────────────
site_rows = []
connector_rows = []

sites = [el for el in root.iter() if strip_ns(el.tag) == "energyInfrastructureSite"]
print(f"Sites found: {len(sites):,}")

for site_idx, site in enumerate(sites, start=1):
    site_name     = find_child_text(site, ["name", "values", "value"])
    last_updated  = find_child_text(site, ["lastUpdated"])
    operator_name = find_child_text(site, ["operator", "name", "values", "value"])
    type_of_site  = find_child_text(site, ["typeOfSite"])
    latitude, longitude = extract_coordinates(site)
    addr = extract_address_parts(site)

    refill_points = [el for el in site.iter() if strip_ns(el.tag) == "refillPoint"]
    site_connector_count = 0
    site_max_power = None

    for rp_idx, rp in enumerate(refill_points, start=1):
        rp_name    = find_child_text(rp, ["name", "values", "value"])
        connectors = [el for el in rp if strip_ns(el.tag) == "connector"]

        for conn_idx, conn in enumerate(connectors, start=1):
            connector_type   = find_child_text(conn, ["connectorType"])
            charging_mode    = find_child_text(conn, ["chargingMode"])
            connector_format = find_child_text(conn, ["connectorFormat"])
            max_power        = safe_float(find_child_text(conn, ["maxPowerAtSocket"]))
            maximum_current  = safe_float(find_child_text(conn, ["maximumCurrent"]))

            site_connector_count += 1
            if max_power is not None:
                site_max_power = max(site_max_power, max_power) if site_max_power else max_power

            connector_rows.append({
                "site_id": site_idx, "site_name": site_name,
                "operator": operator_name, "postcode": addr["postcode"],
                "street": addr["street"], "municipality": addr["municipality"],
                "province": addr["province"],
                "autonomous_community": addr["autonomous_community"],
                "latitude": latitude, "longitude": longitude,
                "type_of_site": type_of_site, "last_updated": last_updated,
                "refill_point_id": rp_idx, "refill_point_name": rp_name,
                "connector_id": conn_idx, "connector_type": connector_type,
                "charging_mode": charging_mode, "connector_format": connector_format,
                "max_power_kw": max_power, "maximum_current_a": maximum_current,
            })

    site_rows.append({
        "site_id": site_idx, "site_name": site_name, "operator": operator_name,
        "postcode": addr["postcode"], "street": addr["street"],
        "municipality": addr["municipality"], "province": addr["province"],
        "autonomous_community": addr["autonomous_community"],
        "latitude": latitude, "longitude": longitude,
        "type_of_site": type_of_site, "last_updated": last_updated,
        "n_refill_points": len(refill_points),
        "n_connectors": site_connector_count,
        "site_max_power_kw": site_max_power,
    })

sites_df      = pd.DataFrame(site_rows)
connectors_df = pd.DataFrame(connector_rows)

print("Sites table:", sites_df.shape)
print("Connectors table:", connectors_df.shape)
sites_df.head()


In [ ]:
# ── Aggregated summaries ──────────────────────────────────────────────────────

summary_province = (
    connectors_df.groupby("province", dropna=False)
    .agg(total_connectors=("connector_id","count"), total_sites=("site_id","nunique"),
         total_operators=("operator","nunique"), max_power_kw=("max_power_kw","max"),
         avg_power_kw=("max_power_kw","mean"))
    .reset_index().sort_values("total_connectors", ascending=False)
)

summary_municipality = (
    connectors_df.groupby(["province","municipality"], dropna=False)
    .agg(total_connectors=("connector_id","count"), total_sites=("site_id","nunique"),
         total_operators=("operator","nunique"), max_power_kw=("max_power_kw","max"),
         avg_power_kw=("max_power_kw","mean"))
    .reset_index().sort_values("total_sites", ascending=False)
)

summary_autonomous_community = (
    connectors_df.groupby("autonomous_community", dropna=False)
    .agg(total_connectors=("connector_id","count"), total_sites=("site_id","nunique"),
         total_operators=("operator","nunique"), max_power_kw=("max_power_kw","max"),
         avg_power_kw=("max_power_kw","mean"))
    .reset_index().sort_values("total_sites", ascending=False)
)

summary_operator = (
    connectors_df.groupby("operator", dropna=False)
    .agg(total_connectors=("connector_id","count"), total_sites=("site_id","nunique"),
         provinces_covered=("province","nunique"), municipalities_covered=("municipality","nunique"),
         max_power_kw=("max_power_kw","max"), avg_power_kw=("max_power_kw","mean"))
    .reset_index().sort_values("total_connectors", ascending=False)
)

summary_operator_connector_type = (
    connectors_df.groupby(["operator","connector_type"], dropna=False)
    .agg(total_connectors=("connector_id","count"), max_power_kw=("max_power_kw","max"),
         avg_power_kw=("max_power_kw","mean"))
    .reset_index().sort_values("total_connectors", ascending=False)
)


In [ ]:
print("Top 5 provinces by connector count:")
print(summary_province.head(5).to_string(index=False))

In [ ]:
print("Top 5 municipalities by connector count:")
print(summary_municipality.head(5).to_string(index=False))

In [ ]:
print("Top 5 autonomous communities by connector count:")
print(summary_autonomous_community.head(5).to_string(index=False))

In [ ]:
print("Top 5 operators by connector count:")
print(summary_operator.head(5).to_string(index=False))

In [ ]:
sites_df["site_max_power_kw"].describe()

#### Map 2 — Existing EV Charging Stations

Markers are **clustered** for readability at low zoom; individual stations appear on zoom-in.

**Colour coding by max connector power:**
- Red — DC Fast (>= 50 kW)
- Orange — AC High (22–49 kW)
- Blue — AC Standard (< 22 kW)
- Grey — Power unknown

Click any marker for station details (operator, connectors, power, municipality).


In [ ]:
def charger_color(max_kw):
    if pd.isna(max_kw): return "#cccccc"
    elif max_kw >= 50:  return "#d73027"
    elif max_kw >= 22:  return "#fc8d59"
    else:               return "#4575b4"

# Build GeoDataFrame
chargers_gdf = gpd.GeoDataFrame(
    sites_df.dropna(subset=["latitude", "longitude"]).copy(),
    geometry=gpd.points_from_xy(
        sites_df.dropna(subset=["latitude", "longitude"])["longitude"],
        sites_df.dropna(subset=["latitude", "longitude"])["latitude"],
    ),
    crs="EPSG:4326"
)[["site_name", "operator", "municipality", "province", "site_max_power_kw", "n_connectors", "geometry"]]

# Convert watts to kW
chargers_gdf["max_kw"] = (chargers_gdf["site_max_power_kw"] / 1000).round(1)
chargers_gdf["color"]  = chargers_gdf["max_kw"].apply(charger_color)

# Drop raw watts column — not needed in the map
chargers_gdf = chargers_gdf.drop(columns=["site_max_power_kw"])

# Round coordinates to 5 decimal places
geojson_dict = json.loads(chargers_gdf.to_json())
for feature in geojson_dict["features"]:
    coords = feature["geometry"]["coordinates"]
    feature["geometry"]["coordinates"] = [round(coords[0], 5), round(coords[1], 5)]

# Build map
m2 = folium.Map(location=[40.2, -3.7], zoom_start=6, tiles=MAP_TILES)

folium.GeoJson(
    geojson_dict,
    marker=folium.CircleMarker(radius=5, fill=True, fill_opacity=0.8),
    style_function=lambda f: {
        "color":       f["properties"]["color"],
        "fillColor":   f["properties"]["color"],
        "fillOpacity": 0.8,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["site_name", "max_kw", "n_connectors"],
        aliases=["Site", "Max kW", "Connectors"],
    ),
).add_to(m2)

m2.save("map2.html")
print(f"Chargers plotted: {len(chargers_gdf):,}")
print(f"File size: {Path('map2.html').stat().st_size / 1024:.0f} KB")
IFrame("map2.html", width="100%", height=500)

In [ ]:
m2

---
### 2.3 Power Grid

CNMC distributor datasets (i-DE, UFD, Endesa, E-Redes). Substation-level available MW.

**Output:** `gdf_grid` (unified GeoDataFrame)

#### Multi-Distributor Grid Data

**What these three datasets represent:**

| File | Code | Distributor | Territory |
|---|---|---|---|---|
| `R1-001_Demanda.csv` | R1-001 | **i-DE (Iberdrola)** | Castilla, País Vasco, Valencia, Murcia, Navarra, Aragón, La Rioja |
| `R1-002_demanda.csv` | R1-002 | **UFD (Naturgy)** | Galicia, Madrid, Castilla-La Mancha, parts of Asturias & León |
| `R1299_demanda.csv` | R1-299 | **e-distribución (Endesa)** | Andalucía, Cataluña, Extremadura, Baleares, Canarias, Aragón |
| `R1005_demanda.csv` | R1-005 | **Viesgo** | Cantabria, Asturias, Castilla y León, Galicia|

**Column meanings:**
- `capacity_available_mw` — headroom available for new connections right now
- `capacity_occupied_mw` — already granted to existing demand connections  
- `capacity_pending_mw` — in the application pipeline (reduces real availability)
- `capacity_regulatory_mw` — reserved by regulatory requirements
- `voltage_kv` — bus bar voltage at the connection point (13.2 / 15 / 20 / 30 / 45 / 66 / 132 kV)
- `positions_free` / `positions_occupied` — physical bay slots on the substation busbar

**Key assumption:** Coordinates are in ETRS89 / UTM Zone 30N (EPSG:25830) for R1-001 and R1-299.  
R1-002 (UFD) uses the same CRS — verified from coordinate magnitudes (easting ~400k–900k, northing ~4M–4.9M).


In [ ]:
df_ide = pd.read_csv("../data/2026_04_01_R1-001_Demanda.csv", sep=";")
df_ide.head()

In [ ]:
# ── Standardise column names ──────────────────────────────────────────────────
df_ide.columns = [c.strip() for c in df_ide.columns]

rename_map = {
    "Gestor de red": "grid_manager",
    "Provincia": "province",
    "Municipio": "municipality",
    "Coordenada UTM X": "utm_x",
    "Coordenada UTM Y": "utm_y",
    "Subestación": "substation",
    "Nivel de Tensión (kV)": "voltage_kv",
    "Capacidad firme disponible (MW)": "capacity_available_mw",
    "Capacidad comprometida por cuestiones regulatorias": "capacity_regulatory_mw",
    "Capacidad de acceso firme de demanda ocupada (MW)": "capacity_occupied_mw",
    "Capacidad de acceso firme admitida y no evaluada (MW)": "capacity_pending_mw",
    "Posiciones ocupadas": "positions_occupied",
    "Posiciones libres": "positions_free",
    "Nudo 0*": "node_0",
    "Comentarios": "comments",
    "Denominación del Punto de Conexión": "connection_point_name",
    "Identificador del Punto de Conexión": "connection_point_id",
}
df_ide = df_ide.rename(columns=rename_map)


In [ ]:
# ── Parse numeric columns: handle Spanish number formatting ──────────────────
# Spanish CSVs often use '.' as thousands separator and ',' as decimal separator
num_cols = [
    "utm_x", "utm_y", "voltage_kv",
    "capacity_available_mw", "capacity_regulatory_mw",
    "capacity_occupied_mw", "capacity_pending_mw",
    "positions_occupied", "positions_free", "node_0", "connection_point_id",
]

for col in num_cols:
    if col in df_ide.columns:
        df_ide[col] = (
            df_ide[col].astype(str).str.strip()
            .str.replace(".", "", regex=False)   # remove thousands separator
            .str.replace(",", ".", regex=False)  # decimal comma -> decimal point
            .replace({"nan": np.nan, "": np.nan})
        )
        df_ide[col] = pd.to_numeric(df_ide[col], errors="coerce")

df_ide.head()


In [ ]:
# ── Build GeoDataFrame — project from ETRS89/UTM30N (EPSG:25830) to WGS84 ───
gdf_ide = gpd.GeoDataFrame(
    df_ide.dropna(subset=["utm_x", "utm_y"]).copy(),
    geometry=gpd.points_from_xy(
        df_ide.dropna(subset=["utm_x", "utm_y"])["utm_x"],
        df_ide.dropna(subset=["utm_x", "utm_y"])["utm_y"]
    ),
    crs="EPSG:25830"
).to_crs(4326)

gdf_ide[["province", "municipality", "utm_x", "utm_y", "geometry"]].head()


In [ ]:
# ── Derived capacity metrics ──────────────────────────────────────────────────

# Total committed capacity (regulatory + occupied + pending requests)
gdf_ide["capacity_constrained_mw"] = (
    gdf_ide["capacity_regulatory_mw"].fillna(0) +
    gdf_ide["capacity_occupied_mw"].fillna(0) +
    gdf_ide["capacity_pending_mw"].fillna(0)
)

# Viability flags for fast-charging deployment
gdf_ide["viable_fast_charging"] = gdf_ide["capacity_available_mw"] >= 1  # >=1 MW: 6+ chargers
gdf_ide["viable_hpc"]           = gdf_ide["capacity_available_mw"] >= 5  # >=5 MW: full hub


In [ ]:
# ── Grid capacity summaries ───────────────────────────────────────────────────

summary_grid_province = (
    gdf_ide.groupby("province", dropna=False)
    .agg(n_nodes=("connection_point_id","count"),
         total_available_mw=("capacity_available_mw","sum"),
         avg_available_mw=("capacity_available_mw","mean"),
         max_available_mw=("capacity_available_mw","max"),
         viable_fast_nodes=("viable_fast_charging","sum"),
         viable_hpc_nodes=("viable_hpc","sum"))
    .reset_index().sort_values("total_available_mw", ascending=False)
)

summary_grid_municipality = (
    gdf_ide.groupby(["province","municipality"], dropna=False)
    .agg(n_nodes=("connection_point_id","count"),
         total_available_mw=("capacity_available_mw","sum"),
         avg_available_mw=("capacity_available_mw","mean"),
         max_available_mw=("capacity_available_mw","max"),
         viable_fast_nodes=("viable_fast_charging","sum"),
         viable_hpc_nodes=("viable_hpc","sum"))
    .reset_index().sort_values(["province","total_available_mw"], ascending=[True,False])
)

print("Top 5 provinces by total available grid capacity:")
print(summary_grid_province.head(5).to_string(index=False))


In [ ]:

# ── Load all three CNMC distributor files and unify ───────────────────────────
# Adjust paths to match your working directory

DISTRIBUTOR_FILES = {
    "i-DE (Iberdrola)":    ("../data/2026_04_01_R1-001_Demanda.csv","utf-8-sig"),
    "UFD (Naturgy)":       ("../data/2026_04_01_R1-002_demanda.csv","utf-8-sig"),
    "e-dist (Endesa)":     ("../data/2026_04_01_R1299_demanda.csv","utf-8-sig"),
    "Viesgo":              ("../data/2026_04_01_R1005_demanda.csv","latin-1"),
}

# R1-002 has footnote suffixes in column names — strip them
def normalise_col(c):
    return c.strip().split("[")[0].strip()

RENAME_UNIFIED = {
    "Gestor de red":                                    "grid_manager",
    "Provincia":                                        "province_raw",
    "Municipio":                                        "municipality_raw",
    "Coordenada UTM X":                                 "utm_x",
    "Coordenada UTM Y":                                 "utm_y",
    "Subestación":                                      "substation",
    "Nivel de Tensión (kV)":                            "voltage_kv",
    "Nivel de tensión (kV)":                            "voltage_kv",
    "Capacidad firme disponible (MW)":                  "capacity_available_mw",
    "Capacidad comprometida por cuestiones regulatorias":"capacity_regulatory_mw",
    "Capacidad de acceso firme de demanda ocupada (MW)":"capacity_occupied_mw",
    "Capacidad de acceso firme de demanda ocupada (MW)":"capacity_occupied_mw",
    "Capacidad de acceso firme admitida y no evaluada (MW)":"capacity_pending_mw",
    "Capacidad firme admitida y no evaluada (MW)":      "capacity_pending_mw",
    "Posiciones ocupadas":                              "positions_occupied",
    "Posiciones libres":                                "positions_free",
    "Nudo 0*":                                          "node_0",
    "Comentarios":                                      "comments",
    # R1-001 specific
    "Denominación del Punto de Conexión":               "connection_point_name",
    "Identificador del Punto de Conexión":              "connection_point_id",
    # R1-002 specific
    "Nombre Subestación":                               "connection_point_name",
    "Matrícula Sub.":                                   "connection_point_id",
    # R1-299 specific  
    "Nombre Subestación":                               "connection_point_name",
    "Comunidad Autónoma":                               "autonomous_community",
    "Provincia.1":                                      "province",
    "Municipio.1":                                      "municipality",
    # R1005 specific
    "Nombre Subestación":                               "connection_point_name",
}

NUM_COLS = [
    "utm_x", "utm_y", "voltage_kv",
    "capacity_available_mw", "capacity_regulatory_mw",
    "capacity_occupied_mw",  "capacity_pending_mw",
    "positions_occupied",    "positions_free",
]

def load_grid_csv(path, label, encoding="utf-8-sig"):
    df = pd.read_csv(path, sep=";", encoding=encoding)
    df.columns = [normalise_col(c) for c in df.columns]
    df = df.rename(columns=RENAME_UNIFIED)
    df["_source"] = label

    # R1-299 uses numeric province codes — use text columns instead
    if "province" not in df.columns:
        df["province"]     = df.get("province_raw", "")
        df["municipality"] = df.get("municipality_raw", "")

    for col in NUM_COLS:
        if col in df.columns:
            df[col] = (
                df[col].astype(str).str.strip()
                .str.replace(".", "", regex=False)
                .str.replace(",", ".", regex=False)
                .replace({"nan": np.nan, "": np.nan})
            )
            df[col] = pd.to_numeric(df[col], errors="coerce")

    print(f"  {label}: {len(df):,} rows | "
          f"avg avail: {df['capacity_available_mw'].mean():.2f} MW | "
          f"viable (≥1MW): {(df['capacity_available_mw'] >= 1).sum():,}")
    return df

print("Loading grid capacity datasets...")

frames = []
for label, (path, enc) in DISTRIBUTOR_FILES.items():
    try:
        frames.append(load_grid_csv(path, label, encoding=enc))
    except FileNotFoundError:
        print(f"  ⚠ {label}: not found — skipping")

df_grid_all = pd.concat(frames, ignore_index=True)

# Fill missing name/id columns from substation code
if "connection_point_name" not in df_grid_all.columns:
    df_grid_all["connection_point_name"] = df_grid_all.get("substation", "").astype(str)
if "connection_point_id" not in df_grid_all.columns:
    df_grid_all["connection_point_id"] = df_grid_all.index

print(f"\nUnified grid dataset: {len(df_grid_all):,} connection points")
print(df_grid_all["_source"].value_counts().to_string())


In [ ]:

# ── Build unified GeoDataFrame ────────────────────────────────────────────────
valid = df_grid_all.dropna(subset=["utm_x", "utm_y"]).copy()

gdf_grid = gpd.GeoDataFrame(
    valid,
    geometry=gpd.points_from_xy(valid["utm_x"], valid["utm_y"]),
    crs="EPSG:25830"
).to_crs(4326)

# ── Derived metrics ───────────────────────────────────────────────────────────
gdf_grid["capacity_constrained_mw"] = (
    gdf_grid["capacity_regulatory_mw"].fillna(0) +
    gdf_grid["capacity_occupied_mw"].fillna(0) +
    gdf_grid["capacity_pending_mw"].fillna(0)
)
gdf_grid["viable_fast"] = gdf_grid["capacity_available_mw"] >= 1   # ≥1 MW → ≥6 × 150 kW chargers
gdf_grid["viable_hpc"]  = gdf_grid["capacity_available_mw"] >= 5   # ≥5 MW → full HPC hub

# ── Capacity stress indicator ─────────────────────────────────────────────────
# Ratio of occupied+pending to available: >1 = over-committed
total_capacity = (
    gdf_grid["capacity_available_mw"] +
    gdf_grid["capacity_occupied_mw"].fillna(0) +
    gdf_grid["capacity_pending_mw"].fillna(0) +
    gdf_grid["capacity_regulatory_mw"].fillna(0)
)
gdf_grid["stress_ratio"] = np.where(
    total_capacity > 0,
    gdf_grid["capacity_constrained_mw"] / total_capacity,
    np.nan
)

print(f"Grid nodes with coordinates: {len(gdf_grid):,}")
print(f"  HPC-viable (≥5 MW):   {gdf_grid['viable_hpc'].sum():,}")
print(f"  Fast-viable (≥1 MW):  {gdf_grid['viable_fast'].sum():,}")
print(f"  Saturated (<1 MW):    {(gdf_grid['capacity_available_mw'] < 1).sum():,}")
print(f"\nBy distributor:")
print(gdf_grid.groupby("_source")[["capacity_available_mw"]].agg(
    count=("capacity_available_mw","count"),
    mean_mw=("capacity_available_mw","mean"),
    viable_hpc=("capacity_available_mw", lambda x: (x>=5).sum())
).round(2).to_string())


In [ ]:

# ── Province-level capacity summary ──────────────────────────────────────────
summary_grid = (
    gdf_grid
    .groupby(["province", "_source"], dropna=False)
    .agg(
        n_nodes=("capacity_available_mw", "count"),
        total_available_mw=("capacity_available_mw", "sum"),
        avg_available_mw=("capacity_available_mw", "mean"),
        viable_fast_nodes=("viable_fast", "sum"),
        viable_hpc_nodes=("viable_hpc", "sum"),
    )
    .reset_index()
    .sort_values("total_available_mw", ascending=False)
)

print("Top 15 provinces by total available grid capacity (MW):")
print(summary_grid.head(15)[
    ["province", "_source", "n_nodes", "total_available_mw", "viable_hpc_nodes"]
].to_string(index=False))


#### Map 3 — Grid Connection Capacity

Each circle is a connection point (substation node). **Size and colour** reflect available MW for new demand connections.

- Red (large) — > 20 MW — abundant capacity, ideal for HPC hubs
- Orange — 5–20 MW — medium hub viable
- Green — 1–5 MW — standard fast-charging viable
- Blue (small) — < 1 MW — constrained; upgrade required

Click any marker for full substation details.


In [ ]:
def grid_color(mw):
    if pd.isna(mw) or mw < 1: return "#cccccc"
    elif mw < 5:               return "#fc8d59"
    elif mw < 20:              return "#91cf60"
    else:                      return "#d73027"

# Keep only columns needed for the map
grid_plot = gdf_grid[["connection_point_name", "province", "municipality",
                       "_source", "voltage_kv", "capacity_available_mw",
                       "geometry"]].copy()

grid_plot["color"] = grid_plot["capacity_available_mw"].apply(grid_color)

# Round coordinates to 5 decimal places
geojson_dict = json.loads(grid_plot.to_json())
for feature in geojson_dict["features"]:
    coords = feature["geometry"]["coordinates"]
    feature["geometry"]["coordinates"] = [round(coords[0], 5), round(coords[1], 5)]

# Build map
m3 = folium.Map(location=[40.2, -3.7], zoom_start=6, tiles=MAP_TILES)

folium.GeoJson(
    geojson_dict,
    marker=folium.CircleMarker(radius=5, fill=True, fill_opacity=0.8),
    style_function=lambda f: {
        "color":       f["properties"]["color"],
        "fillColor":   f["properties"]["color"],
        "fillOpacity": 0.8,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["connection_point_name", "capacity_available_mw", "voltage_kv", "_source"],
        aliases=["Substation", "Available MW", "Voltage (kV)", "Distributor"],
    ),
).add_to(m3)

m3.save("map3.html")
print(f"Grid nodes plotted: {len(grid_plot):,}")
print(f"File size: {Path('map3.html').stat().st_size / 1024:.0f} KB")
IFrame("map3.html", width="100%", height=500)

---
### 2.4 Points of Interest

Candidate hosting locations from OpenStreetMap (Overpass API). Cached to `outputs/pois_cache.gpkg`.

**Output:** `sites_candidate` GeoDataFrame

In [ ]:
OVERPASS_MIRRORS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
]

POI_CACHE = OUT / "pois_cache.gpkg"
CRS_GEO   = "EPSG:4326"
SPAIN_BBOX = "27.6,-18.2,43.8,4.5"   # full Spain including islands

# ── One entry per category — simple, independent, sequential ─────────────────
# Each fetches a single tag across all of Spain in one clean query.
# No splitting, no parallelism — Overpass handles it fine at this granularity.
CATEGORIES = [
    ("[amenity=fuel]",                              "petrol_station"),
    ("[highway=services]",                          "motorway_services"),
    ("[highway=rest_area]",                         "rest_area"),
    ("[shop=mall]",                                 "mall"),
    ("[shop=department_store]",                     "department_store"),
    ("[shop=supermarket]",                          "supermarket"),
    ("[tourism=hotel]",                             "hotel"),
    ("[amenity=hospital]",                          "hospital"),
    ("[amenity=university]",                        "university"),
    ("[amenity=bus_station]",                       "bus_station"),
    ("[railway=station]",                           "train_station"),
    ("[aeroway=terminal]",                          "airport"),
    ("[amenity=parking][access!=private]",          "parking"),
    ("[leisure=sports_centre]",                     "sports_centre"),
    ("[tourism=attraction]",                        "attraction"),
    ("[amenity=cinema]",                            "cinema"),
    ("[amenity=theatre]",                           "theatre"),
    ("[leisure=stadium]",                           "stadium"),
]


def fetch_category(tag_filter, category, bbox=SPAIN_BBOX, retries=3):
    """One query, one category, full Spain bbox. Sequential and simple."""
    query = (
        f"[out:json][timeout:120][maxsize:536870912][bbox:{bbox}];"
        f"(node{tag_filter};way{tag_filter};);"
        f"out center tags qt;"
    )
    for attempt in range(retries):
        mirror = OVERPASS_MIRRORS[attempt % len(OVERPASS_MIRRORS)]
        try:
            resp = requests.post(
                mirror,
                data={"data": query},
                headers={"Content-Type": "application/x-www-form-urlencoded"},
                timeout=150,
            )
            if resp.status_code == 429:
                print(f"  ⏳ rate limited — waiting 60s")
                time.sleep(60)
                continue
            resp.raise_for_status()
            elements = resp.json().get("elements", [])
            print(f"  ✓ {category:<25} {len(elements):>7,}  [{mirror.split('/')[2]}]")
            return elements
        except Exception as e:
            wait = 15 * (attempt + 1)
            print(f"  ✗ {category} attempt {attempt+1}: {type(e).__name__} — retry in {wait}s")
            time.sleep(wait)
    print(f"  ⚠ {category}: failed — skipping")
    return []


def parse_elements(elements, category):
    rows = []
    for el in elements:
        lat = el.get("lat") or el.get("center", {}).get("lat")
        lon = el.get("lon") or el.get("center", {}).get("lon")
        if lat is None or lon is None:
            continue
        tags = el.get("tags", {})
        rows.append({
            "osm_id":   el.get("id"),
            "category": category,
            "name":     tags.get("name", ""),
            "lat":      lat,
            "lon":      lon,
        })
    return rows

In [ ]:
# ── Fetch all categories one by one ──────────────────────────────────────────
# Sequential with 10s pause — respects Overpass fair-use policy.
# Each category is independent: if one fails it's skipped, others still run.

if POI_CACHE.exists():
    print(f"Loading from cache: {POI_CACHE}")
    sites_candidate = gpd.read_file(POI_CACHE)
    print(f"Loaded {len(sites_candidate):,} POI sites")

else:
    all_rows = []

    for tag_filter, category in CATEGORIES:
        rows = fetch_category(tag_filter, category)
        all_rows.extend(parse_elements(rows, category))
        time.sleep(10)   # 10s between requests — Overpass ToS

    df_pois = (
        pd.DataFrame(all_rows)
        .drop_duplicates(subset=["osm_id"])
        .reset_index(drop=True)
    )

    sites_candidate = gpd.GeoDataFrame(
        df_pois,
        geometry=gpd.points_from_xy(df_pois["lon"], df_pois["lat"]),
        crs=CRS_GEO
    )

    sites_candidate.to_file(POI_CACHE, driver="GPKG")
    print(f"\nSaved to cache: {POI_CACHE}")

print(f"\nTotal POI sites: {len(sites_candidate):,}")
print(sites_candidate["category"].value_counts().to_string())

#### Map 4 — Points of Interest

Candidate host sites fetched from OSM — clustered by category. These are the physical locations
where an EV charger could realistically be installed (forecourt, parking, dwell-time location).

> Click any cluster to expand. Click any marker for site details.


In [ ]:
CATEGORY_COLORS = {
    "petrol_station":   "#e74c3c",
    "motorway_services":"#e67e22",
    "rest_area":        "#f39c12",
    "mall":             "#9b59b6",
    "department_store": "#8e44ad",
    "supermarket":      "#3498db",
    "hotel":            "#1abc9c",
    "hospital":         "#e91e63",
    "university":       "#2980b9",
    "bus_station":      "#27ae60",
    "train_station":    "#16a085",
    "airport":          "#2c3e50",
    "parking":          "#95a5a6",
    "sports_centre":    "#f1c40f",
    "attraction":       "#d35400",
    "cinema":           "#c0392b",
    "theatre":          "#8e44ad",
    "stadium":          "#e74c3c",
}

fig = go.Figure()

for cat, color in CATEGORY_COLORS.items():
    subset = sites_candidate[sites_candidate["category"] == cat]
    if subset.empty:
        continue

    fig.add_trace(go.Scattermap(
        lat=subset["lat"].round(4),
        lon=subset["lon"].round(4),
        mode="markers",
        marker=dict(size=5, color=color, opacity=0.8),
        name=f"{cat.replace('_',' ').title()} ({len(subset):,})",
        text=subset["name"].fillna(""),
        hovertemplate="<b>%{text}</b><br>" + cat + "<extra></extra>",
        visible=True,
    ))

fig.update_layout(
    map=dict(
        style="carto-positron",
        center=dict(lat=40.2, lon=-3.7),
        zoom=5,
    ),
    margin=dict(r=0, t=30, l=0, b=0),
    height=600,
    title="Section 4 — Points of Interest",
    legend=dict(
        x=0.01, y=0.99,
        bgcolor="rgba(255,255,255,0.85)",
        font=dict(size=10),
    ),
)

fig.write_html(
    "outputs/map4_pois.html",
    include_plotlyjs="cdn",    # ← loads Plotly from CDN instead of embedding it
    full_html=True,
)

size_kb = Path("outputs/map4_pois.html").stat().st_size / 1024
print(f"File size: {size_kb:,.0f} KB")
print(f"Total points: {len(sites_candidate):,}")
fig.show()
IFrame("outputs/map4_pois.html", width="100%", height=600)

---
## 3. Combined Analysis

Synthesises Section 1 (demand) and Section 2 (supply) to produce all datathon deliverables.

**Variables consumed from earlier sections:**

| Variable | Source |
|---|---|
| `traffic_map`, `traffic_enriched` | §2.1 Roads & Traffic |
| `chargers_gdf`, `sites_gdf` | §2.2 Existing Chargers |
| `gdf_grid` | §2.3 Power Grid |
| `sites_candidate` | §2.4 Points of Interest |
| `demand` | §1.2 Demand per Region |
| `TOTAL_EV_PROJECTED_2027` | §1.1 EV Fleet Forecast |
| All thresholds & constants | §0 Setup |

**Datathon deliverables produced here:**

| Deliverable | File | Rule |
|---|---|---|
| Global Network KPIs | `File 1.csv` | Row counts from File 2 + File 3 + §1.1 |
| Proposed Charging Locations | `File 2.csv` | Rule 2: `estimated_demand_kw = n_chargers × 150` |
| Friction Points | `File 3.csv` | Rule 3: only Moderate or Congested from File 2 |
| BI Visualisation | `BI_map.html` | Green=Sufficient · Yellow=Moderate · Red=Congested |


---
### 3.1 Filter & Score Candidate Sites

Filters `sites_candidate` (§2.4) to interurban-only locations, then scores each site
using five signals drawn entirely from Sections 1 and 2.

**Hard filters — both must pass:**
- Within `TRAFFIC_BUFFER_M` of an interurban road (`traffic_map` §2.1)
- Nearest grid node within `GRID_SNAP_RADIUS_M` (`gdf_grid` §2.3)

> Charger gap is **not** a hard filter — it is a scoring dimension (`s_gap`).
> A hard cutoff caused a gap/grid mutual exclusion leaving 0 candidates (see diagnostics).

**Composite score — five signals:**

| Signal | Weight | Source |
|---|---|---|
| `s_traffic` | 30 % | Mean daily travellers on roads within buffer (§2.1) |
| `s_gap` | 25 % | Distance to nearest existing charger (§2.2) |
| `s_grid` | 20 % | Available MW at nearest substation (§2.3) |
| `s_poi` | 15 % | POI category suitability weight (§2.4) |
| `s_demand` | 10 % | Province EV fleet 2027 share (§1.2 + §1.1) |


In [131]:
# ── Step 1: Filter candidate sites ───────────────────────────────────────────
# Sources: traffic_map (§2.1), chargers_gdf (§2.2), gdf_grid (§2.3), sites_candidate (§2.4)
CRS_M = "EPSG:3857"

print("Projecting all layers to metric CRS...")
_traffic_m = traffic_map[["ref", "Total", "geometry"]].to_crs(CRS_M).copy()
_traffic_m = _traffic_m[_traffic_m["Total"].notna()].copy()
_traffic_m["centroid"] = _traffic_m.geometry.centroid

_chargers_m = chargers_gdf.to_crs(CRS_M).copy()
# Aggregate gdf_grid by physical substation (100-m grid rounding)
# Multiple connection-point rows at the same substation are summed so sjoin_nearest
# gets the real total available capacity per physical site, not a single saturated slot.
_grid_raw = gdf_grid.to_crs(CRS_M).copy()
_grid_raw["_xr"] = _grid_raw.geometry.x.round(-2)  # 100 m
_grid_raw["_yr"] = _grid_raw.geometry.y.round(-2)
_grid_sub = (
    _grid_raw.groupby(["_xr", "_yr"], as_index=False)
    .agg(
        capacity_available_mw=("capacity_available_mw", "sum"),
        _source=("_source", "first"),
        geometry=("geometry", "first"),
    )
)
_grid_m = gpd.GeoDataFrame(_grid_sub, geometry="geometry", crs=CRS_M).copy()
print(f"Grid substations (aggregated): {len(_grid_m):,}")
print(f"  Congested (<1 MW):  {(_grid_m.capacity_available_mw < 1).sum():,}")
print(f"  Moderate (1-5 MW):  {((_grid_m.capacity_available_mw >= 1) & (_grid_m.capacity_available_mw < 5)).sum():,}")
print(f"  Sufficient (≥5 MW): {(_grid_m.capacity_available_mw >= 5).sum():,}")
_sites_m    = sites_candidate.to_crs(CRS_M).copy()

# Road buffer union — interurban proximity filter
_road_buf             = _traffic_m.copy()
_road_buf["geometry"] = _traffic_m.geometry.buffer(TRAFFIC_BUFFER_M)
_road_union           = _road_buf.unary_union

# Unary unions for distance scoring (used in Step 2, not as hard filters)
_charger_union = _chargers_m.unary_union
_grid_union    = _grid_m.unary_union if not _grid_m.empty else None

# Hard filter: must be within TRAFFIC_BUFFER_M of an interurban road
# Grid proximity is NOT a hard filter — it is scored as s_grid in Step 2.
# (A hard grid filter would eliminate rural/highway sites that are far from
#  substations but have the highest charger gap, causing a 0-candidate result.)
_mask_road = pd.Series(
    [bool(g.within(_road_union)) for g in _sites_m.geometry],
    index=_sites_m.index
)

candidates_m = _sites_m[_mask_road].copy()

print(f"Filter results:")
print(f"  Road proximity (within {TRAFFIC_BUFFER_M/1000:.0f} km): {_mask_road.sum():,} pass")
print(f"  → {len(candidates_m):,} candidate sites (grid scored in §3.1, not hard-filtered)")


Projecting all layers to metric CRS...
Grid substations (aggregated): 2,537
  Congested (<1 MW):  2,254
  Moderate (1-5 MW):  109
  Sufficient (≥5 MW): 174
Filter results:
  Road proximity (within 8 km): 10,758 pass
  → 10,758 candidate sites (grid scored in §3.1, not hard-filtered)


In [132]:
# ── Step 2: Score candidates ──────────────────────────────────────────────────
# All five signals derived from Sections 1 and 2 — no new data introduced here.

# Drop any previously computed columns so this cell is safe to re-run
_score_cols = [
    "dist_charger_m", "nearest_mw", "distributor_network", "capacity_available_mw",
    "_source", "traffic_score", "road_weight", "nearest_road_ref", "poi_weight",
    "ev_fleet", "s_traffic", "s_gap", "s_grid", "s_road", "s_poi", "s_demand",
    "composite_score",
]
candidates_m = candidates_m.drop(
    columns=[c for c in _score_cols if c in candidates_m.columns], errors="ignore"
)

# ── POI category weights (from sites_candidate["category"], §2.4) ─────────────
POI_WEIGHT = {
    "motorway_services": 1.00, "rest_area": 0.95, "petrol_station": 0.90,
    "mall": 0.75, "department_store": 0.70, "supermarket": 0.65,
    "hotel": 0.60, "train_station": 0.55, "bus_station": 0.50,
    "hospital": 0.50, "university": 0.45, "airport": 0.45,
    "sports_centre": 0.40, "parking": 0.40, "attraction": 0.35,
    "cinema": 0.30, "theatre": 0.30, "stadium": 0.30,
}

# ── 2a. Nearest charger distance — vectorised (sjoin_nearest) ─────────────────
_cand_charger = gpd.sjoin_nearest(
    candidates_m[["osm_id", "geometry"]],
    _chargers_m[["geometry"]],
    how="left", distance_col="dist_charger_m"
).drop_duplicates(subset="osm_id")
candidates_m = candidates_m.merge(
    _cand_charger[["osm_id", "dist_charger_m"]], on="osm_id", how="left"
)
candidates_m["dist_charger_m"] = candidates_m["dist_charger_m"].fillna(0)

# ── 2b. Nearest grid node: MW + distributor — vectorised (sjoin_nearest) ──────
_grid_proj = _grid_m[["capacity_available_mw", "_source", "geometry"]].copy()
_cand_grid = gpd.sjoin_nearest(
    candidates_m[["osm_id", "geometry"]],
    _grid_proj,
    how="left", distance_col="_dist_grid"
).drop_duplicates(subset="osm_id")
candidates_m = candidates_m.merge(
    _cand_grid[["osm_id", "capacity_available_mw", "_source"]], on="osm_id", how="left"
)
candidates_m["nearest_mw"]          = candidates_m["capacity_available_mw"].fillna(0)
candidates_m["distributor_network"] = (
    candidates_m["_source"].map(DISTRIBUTOR_MAP).fillna("Unknown")
)
candidates_m = candidates_m.drop(columns=["capacity_available_mw", "_source"])

# ── 2c. Traffic score: mean travellers/day on roads within buffer ─────────────
_road_pts           = _traffic_m.set_geometry("centroid")
_buf_df             = candidates_m[["osm_id", "geometry"]].copy()
_buf_df["geometry"] = candidates_m.geometry.buffer(TRAFFIC_BUFFER_M)
_joined = gpd.sjoin(
    _road_pts[["Total", "centroid"]].set_geometry("centroid"),
    _buf_df, how="left", predicate="within"
)
_t_agg       = _joined.groupby("osm_id")["Total"].mean().rename("traffic_score")
candidates_m = candidates_m.merge(_t_agg, on="osm_id", how="left")
candidates_m["traffic_score"] = candidates_m["traffic_score"].fillna(0)

# ── 2d. Road type weight + nearest road ref — vectorised (sjoin_nearest) ──────
_road_proj = _traffic_m[["ref", "geometry"]].copy()
_cand_road = gpd.sjoin_nearest(
    candidates_m[["osm_id", "geometry"]],
    _road_proj,
    how="left", distance_col="_dist_road"
).drop_duplicates(subset="osm_id")
_cand_road["_prefix"]      = _cand_road["ref"].astype(str).apply(
    lambda r: r.split("-")[0] if "-" in r else r[:2]
)
_cand_road["_road_weight"] = _cand_road["_prefix"].map(ROAD_PRIORITY).fillna(0.5)
candidates_m = candidates_m.merge(
    _cand_road[["osm_id", "ref", "_road_weight"]].rename(
        columns={"ref": "nearest_road_ref", "_road_weight": "road_weight"}
    ),
    on="osm_id", how="left"
)
candidates_m["road_weight"]      = candidates_m["road_weight"].fillna(0.5)
candidates_m["nearest_road_ref"] = candidates_m["nearest_road_ref"].fillna("")

# ── 2e. POI category weight (§2.4) ───────────────────────────────────────────
candidates_m["poi_weight"] = candidates_m["category"].map(POI_WEIGHT).fillna(0.35)

# ── 2f. Province EV demand weight (§1.2 + §1.1) ──────────────────────────────
# Spatial join to province polygons → ev_fleet_2027 → normalise against national total
try:
    _geo_url  = ("https://raw.githubusercontent.com/codeforgermany/"
                 "click_that_hood/main/public/data/spain-provinces.geojson")
    _prov_geo = gpd.read_file(_geo_url).to_crs(CRS_M)

    def _norm(s):
        import unicodedata
        return "".join(c for c in unicodedata.normalize("NFD", str(s).lower().strip())
                       if unicodedata.category(c) != "Mn")

    _prov_geo["_norm"] = _prov_geo["name"].apply(_norm)
    _name_to_fleet     = {
        _norm(row["province_name"]): row["ev_fleet_2027"]
        for _, row in demand.reset_index().iterrows()
    }
    _prov_geo["ev_fleet"] = _prov_geo["_norm"].map(_name_to_fleet).fillna(0)

    _cand_pts  = candidates_m[["osm_id", "geometry"]].copy()
    _cand_prov = gpd.sjoin(_cand_pts, _prov_geo[["ev_fleet", "geometry"]],
                           how="left", predicate="within")
    _fleet_map = _cand_prov.groupby("osm_id")["ev_fleet"].first()
    candidates_m = candidates_m.merge(_fleet_map, on="osm_id", how="left")
    candidates_m["ev_fleet"] = candidates_m["ev_fleet"].fillna(0)

    _max_fleet = max(float(candidates_m["ev_fleet"].quantile(0.95)), 1.0)
    candidates_m["s_demand"] = (candidates_m["ev_fleet"] / _max_fleet).clip(0, 1)
    print("✓ Province EV demand signal added (§1.2)")
except Exception as e:
    candidates_m["s_demand"] = 0.0
    print(f"  Province demand join failed ({e}) — s_demand = 0")

# ── 2g. Composite score ───────────────────────────────────────────────────────
_max_t = max(float(candidates_m["traffic_score"].quantile(0.95)), 1.0)
_max_g = max(float(candidates_m["nearest_mw"].quantile(0.95)), 0.1)

candidates_m["s_traffic"] = (candidates_m["traffic_score"]  / _max_t).clip(0, 1)
candidates_m["s_gap"]     = (candidates_m["dist_charger_m"] / 80_000).clip(0, 1)
candidates_m["s_grid"]    = (candidates_m["nearest_mw"]     / _max_g).clip(0, 1)
candidates_m["s_road"]    = candidates_m["road_weight"]
candidates_m["s_poi"]     = candidates_m["poi_weight"]

candidates_m["composite_score"] = (
    0.30 * candidates_m["s_traffic"] +
    0.25 * candidates_m["s_gap"]     +
    0.20 * candidates_m["s_grid"]    +
    0.15 * candidates_m["s_poi"]     +
    0.10 * candidates_m["s_demand"]
)

candidates_wgs = (
    candidates_m
    .sort_values("composite_score", ascending=False)
    .to_crs("EPSG:4326")
)

print(f"\nScored {len(candidates_wgs):,} candidate sites")
print(f"\nMean signal values:")
for col, label in [("s_traffic","Traffic"), ("s_gap","Charger gap"),
                   ("s_grid","Grid"), ("s_poi","POI type"), ("s_demand","EV demand")]:
    print(f"  {label:<14} {candidates_wgs[col].mean():.3f}")
print(f"\nTop 10 candidates:")
print(candidates_wgs[["name", "category", "nearest_road_ref",
                       "nearest_mw", "traffic_score", "composite_score"]]
      .head(10).round(3).to_string(index=False))


✓ Province EV demand signal added (§1.2)

Scored 10,758 candidate sites

Mean signal values:
  Traffic        0.396
  Charger gap    0.037
  Grid           0.078
  POI type       0.911
  EV demand      0.122

Top 10 candidates:
                    name          category nearest_road_ref  nearest_mw  traffic_score  composite_score
                     EXD motorway_services            M-502       36.87      32994.492            0.754
       Sistemas Ambiente motorway_services            M-502       36.87      32915.870            0.754
Suministros Industriales motorway_services              A-5       36.87      33417.208            0.754
                Asesoria motorway_services            M-511       36.87      32497.321            0.753
                   Moeve    petrol_station             M-40       19.41      36618.301            0.751
                   Moeve    petrol_station             M-40       19.41      36173.157            0.751
                                 rest_area  

---
### 3.2 Grid Status Assignment

Assigns `grid_status` to each candidate from `nearest_mw` (nearest substation in `gdf_grid`, §2.3).
Thresholds defined in §0 (`GRID_SUFFICIENT_MW`, `GRID_MODERATE_MW`) and justified in the Appendix.

> **Rule 1 (mandatory):** `grid_status` must be assigned from the distributor's
> `capacity_available_mw` field. Spatial matching: nearest substation within `GRID_SNAP_RADIUS_M`.

| `grid_status` | Condition | Implication |
|---|---|---|
| `Sufficient` | `nearest_mw ≥ GRID_SUFFICIENT_MW` (5.0 MW) | Supports ≥10 × 150 kW with headroom |
| `Moderate` | `GRID_MODERATE_MW ≤ nearest_mw < GRID_SUFFICIENT_MW` (1.0–5.0 MW) | AFIR minimum (2 chargers) supportable |
| `Congested` | `nearest_mw < GRID_MODERATE_MW` (1.0 MW) | Grid reinforcement required |


In [133]:
# ── Step 3: Assign grid_status ────────────────────────────────────────────────
# Rule:
#   Sufficient : nearest_mw >= GRID_SUFFICIENT_MW  (≥ 5.0 MW)
#   Moderate   : GRID_MODERATE_MW <= nearest_mw < GRID_SUFFICIENT_MW  (1.0–5.0 MW)
#   Congested  : nearest_mw < GRID_MODERATE_MW  (< 1.0 MW)
#
# Documented thresholds (CNMC Circular 1/2024 + AFIR 150 kW standard):
#   A 6-charger HPC station at 150 kW each = 0.9 MW demand
#   Sufficient ≥5 MW: any realistic station size covered without upgrade
#   Congested <1 MW: cannot host a standard 6-charger HPC station (0.9 MW)

def assign_grid_status(mw):
    if pd.isna(mw) or mw < GRID_MODERATE_MW:
        return "Congested"
    elif mw < GRID_SUFFICIENT_MW:
        return "Moderate"
    else:
        return "Sufficient"

candidates_wgs["grid_status"] = candidates_wgs["nearest_mw"].apply(assign_grid_status)

status_counts = candidates_wgs["grid_status"].value_counts()
print("Grid status distribution:")
print(status_counts.to_string())


Grid status distribution:
grid_status
Congested     9121
Sufficient    1228
Moderate       409


---
### 3.3 Charger Sizing

Derives `n_chargers_proposed` from `s_traffic` (traffic score) and `grid_status`.
`estimated_demand_kw` is always `n_chargers_proposed × 150 kW` — fixed per **Rule 2**.

Grid status hard caps: `Congested` → max 2 · `Moderate` → max 6 · `Sufficient` → max 12.


In [134]:
# ── Step 4: Propose number of chargers per station ────────────────────────────
# Logic:
#   n_chargers = f(traffic_score, grid_status, road_weight)
#   Bounded by AFIR minimum (2) and practical maximum (12)
#
# Rationale:
#   - Traffic score normalised to [0,1] drives base charger count
#   - High-priority roads (AP/A) get +2 bonus chargers
#   - Congested grid caps at 2 (AFIR minimum only)
#   - Moderate grid caps at 6
#   - Sufficient grid allows up to 12

def propose_n_chargers(row):
    base = 2 + round(row["s_traffic"] * 8)      # 2–10 based on traffic
    base += 2 if row["road_weight"] >= 0.9 else 0  # +2 for motorways
    if   row["grid_status"] == "Congested":  return max(2, min(base, 2))
    elif row["grid_status"] == "Moderate":   return max(2, min(base, 6))
    else:                                    return max(2, min(base, 12))

candidates_wgs["n_chargers_proposed"] = candidates_wgs.apply(propose_n_chargers, axis=1)
candidates_wgs["estimated_demand_kw"] = candidates_wgs["n_chargers_proposed"] * CHARGER_KW_STANDARD

print("Charger count distribution:")
print(candidates_wgs["n_chargers_proposed"].value_counts().sort_index().to_string())
print(f"\nTotal charger capacity proposed: {candidates_wgs['estimated_demand_kw'].sum():,.0f} kW")


Charger count distribution:
n_chargers_proposed
2     9147
3      237
4      249
5      245
6      382
7      146
8      113
9       85
10      80
11      33
12      41

Total charger capacity proposed: 4,179,600 kW


---
### 3.4 File 2 — Proposed Charging Locations

All proposed stations from `candidates_wgs`, top-ranked by `composite_score`.

**Required columns (datathon spec):** `location_id` · `latitude` · `longitude` ·
`route_segment` · `n_chargers_proposed` · `grid_status`

**Saved to:** `outputs/File 2.csv`


In [135]:
# ── Top-N selection: best-scoring candidates with spatial diversity ──────────
# Greedy spatial filter: keep highest-scoring sites that are ≥40 km apart.
# This enforces AFIR compliance and produces a realistic station count.

import math

TOP_N_CAP = 300          # hard cap
GAP_FILTER_M = 40_000   # 40 km AFIR minimum gap

def _haversine_m(lat1, lon1, lat2, lon2):
    R = 6_371_000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2
    return R * 2 * math.asin(math.sqrt(a))

_pool = candidates_wgs.copy()
_pool["_lat"] = _pool.geometry.y
_pool["_lon"] = _pool.geometry.x

_selected_idx = []
_selected_coords = []

for idx, row in _pool.iterrows():
    _lat, _lon = row["_lat"], row["_lon"]
    if any(_haversine_m(_lat, _lon, slat, slon) < GAP_FILTER_M
           for slat, slon in _selected_coords):
        continue
    _selected_idx.append(idx)
    _selected_coords.append((_lat, _lon))
    if len(_selected_idx) >= TOP_N_CAP:
        break

candidates_final = _pool.loc[_selected_idx].drop(columns=["_lat", "_lon"])
print(f"Spatial selection: {len(_pool):,} scored → {len(candidates_final):,} proposed")
print(f"  (≥{GAP_FILTER_M/1000:.0f} km spacing · max {TOP_N_CAP})")

# ── File 2: Proposed Charging Locations ──────────────────────────────────────
# Rule 2: estimated_demand_kw = n_chargers_proposed × 150 kW (fixed)
# Filename: exactly "File 2.csv" per datathon specification

candidates_final["lat"] = candidates_final.geometry.y.round(6)
candidates_final["lon"] = candidates_final.geometry.x.round(6)

file2 = pd.DataFrame({
    "location_id":         [f"IBE_{i+1:03d}" for i in range(len(candidates_final))],
    "latitude":            candidates_final["lat"].values,
    "longitude":           candidates_final["lon"].values,
    "route_segment":       candidates_final["nearest_road_ref"].values,
    "n_chargers_proposed": candidates_final["n_chargers_proposed"].values,
    "grid_status":         candidates_final["grid_status"].values,
    "distributor_network": candidates_final["distributor_network"].values,
    "estimated_demand_kw": candidates_final["n_chargers_proposed"].values * 150,
    "composite_score":     candidates_final["composite_score"].round(4).values,
    "site_name":           candidates_final["name"].fillna("").values,
    "category":            candidates_final["category"].values,
})

file2.to_csv(OUT / "File 2.csv", index=False)

print(f"File 2: {len(file2):,} proposed stations  →  outputs/File 2.csv")
print(f"  Sufficient : {(file2['grid_status']=='Sufficient').sum():,}")
print(f"  Moderate   : {(file2['grid_status']=='Moderate').sum():,}")
print(f"  Congested  : {(file2['grid_status']=='Congested').sum():,}")
print(f"Structure (dtypes):")
print(file2.dtypes.to_string())
print(f"First 15 rows:")
print(file2.head(15).to_string(index=False))


Spatial selection: 10,758 scored → 177 proposed
  (≥40 km spacing · max 300)
File 2: 177 proposed stations  →  outputs/File 2.csv
  Sufficient : 38
  Moderate   : 8
  Congested  : 131
Structure (dtypes):
location_id             object
latitude               float64
longitude              float64
route_segment           object
n_chargers_proposed      int64
grid_status             object
distributor_network     object
estimated_demand_kw      int64
composite_score        float64
site_name               object
category                object
First 15 rows:
location_id  latitude  longitude route_segment  n_chargers_proposed grid_status distributor_network  estimated_demand_kw  composite_score                      site_name          category
    IBE_001 40.397195  -3.772053         M-502                   10  Sufficient                i-DE                 1500           0.7542                            EXD motorway_services
    IBE_002 41.369806   2.106722         N-340                   1

---
### 3.5 File 3 — Friction Points

Locations where high charging demand collides with limited grid capacity.
Subset of File 2 where `grid_status` is `Moderate` or `Congested` — **Rule 3**.

`Sufficient` sites must not appear here.

**Saved to:** `outputs/File 3.csv`


In [136]:
# ── File 3: Friction Points ───────────────────────────────────────────────────
# Rule 3: only Moderate or Congested rows from File 2 — no Sufficient allowed
# Filename: exactly "File 3.csv" per datathon specification

file3_base = file2[file2["grid_status"].isin(["Moderate", "Congested"])].copy()
file3_base = file3_base.reset_index(drop=True)

file3 = pd.DataFrame({
    "bottleneck_id":       [f"FRIC_{i+1:03d}" for i in range(len(file3_base))],
    "latitude":            file3_base["latitude"].values,
    "longitude":           file3_base["longitude"].values,
    "route_segment":       file3_base["route_segment"].values,
    "distributor_network": file3_base["distributor_network"].values,
    "estimated_demand_kw": file3_base["estimated_demand_kw"].values,
    "grid_status":         file3_base["grid_status"].values,
})

file3.to_csv(OUT / "File 3.csv", index=False)

print(f"File 3: {len(file3):,} friction points  →  outputs/File 3.csv")
print(f"  Congested : {(file3['grid_status']=='Congested').sum():,}")
print(f"  Moderate  : {(file3['grid_status']=='Moderate').sum():,}")
print(f"\nStructure (dtypes):")
print(file3.dtypes.to_string())
print(f"\nFirst 15 rows:")
print(file3.head(15).to_string(index=False))


File 3: 139 friction points  →  outputs/File 3.csv
  Congested : 131
  Moderate  : 8

Structure (dtypes):
bottleneck_id           object
latitude               float64
longitude              float64
route_segment           object
distributor_network     object
estimated_demand_kw      int64
grid_status             object

First 15 rows:
bottleneck_id  latitude  longitude route_segment distributor_network  estimated_demand_kw grid_status
     FRIC_001 40.290222  -3.299778           A-3                i-DE                  300   Congested
     FRIC_002 41.643578   2.517931         AP-7N              Endesa                  300   Congested
     FRIC_003 37.397934  -6.484729          A-49              Endesa                  300   Congested
     FRIC_004 39.583807  -0.286127           A-7                i-DE                  300   Congested
     FRIC_005 36.659067  -4.570736           A-7              Endesa                  300   Congested
     FRIC_006 36.456317  -6.218776           A-4 

---
### 3.6 File 1 — Global Network KPIs

Single summary row. All four values are derived from earlier outputs — no new data.

| Field | Source |
|---|---|
| `total_proposed_stations` | `len(file2)` |
| `total_existing_stations_baseline` | interurban stations from `chargers_gdf` (§2.2) |
| `total_friction_points` | `len(file3)` |
| `total_ev_projected_2027` | `TOTAL_EV_PROJECTED_2027` (§1.1) |

**Saved to:** `outputs/File 1.csv`


In [137]:
# ── File 1: Global Network KPIs ──────────────────────────────────────────────
# Filename: exactly "File 1.csv" per datathon specification

# Baseline: interurban chargers from §2.2 (filtered to roads within 2 km buffer)
_road_buf_wgs             = traffic_map.to_crs(3857).copy()
_road_buf_wgs["geometry"] = _road_buf_wgs.geometry.buffer(2000)
_road_union_baseline      = _road_buf_wgs.to_crs(4326).unary_union
existing_interurban       = chargers_gdf[chargers_gdf.geometry.within(_road_union_baseline)]

file1 = pd.DataFrame([{
    "total_proposed_stations":          len(file2),
    "total_existing_stations_baseline": len(existing_interurban),
    "total_friction_points":            len(file3),
    "total_ev_projected_2027":          int(TOTAL_EV_PROJECTED_2027),
}])

file1.to_csv(OUT / "File 1.csv", index=False)

print("=" * 55)
print("File 1 — Global Network KPIs  →  outputs/File 1.csv")
print("=" * 55)
for col in file1.columns:
    print(f"  {col:<45} {file1[col].iloc[0]:>10,}")
print("=" * 55)
print(f"\nStructure (dtypes):")
print(file1.dtypes.to_string())
print(f"\nFull row:")
print(file1.to_string(index=False))
if TOTAL_EV_PROJECTED_2027 == 0:
    print("\n⚠  total_ev_projected_2027 = 0 — ensure Section 1.1 ran successfully")


File 1 — Global Network KPIs  →  outputs/File 1.csv
  total_proposed_stations                              177
  total_existing_stations_baseline                   6,896
  total_friction_points                                139
  total_ev_projected_2027                        1,412,640

Structure (dtypes):
total_proposed_stations             int64
total_existing_stations_baseline    int64
total_friction_points               int64
total_ev_projected_2027             int64

Full row:
 total_proposed_stations  total_existing_stations_baseline  total_friction_points  total_ev_projected_2027
                     177                              6896                    139                  1412640


---
### 3.7 BI Visualisation

Self-contained HTML map. Meets all datathon BI requirements (§5.3):

- All File 2 stations plotted and colour-coded: **Green** = Sufficient · **Yellow** = Moderate · **Red** = Congested
- Popup shows: `location_id`, `route_segment`, `n_chargers_proposed`, `grid_status`
- No internet connection, login, or software installation required
- Additional layers: traffic intensity, existing chargers, grid capacity nodes

**Saved to:** `outputs/BI_map.html`


In [128]:
# ── BI Map — Self-contained HTML ─────────────────────────────────────────────
# Mandatory layer: File 2 stations, colour-coded by grid_status (§5.3)
# Additional layers: traffic, existing chargers, grid nodes (positively valued)

STATUS_COLOR = {"Sufficient": "#27ae60", "Moderate": "#f39c12", "Congested": "#e74c3c"}

m_bi = folium.Map(location=SPAIN_CENTER, zoom_start=DEFAULT_ZOOM, tiles=MAP_TILES)

# ── Layer 1 (mandatory): Proposed stations from File 2 ───────────────────────
fg_proposed = folium.FeatureGroup(name="Proposed Stations (File 2)", show=True)
for _, row in file2.iterrows():
    color = STATUS_COLOR.get(row["grid_status"], "#95a5a6")
    popup_html = (
        f"<b>{row['location_id']}</b><br>"
        f"Road: {row['route_segment']}<br>"
        f"Chargers: {int(row['n_chargers_proposed'])} × 150 kW<br>"
        f"Demand: {int(row['estimated_demand_kw'])} kW<br>"
        f"Grid: <b style='color:{color}'>{row['grid_status']}</b><br>"
        f"Distributor: {row['distributor_network']}<br>"
        f"Site: {row['site_name'] or '—'} ({row['category']})"
    )
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=7 + row["n_chargers_proposed"],
        color=color, fill=True, fill_color=color, fill_opacity=0.85,
        popup=folium.Popup(popup_html, max_width=280),
        tooltip=f"{row['location_id']} | {row['route_segment']} | {row['grid_status']}"
    ).add_to(fg_proposed)
fg_proposed.add_to(m_bi)

# ── Layer 2 (additional): Traffic intensity ───────────────────────────────────
fg_traffic = folium.FeatureGroup(name="Traffic Intensity", show=False)
for _, row in traffic_map.iterrows():
    total = row.get("Total")
    if pd.isna(total): continue
    geom  = row.geometry
    lines = geom.geoms if geom.geom_type == "MultiLineString" else [geom]
    for line in lines:
        coords = [(lat, lon) for lon, lat in line.coords]
        if len(coords) < 2: continue
        folium.PolyLine(coords, color=traffic_color(total), weight=2,
                        opacity=0.7,
                        tooltip=f"{row.get('ref','—')} | {int(total):,}/day"
                        ).add_to(fg_traffic)
fg_traffic.add_to(m_bi)

# ── Layer 3 (additional): Existing chargers ───────────────────────────────────
fg_existing = folium.FeatureGroup(name="Existing Chargers (baseline)", show=False)
for _, row in chargers_gdf.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=4, color="#7f8c8d", fill=True, fill_opacity=0.6,
        tooltip=f"{row.get('site_name') or 'Charger'} | {row.get('site_max_power_kw','?')} kW"
    ).add_to(fg_existing)
fg_existing.add_to(m_bi)

# ── Layer 4 (additional): Grid capacity nodes ─────────────────────────────────
fg_grid = folium.FeatureGroup(name="Grid Capacity (substations)", show=False)
for _, row in gdf_grid.iterrows():
    mw = row.get("capacity_available_mw")
    if pd.isna(mw): continue
    color = "#2ca02c" if mw >= GRID_SUFFICIENT_MW else "#f39c12" if mw >= GRID_MODERATE_MW else "#e74c3c"
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=4, color=color, fill=True, fill_opacity=0.5,
        tooltip=f"{row.get('_source','—')} | {mw:.1f} MW"
    ).add_to(fg_grid)
fg_grid.add_to(m_bi)

# ── Legend + layer control ─────────────────────────────────────────────────────
folium.LayerControl(collapsed=False).add_to(m_bi)

legend_html = (
    '<div style="position:fixed;bottom:30px;left:30px;z-index:9999;background:white;'
    'padding:12px 16px;border-radius:8px;border:1px solid #ccc;font-size:11px;line-height:1.8">'
    '<b style="font-size:12px">Proposed Stations — Grid Status</b><br>'
    '<span style="color:#27ae60">&#9679;</span> Sufficient (≥5 MW)<br>'
    '<span style="color:#f39c12">&#9679;</span> Moderate (1–5 MW)<br>'
    '<span style="color:#e74c3c">&#9679;</span> Congested (&lt;1 MW)<br>'
    '<i style="font-size:10px">Marker size = n_chargers_proposed</i>'
    '</div>'
)
m_bi.get_root().html.add_child(folium.Element(legend_html))

m_bi.save(str(OUT / "BI_map.html"))
print("BI map saved  →  outputs/BI_map.html")
print(f"  Stations plotted : {len(file2):,}")
print(f"  Sufficient       : {(file2['grid_status']=='Sufficient').sum():,}  (green)")
print(f"  Moderate         : {(file2['grid_status']=='Moderate').sum():,}  (yellow)")
print(f"  Congested        : {(file2['grid_status']=='Congested').sum():,}  (red)")
display(IFrame(src=str(OUT / "BI_map.html"), width="100%", height=600))

BI map saved  →  outputs/BI_map.html
  Stations plotted : 177
  Sufficient       : 38  (green)
  Moderate         : 8  (yellow)
  Congested        : 131  (red)


---
### 3.8 Output Compliance Check

Verifies all three output files against the mandatory datathon rules (§5.2, §6.4).
Disqualification risks checked: Rule 1 (grid_status from distributor data) ·
Rule 2 (estimated_demand_kw = n_chargers × 150) · Rule 3 (no Sufficient in File 3).


In [129]:
# ── Datathon output compliance check ─────────────────────────────────────────
# Covers all mandatory rules and disqualification criteria from §5.2 and §6.4

print("=" * 65)
print("DATATHON OUTPUT COMPLIANCE CHECK")
print("=" * 65)

FILE1_COLS = ["total_proposed_stations", "total_existing_stations_baseline",
              "total_friction_points", "total_ev_projected_2027"]
FILE2_COLS = ["location_id", "latitude", "longitude", "route_segment",
              "n_chargers_proposed", "grid_status"]
FILE3_COLS = ["bottleneck_id", "latitude", "longitude", "route_segment",
              "distributor_network", "estimated_demand_kw", "grid_status"]

all_pass = True

for fname, df, required_cols in [
    ("File 1", file1, FILE1_COLS),
    ("File 2", file2, FILE2_COLS),
    ("File 3", file3, FILE3_COLS),
]:
    missing = [c for c in required_cols if c not in df.columns]
    status  = "✓ COMPLIANT" if not missing else f"✗ MISSING COLUMNS: {missing}"
    if missing: all_pass = False
    print(f"\n{fname}  ({len(df):,} rows)  →  {status}")
    print(f"  {'Column':<35} {'Type':<12} Sample value")
    print(f"  {'-'*60}")
    for col in required_cols:
        if col in df.columns:
            val  = str(df[col].iloc[0])[:30] if len(df) > 0 else "—"
            dtype = str(df[col].dtype)
            print(f"  ✓ {col:<33} {dtype:<12} {val}")
        else:
            print(f"  ✗ {col:<33} MISSING")

# ── Rule 1: grid_status values are valid ──────────────────────────────────────
print("\n" + "─" * 65)
valid_statuses = {"Sufficient", "Moderate", "Congested"}
f2_invalid = set(file2["grid_status"].unique()) - valid_statuses
r1 = len(f2_invalid) == 0
if not r1: all_pass = False
print(f"Rule 1 — grid_status valid values only:           {'✓' if r1 else '✗'}")
if not r1: print(f"  Invalid values found: {f2_invalid}")

# ── Rule 2: estimated_demand_kw = n_chargers_proposed × 150 ─────────────────
r2 = (file2["estimated_demand_kw"] == file2["n_chargers_proposed"] * 150).all()
if not r2: all_pass = False
print(f"Rule 2 — estimated_demand_kw = n_chargers × 150:  {'✓' if r2 else '✗'}")

# ── Rule 3: no Sufficient in File 3 ──────────────────────────────────────────
r3 = not (file3["grid_status"] == "Sufficient").any()
if not r3: all_pass = False
print(f"Rule 3 — no Sufficient rows in File 3:            {'✓' if r3 else '✗'}")

# ── File 3 distributor values ─────────────────────────────────────────────────
valid_distributors = {"i-DE", "Endesa", "Viesgo"}
f3_dist_invalid = set(file3["distributor_network"].unique()) - valid_distributors
rd = len(f3_dist_invalid) == 0
if not rd: all_pass = False
print(f"Distributor values (i-DE / Endesa / Viesgo only): {'✓' if rd else '✗'}")
if not rd: print(f"  Unexpected values: {f3_dist_invalid}")

# ── Output files on disk ──────────────────────────────────────────────────────
print("\n" + "─" * 65)
print("Output files on disk:")
for fname in ["File 1.csv", "File 2.csv", "File 3.csv", "BI_map.html"]:
    p = OUT / fname
    exists = p.exists()
    if not exists: all_pass = False
    size   = f"{p.stat().st_size // 1024} KB" if exists else "MISSING"
    print(f"  {'✓' if exists else '✗'}  {fname:<20} {size}")

print("\n" + "=" * 65)
print(f"OVERALL: {'✓ ALL CHECKS PASSED' if all_pass else '✗ ISSUES FOUND — see above'}")
print("=" * 65)


DATATHON OUTPUT COMPLIANCE CHECK

File 1  (1 rows)  →  ✓ COMPLIANT
  Column                              Type         Sample value
  ------------------------------------------------------------
  ✓ total_proposed_stations           int64        177
  ✓ total_existing_stations_baseline  int64        6896
  ✓ total_friction_points             int64        139
  ✓ total_ev_projected_2027           int64        1412640

File 2  (177 rows)  →  ✓ COMPLIANT
  Column                              Type         Sample value
  ------------------------------------------------------------
  ✓ location_id                       object       IBE_001
  ✓ latitude                          float64      40.397195
  ✓ longitude                         float64      -3.772053
  ✓ route_segment                     object       M-502
  ✓ n_chargers_proposed               int64        10
  ✓ grid_status                       object       Sufficient

File 3  (139 rows)  →  ✓ COMPLIANT
  Column                    

---
---
## Appendix — Data Sources & Regulatory References

### Data Sources

| Dataset | Provider | Licence | URL |
|---|---|---|---|
| Road network (OSM) | OpenStreetMap contributors | ODbL | https://www.openstreetmap.org |
| Traffic flows (BigData Movilidad) | Ministerio de Transportes | Open data | https://mapas.fomento.gob.es |
| EV charging stations (DATEX II) | DGT | Open data | https://infocar.dgt.es |
| Grid demand capacity | i-DE / CNMC | Regulatory publication | https://www.cnmc.es |

### Regulatory Framework

- **EU AFIR Regulation 2023/1804** — HPC chargers required every 60 km on TEN-T Core corridors and every 100 km on Comprehensive corridors by 2025–2030.
- **Royal Decree 569/2020** — Spanish transposition framework for EV infrastructure.
- **CNMC Resolution** — grid access capacity rules for demand connection points.

### Known Limitations

1. **Traffic data** reflects 2023 mobility patterns; EV charging demand is growing ~15%/year per DGT projections.
2. **Grid capacity** covers i-DE (Iberdrola) territory only. Endesa, UFD, and Viesgo zones require separate CNMC datasets.
3. **Charger power** values are self-reported; operational availability is not tracked in the DGT feed.
4. **AFIR gap analysis** (150 km rule) not yet implemented — requires network-graph routing along OSM road segments.


---
### 3.9 Sync Outputs to Streamlit

Copies `File 1.csv`, `File 2.csv`, `File 3.csv` from `notebooks/outputs/` to `streamlit_app/data/` so the dashboard shows real results.

In [130]:
# ── Copy outputs to Streamlit data directory ──────────────────────────────────
import shutil
import pandas as pd

_streamlit_data = Path("..") / "streamlit_app" / "data"
_streamlit_data.mkdir(parents=True, exist_ok=True)

# ── 1. File 1/2/3 ─────────────────────────────────────────────────────────────
_copy_map = {
    OUT / "File 1.csv": _streamlit_data / "file1.csv",
    OUT / "File 2.csv": _streamlit_data / "file2.csv",
    OUT / "File 3.csv": _streamlit_data / "file3.csv",
}
for src_path, dst_path in _copy_map.items():
    if src_path.exists():
        shutil.copy(src_path, dst_path)
        print(f"  Copied {src_path.name}  →  streamlit_app/data/{dst_path.name}")
    else:
        print(f"  ⚠  {src_path.name} not found — run sections 3.4–3.6 first")

# ── 2. EV forecast (quarterly, for Streamlit chart) ───────────────────────────
# Combines historical monthly series → quarterly actual + SARIMA forecast → quarterly forecast
try:
    # Historical (actual) — use the monthly series variable from §1.1
    _hist_q = (
        monthly["ev_reg"]
        .resample("QS")
        .sum()
        .reset_index()
        .rename(columns={"date": "date", "ev_reg": "ev_registrations"})
    )
    _hist_q["type"] = "actual"

    # Forecast — load from saved monthly CSV
    _fc_monthly = pd.read_csv(OUT / "ev_monthly_forecast_2025_2027.csv", parse_dates=["date"])
    _fc_q = (
        _fc_monthly.set_index("date")["monthly_ev_registrations_forecast"]
        .resample("QS")
        .sum()
        .reset_index()
        .rename(columns={"date": "date", "monthly_ev_registrations_forecast": "ev_registrations"})
    )
    _fc_q["type"] = "forecast"

    _ev_forecast = pd.concat([_hist_q, _fc_q], ignore_index=True)
    _ev_forecast["date"] = _ev_forecast["date"].dt.strftime("%Y-%m-%d")
    _ev_forecast["ev_registrations"] = _ev_forecast["ev_registrations"].astype(int)
    _ev_forecast.to_csv(_streamlit_data / "ev_forecast.csv", index=False)
    print(f"  Built ev_forecast.csv  ({len(_hist_q)} actual + {len(_fc_q)} forecast quarters)")
except Exception as e:
    print(f"  ⚠  ev_forecast.csv skipped ({e})")

print("✓ Streamlit data directory updated")
print(f"  Path: {_streamlit_data.resolve()}")


  Copied File 1.csv  →  streamlit_app/data/file1.csv
  Copied File 2.csv  →  streamlit_app/data/file2.csv
  Copied File 3.csv  →  streamlit_app/data/file3.csv
  Built ev_forecast.csv  (36 actual + 16 forecast quarters)
✓ Streamlit data directory updated
  Path: C:\Users\steve\OneDrive - IE University\Term 2\DATATHON_2\Iberdrola\SpanishElecticGrid\streamlit_app\data
